# Sensors Final Experiment: Multi-Objective Radar-Only Deployment

## Constrained Robustness Fine-Tuning on RTPD-Net

This notebook follows the supervisor feedback exactly:

- **Do not add another architecture.**
- **Do not position clean accuracy as the only objective.**
- Use **RTPD-Net** as the clean-performance reference model.
- Use **PARETO-RKD** and **AURORA-RKD** as complementary evidence for other deployment priorities.
- Run one final **constrained fine-tuning** experiment to test whether RTPD-Net’s clean accuracy can be preserved while improving radar-corruption robustness.

### Paper framing

> Wearable-assisted radar learning improves radar-only rehabilitation monitoring, but optimal model choice depends on deployment priority.

The final experiment starts from the already trained RTPD-Net radar-only student and applies a small, validation-constrained robust fine-tuning stage. It does **not** introduce a new architecture and does **not** combine temporal, relational, prototype, adversarial, calibration and robustness losses all at once.

The key question is:

> Can RTPD-Net retain its clean recognition advantage while gaining corruption robustness?

## Supervisor-aligned experimental design

The project is now reframed as a **multi-objective comparative deployment study**:

1. **RTPD-Net**: clean-performance reference.
2. **PARETO-RKD**: robustness and cross-subject stability evidence.
3. **AURORA-RKD**: calibration and complete missing-modality safety evidence.
4. **Constrained RTPD Fine-Tuning**: final targeted experiment, testing whether the clean-performance reference can be made more robust without sacrificing its clean validation performance.

The final model is selected by validation constraints, not by test-set inspection.

A fine-tuned checkpoint is accepted only when it satisfies:

- clean validation macro-F1 ≥ base RTPD validation macro-F1 minus a small tolerance;
- clean validation accuracy ≥ base RTPD validation accuracy minus a small tolerance;
- corrupted validation macro-F1 improves over base RTPD.

If no candidate satisfies these conditions, the notebook reports that RTPD-Net remains the clean deployment reference and the robustness gain is not reliable enough.

# RTPD-Net for mRI Rehabilitation-Exercise Recognition

## Reliability-Aware Temporal Prototype Distillation

This notebook implements a publication-oriented extension of the original radar–IMU knowledge-distillation experiment.

### Proposed contributions

1. **Radar self-supervised pretraining** using masked temporal reconstruction and cross-view contrastive learning.
2. **Reliability-aware radar–IMU teacher** with cross-modal temporal alignment and corruption-supervised modality weights.
3. **Confidence-adaptive radar-only student** trained with:
   - hard-label supervision,
   - confidence-weighted logit distillation,
   - soft temporal token alignment,
   - batch-relation distillation,
   - class-prototype distillation,
   - subject-adversarial invariance.
4. **Reliability evaluation** using calibration, uncertainty-aware selective prediction, radar/IMU corruption tests, paired bootstrap confidence intervals, ablations, and optional five-fold subject evaluation.

The deployment model is the **radar-only student**. IMU is required only during teacher training.

## Execution plan

1. Attach the extracted mRI Kaggle dataset.
2. Select a compatible GPU accelerator.
3. Run the notebook from top to bottom.
4. Start with `QUICK_RUN=True` to verify the complete pipeline.
5. Set `QUICK_RUN=False` for the main experiment.
6. Enable `RUN_ABLATIONS` and `RUN_GROUPED_CV` only after the main run succeeds.

The default protocol is **P2 with 10 rehabilitation exercises**. The notebook inherits the corrected mRI label mapping:

- `pose_1`–`pose_10` → the 10 P2 classes;
- `free_form` and `walk` → P1-only classes;
- `T pose` and `T pose 2` → calibration/background.

In [1]:
from dataclasses import dataclass, asdict
from pathlib import Path

@dataclass
class CFG:
    # Data
    DATA_ROOT: str | None = None
    WORK_DIR: str = "/kaggle/working/rtpdnet_mri"
    PROTOCOL: str = "P2_10"
    SEQ_LEN: int = 96
    MIN_SEGMENT_FRACTION: float = 0.001
    NUMERIC_LABEL_MODE: str = "auto"
    EXTRA_BACKGROUND_NUMERIC_IDS: tuple = (-1,)
    MIN_TOTAL_MANIFEST_SEGMENTS: int = 50
    MIN_SUBJECTS_PER_CLASS: int = 3
    MAX_SEGMENTS_PER_CLASS_PER_SUBJECT: int | None = None

    # Split
    TEST_SUBJECT_FRACTION: float = 0.20
    VAL_SUBJECT_FRACTION_OF_REMAINDER: float = 0.25
    SPLIT_SEED: int = 2026
    SPLIT_SEARCH_ATTEMPTS: int = 50000

    # Loader
    BATCH_SIZE: int = 24
    NUM_WORKERS: int = 2
    CACHE_SUBJECTS_PER_WORKER: int = 3
    USE_WEIGHTED_SAMPLER: bool = False
    STAT_MAX_FRAMES_PER_SUBJECT: int = 4000

    # Architecture
    EMBED_DIM: int = 128
    TCN_CHANNELS: tuple = (128, 128, 128, 128)
    TCN_DILATIONS: tuple = (1, 2, 4, 8)
    DROPOUT: float = 0.25
    NUM_ATTENTION_HEADS: int = 4
    PROJECTION_DIM: int = 96

    # Optimisation
    LEARNING_RATE: float = 3e-4
    SSL_LEARNING_RATE: float = 4e-4
    WEIGHT_DECAY: float = 1e-4
    PATIENCE: int = 5
    GRAD_CLIP: float = 5.0
    LABEL_SMOOTHING: float = 0.03

    # Stages
    RUN_SSL: bool = True
    RUN_STRONG_BASELINES: bool = True
    SSL_EPOCHS: int = 10
    STRONG_BASELINE_EPOCHS: int = 14
    BASELINE_EPOCHS: int = 14
    TEACHER_EPOCHS: int = 20
    STUDENT_EPOCHS: int = 22

    # SSL
    SSL_MASK_PROB: float = 0.30
    SSL_CONTRASTIVE_TEMP: float = 0.15
    SSL_RECON_WEIGHT: float = 1.0
    SSL_CONTRASTIVE_WEIGHT: float = 0.35

    # Teacher losses
    TEACHER_AUX_WEIGHT: float = 0.20
    RELIABILITY_WEIGHT: float = 0.30
    CONSISTENCY_WEIGHT: float = 0.20
    CORRUPTION_PROB: float = 0.85

    # Student losses
    KD_TEMPERATURE: float = 4.0
    KD_LOGIT_WEIGHT: float = 0.90
    KD_TEMPORAL_WEIGHT: float = 0.45
    KD_RELATION_WEIGHT: float = 0.20
    KD_PROTOTYPE_WEIGHT: float = 0.45
    SUBJECT_ADV_WEIGHT: float = 0.08
    TEMPORAL_ALIGNMENT_TEMP: float = 0.10
    PROTOTYPE_TEMP: float = 0.12

    # Evaluation
    AMP: bool = True
    CALIBRATION_BINS: int = 10
    BOOTSTRAP_SAMPLES: int = 1000
    RUN_ROBUSTNESS: bool = True
    RUN_ABLATIONS: bool = False
    ABLATION_EPOCHS: int = 12
    RUN_GROUPED_CV: bool = False
    CV_USE_SSL: bool = True
    CV_SSL_EPOCHS: int = 4
    CV_EPOCHS: int = 8
    CV_FOLDS: int = 5

    # Runtime
    REQUIRE_GPU: bool = True
    QUICK_RUN: bool = False
    SEED: int = 2026

cfg = CFG()

if cfg.QUICK_RUN:
    cfg.SSL_EPOCHS = 2
    cfg.STRONG_BASELINE_EPOCHS = 2
    cfg.BASELINE_EPOCHS = 2
    cfg.TEACHER_EPOCHS = 2
    cfg.STUDENT_EPOCHS = 2
    cfg.ABLATION_EPOCHS = 2
    cfg.CV_SSL_EPOCHS = 1
    cfg.CV_EPOCHS = 2
    cfg.BOOTSTRAP_SAMPLES = 200

Path(cfg.WORK_DIR).mkdir(parents=True, exist_ok=True)
print(asdict(cfg))

{'DATA_ROOT': None, 'WORK_DIR': '/kaggle/working/rtpdnet_mri', 'PROTOCOL': 'P2_10', 'SEQ_LEN': 96, 'MIN_SEGMENT_FRACTION': 0.001, 'NUMERIC_LABEL_MODE': 'auto', 'EXTRA_BACKGROUND_NUMERIC_IDS': (-1,), 'MIN_TOTAL_MANIFEST_SEGMENTS': 50, 'MIN_SUBJECTS_PER_CLASS': 3, 'MAX_SEGMENTS_PER_CLASS_PER_SUBJECT': None, 'TEST_SUBJECT_FRACTION': 0.2, 'VAL_SUBJECT_FRACTION_OF_REMAINDER': 0.25, 'SPLIT_SEED': 2026, 'SPLIT_SEARCH_ATTEMPTS': 50000, 'BATCH_SIZE': 24, 'NUM_WORKERS': 2, 'CACHE_SUBJECTS_PER_WORKER': 3, 'USE_WEIGHTED_SAMPLER': False, 'STAT_MAX_FRAMES_PER_SUBJECT': 4000, 'EMBED_DIM': 128, 'TCN_CHANNELS': (128, 128, 128, 128), 'TCN_DILATIONS': (1, 2, 4, 8), 'DROPOUT': 0.25, 'NUM_ATTENTION_HEADS': 4, 'PROJECTION_DIM': 96, 'LEARNING_RATE': 0.0003, 'SSL_LEARNING_RATE': 0.0004, 'WEIGHT_DECAY': 0.0001, 'PATIENCE': 5, 'GRAD_CLIP': 5.0, 'LABEL_SMOOTHING': 0.03, 'RUN_SSL': True, 'RUN_STRONG_BASELINES': True, 'SSL_EPOCHS': 10, 'STRONG_BASELINE_EPOCHS': 14, 'BASELINE_EPOCHS': 14, 'TEACHER_EPOCHS': 20, 'STU

In [2]:
# ============================================================
# Environment and imports with Kaggle Tesla P100 compatibility
# ============================================================

import os
import sys
import gc
import re
import json
import copy
import math
import pickle
import random
import shutil
import signal
import warnings
import subprocess
from collections import Counter, defaultdict, OrderedDict
from contextlib import nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    log_loss,
)
from sklearn.model_selection import GroupShuffleSplit


# ------------------------------------------------------------
# P100 compatibility bootstrap
# ------------------------------------------------------------

def install_p100_compatible_torch():
    """
    Kaggle sometimes installs torch + cu128/cu129 wheels that do not include
    sm_60 kernels. Tesla P100 is sm_60, so CUDA tensor creation fails with:
    'no kernel image is available for execution on the device'.

    This installs a CUDA 12.6 PyTorch wheel that is compatible with P100,
    then restarts the current kernel. After restart, run the notebook again.
    """
    print("\nDetected incompatible PyTorch CUDA build for Tesla P100.")
    print("Installing PyTorch 2.8.0 + CUDA 12.6 wheels...")
    print("Kaggle Internet must be ON for this installation step.\n")

    commands = [
        [
            sys.executable,
            "-m",
            "pip",
            "uninstall",
            "-y",
            "torch",
            "torchvision",
            "torchaudio",
        ],
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--force-reinstall",
            "torch==2.8.0",
            "torchvision==0.23.0",
            "torchaudio==2.8.0",
            "--index-url",
            "https://download.pytorch.org/whl/cu126",
        ],
    ]

    for command in commands:
        print("Running:", " ".join(command))
        subprocess.check_call(command)

    print("\nPyTorch was reinstalled successfully.")
    print("Restarting the kernel now. After restart, run the notebook from the top.\n")

    os.kill(os.getpid(), signal.SIGKILL)


try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

except Exception as import_error:
    print("Torch import failed before training starts.")
    print("Error:", repr(import_error))
    install_p100_compatible_torch()


def torch_build_is_incompatible_with_p100():
    """
    Returns True when the current visible GPU is Tesla P100/sm_60 but the
    installed torch wheel does not contain sm_60 kernels.
    """
    if not torch.cuda.is_available():
        return False

    capability = torch.cuda.get_device_capability(0)

    try:
        arch_list = torch.cuda.get_arch_list()
    except Exception:
        arch_list = []

    is_p100_like = capability == (6, 0)
    has_sm60 = ("sm_60" in arch_list) or ("compute_60" in arch_list)

    return is_p100_like and (not has_sm60)


if torch_build_is_incompatible_with_p100():
    print("Current PyTorch:", torch.__version__)
    print("Current PyTorch CUDA build:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))

    try:
        print("Compiled CUDA architectures:", torch.cuda.get_arch_list())
    except Exception:
        pass

    install_p100_compatible_torch()


# ------------------------------------------------------------
# Safe normal environment setup after compatibility check
# ------------------------------------------------------------

warnings.filterwarnings("ignore", category=UserWarning)


def seed_everything(seed=2026):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.deterministic = False


seed_everything(cfg.SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if cfg.REQUIRE_GPU and DEVICE.type != "cuda":
    raise RuntimeError(
        "A CUDA GPU is required by cfg.REQUIRE_GPU=True."
    )

print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("Selected device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))

    try:
        print("Compiled CUDA architectures:", torch.cuda.get_arch_list())
    except Exception:
        pass

    # Final CUDA smoke test.
    # This must pass before any model training begins.
    probe = torch.randn(2, 5, 14, 14, device=DEVICE)
    probe_layer = nn.Conv2d(5, 8, 3, padding=1).to(DEVICE)

    with torch.inference_mode():
        probe_output = probe_layer(probe)

    torch.cuda.synchronize()

    print("CUDA tensor creation:", tuple(probe.shape))
    print("CUDA convolution output:", tuple(probe_output.shape))
    print("CUDA convolution smoke test passed.")

    del probe, probe_layer, probe_output
    torch.cuda.empty_cache()

else:
    print("Warning: training will be substantially slower on CPU.")

PyTorch: 2.8.0+cu126
PyTorch CUDA build: 12.6
Selected device: cuda:0
GPU: Tesla P100-PCIE-16GB
Capability: (6, 0)
Compiled CUDA architectures: ['sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']
CUDA tensor creation: (2, 5, 14, 14)
CUDA convolution output: (2, 8, 14, 14)
CUDA convolution smoke test passed.


In [4]:
from pathlib import Path

def looks_like_mri_root(path: Path) -> bool:
    return (
        path.exists()
        and path.is_dir()
        and (path / "features" / "radar").exists()
        and (path / "features" / "imu").exists()
        and (path / "aligned_data").exists()
        and (path / "raw_data").exists()
    )

def discover_mri_root(search_roots=None) -> Path | None:
    if cfg.DATA_ROOT is not None:
        configured = Path(cfg.DATA_ROOT)
        if looks_like_mri_root(configured):
            print("Using configured DATA_ROOT:", configured)
            return configured
        raise FileNotFoundError(
            f"Configured DATA_ROOT is not valid: {configured}\n"
            "It must directly contain features/, aligned_data/, and raw_data/."
        )

    exact_candidates = [
        Path("/kaggle/input/dataset-mri/dataset_release/dataset_release"),
        Path("/kaggle/input/dataset-mri/dataset_release"),
        Path("/kaggle/input/dataset-mri"),
    ]

    for candidate in exact_candidates:
        if looks_like_mri_root(candidate):
            print("Found mRI root:", candidate)
            return candidate

    search_roots = search_roots or [
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ]

    for base in search_roots:
        if not base.exists():
            continue
        for features_dir in base.rglob("features"):
            candidate = features_dir.parent
            if looks_like_mri_root(candidate):
                print("Automatically discovered mRI root:", candidate)
                return candidate

    return None

DATA_ROOT = discover_mri_root()

if DATA_ROOT is None:
    print("Could not locate the mRI root. Shallow Kaggle input tree:")
    base = Path("/kaggle/input")
    if base.exists():
        for path in sorted(base.rglob("*")):
            try:
                rel = path.relative_to(base)
                if len(rel.parts) <= 5:
                    print("  " * (len(rel.parts) - 1) + rel.name + ("/" if path.is_dir() else ""))
            except Exception:
                pass

    raise FileNotFoundError(
        "The extracted mRI dataset could not be located. "
        "Set cfg.DATA_ROOT to the directory that directly contains "
        "aligned_data/, features/, model/, and raw_data/."
    )

RADAR_DIR = DATA_ROOT / "features" / "radar"
IMU_DIR = DATA_ROOT / "features" / "imu"
POSE_LABEL_DIR = DATA_ROOT / "aligned_data" / "pose_labels"
RAW_VIDEO_LABEL_DIR = DATA_ROOT / "raw_data" / "videolabels"

print("\nDataset root:", DATA_ROOT)
print("Radar features:", RADAR_DIR)
print("IMU features:", IMU_DIR)
print("Pose labels:", POSE_LABEL_DIR)
print("Raw video labels:", RAW_VIDEO_LABEL_DIR)

Automatically discovered mRI root: /kaggle/input/datasets/hrithikmajumdar/dataset-mri/dataset_release/dataset_release

Dataset root: /kaggle/input/datasets/hrithikmajumdar/dataset-mri/dataset_release/dataset_release
Radar features: /kaggle/input/datasets/hrithikmajumdar/dataset-mri/dataset_release/dataset_release/features/radar
IMU features: /kaggle/input/datasets/hrithikmajumdar/dataset-mri/dataset_release/dataset_release/features/imu
Pose labels: /kaggle/input/datasets/hrithikmajumdar/dataset-mri/dataset_release/dataset_release/aligned_data/pose_labels
Raw video labels: /kaggle/input/datasets/hrithikmajumdar/dataset-mri/dataset_release/dataset_release/raw_data/videolabels


In [5]:
P1_CLASS_NAMES = [
    "left upper-limb extension",
    "right upper-limb extension",
    "both upper-limb extension",
    "left front lunge",
    "right front lunge",
    "squat",
    "left side lunge",
    "right side lunge",
    "left limb extension",
    "right limb extension",
    "free-form stretching/relaxing",
    "walking",
]
P2_CLASS_NAMES = P1_CLASS_NAMES[:10]
CLASS_NAMES = P2_CLASS_NAMES if cfg.PROTOCOL == "P2_10" else P1_CLASS_NAMES
NUM_CLASSES = len(CLASS_NAMES)

def normalise_text(x):
    x = str(x).strip().lower().replace("_", " ").replace("-", " ")
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    return re.sub(r"\s+", " ", x).strip()

ALIASES = {
    "left upper limb extension": 0,
    "left upper limbs extension": 0,
    "right upper limb extension": 1,
    "right upper limbs extension": 1,
    "both upper limb extension": 2,
    "both upper limbs extension": 2,
    "left front lunge": 3,
    "left front lung": 3,
    "right front lunge": 4,
    "right front lung": 4,
    "squat": 5,
    "left side lunge": 6,
    "left side lung": 6,
    "right side lunge": 7,
    "right side lung": 7,
    "left limb extension": 8,
    "left lower limb extension": 8,
    "right limb extension": 9,
    "right lower limb extension": 9,
    "free form stretching relaxing": 10,
    "free form stretch relax": 10,
    "stretching relaxing": 10,
    "relaxing stretching": 10,
    "walking": 11,
    "walk": 11,
}
BACKGROUND_ALIASES = {
    "", "none", "nan", "background", "bg", "transition", "idle",
    "t pose", "tpose", "calibration", "unknown", "other"
}

print(f"Protocol: {cfg.PROTOCOL} | classes: {NUM_CLASSES}")
for i, name in enumerate(CLASS_NAMES):
    print(f"{i:02d}: {name}")

Protocol: P2_10 | classes: 10
00: left upper-limb extension
01: right upper-limb extension
02: both upper-limb extension
03: left front lunge
04: right front lunge
05: squat
06: left side lunge
07: right side lunge
08: left limb extension
09: right limb extension


In [6]:
def safe_pickle_load(path):
    with open(path, "rb") as handle:
        return pickle.load(handle)

def load_any(path):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".npy":
        return np.load(path, mmap_mode="r")
    if suffix in {".pt", ".pth"}:
        return torch.load(path, map_location="cpu", weights_only=False)
    if suffix in {".cpl", ".pkl", ".pickle"}:
        return safe_pickle_load(path)
    if suffix == ".json":
        with open(path, "r", encoding="utf-8") as handle:
            return json.load(handle)
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".txt", ".tsv"}:
        try:
            return pd.read_csv(path, sep=None, engine="python")
        except Exception:
            return path.read_text(errors="ignore")

    raise ValueError(f"Unsupported file: {path}")

def summarise_object(obj, depth=0, max_depth=3):
    indent = "  " * depth
    if depth > max_depth:
        return f"{indent}..."

    if isinstance(obj, dict):
        lines = [f"{indent}dict(keys={list(obj.keys())[:30]})"]
        for key, value in list(obj.items())[:8]:
            lines.append(
                f"{indent}- {key!r}: "
                f"{summarise_object(value, depth + 1, max_depth)}"
            )
        return "\n".join(lines)

    if isinstance(obj, pd.DataFrame):
        return f"{indent}DataFrame(shape={obj.shape}, columns={list(obj.columns)})"

    if torch.is_tensor(obj):
        preview = obj.reshape(-1)[:5].detach().cpu().tolist() if obj.numel() else []
        return (
            f"{indent}Tensor(shape={tuple(obj.shape)}, dtype={obj.dtype}, "
            f"preview={preview})"
        )

    if isinstance(obj, np.ndarray):
        preview = obj.reshape(-1)[:5].tolist() if obj.size else []
        return (
            f"{indent}ndarray(shape={obj.shape}, dtype={obj.dtype}, "
            f"preview={preview})"
        )

    if isinstance(obj, (list, tuple)):
        first = summarise_object(obj[0], depth + 1, max_depth) if obj else "empty"
        return f"{indent}{type(obj).__name__}(len={len(obj)}, first=\n{first})"

    return f"{indent}{type(obj).__name__}: {str(obj)[:300]}"

def extract_subject_id(path) -> str:
    text = str(path).lower()

    patterns = [
        r"subject[\s_\-]*0*(\d+)",
        r"subj(?:ect)?[\s_\-]*0*(\d+)",
        r"participant[\s_\-]*0*(\d+)",
        r"(?:^|[/_\-])s0*(\d+)(?:[/_\-.]|$)",
    ]

    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return f"subject{int(match.group(1)):02d}"

    numbers = re.findall(r"(?<!\d)(\d{1,2})(?!\d)", Path(path).stem)
    if numbers:
        return f"subject{int(numbers[-1]):02d}"

    return Path(path).parent.name.lower().replace(" ", "_")

def choose_subject_file_map(paths, modality_name):
    grouped = defaultdict(list)
    for path in paths:
        grouped[extract_subject_id(path)].append(Path(path))

    selected = {}
    for subject, candidates in sorted(grouped.items()):
        candidates = sorted(candidates, key=lambda p: (len(str(p)), str(p)))
        selected[subject] = candidates[0]
        if len(candidates) > 1:
            print(
                f"Warning: {modality_name} has {len(candidates)} files for {subject}; "
                f"using {candidates[0]}"
            )
    return selected

def collect_files():
    radar_files = sorted(RADAR_DIR.rglob("*featuremap.npy"))
    if not radar_files:
        radar_files = sorted(RADAR_DIR.rglob("*.npy"))

    imu_files = sorted(IMU_DIR.rglob("acc_ori.pt"))
    if not imu_files:
        imu_files = sorted(IMU_DIR.rglob("*.pt"))

    pose_label_files = []
    if POSE_LABEL_DIR.exists():
        for extension in ("*.cpl", "*.pkl", "*.pickle", "*.json", "*.npy"):
            pose_label_files.extend(POSE_LABEL_DIR.rglob(extension))

    raw_label_files = []
    if RAW_VIDEO_LABEL_DIR.exists():
        for extension in (
            "*.csv", "*.txt", "*.tsv", "*.json",
            "*.npy", "*.cpl", "*.pkl", "*.pickle"
        ):
            raw_label_files.extend(RAW_VIDEO_LABEL_DIR.rglob(extension))

    return (
        sorted(set(radar_files)),
        sorted(set(imu_files)),
        sorted(set(pose_label_files)),
        sorted(set(raw_label_files)),
    )

radar_files, imu_files, pose_label_files, raw_label_files = collect_files()

radar_by_subject = choose_subject_file_map(radar_files, "radar")
imu_by_subject = choose_subject_file_map(imu_files, "IMU")
pose_label_by_subject = choose_subject_file_map(pose_label_files, "pose-label")
raw_label_by_subject = choose_subject_file_map(raw_label_files, "raw-label")

label_by_subject = dict(raw_label_by_subject)
label_by_subject.update(pose_label_by_subject)

common_subjects = sorted(
    set(radar_by_subject)
    & set(imu_by_subject)
    & set(label_by_subject)
)

print("Radar files:", len(radar_files))
print("IMU files:", len(imu_files))
print("Pose-label files:", len(pose_label_files))
print("Raw video-label files:", len(raw_label_files))
print("Matched subjects:", len(common_subjects))
print(common_subjects)

if not common_subjects:
    print("Radar subject keys:", sorted(radar_by_subject))
    print("IMU subject keys:", sorted(imu_by_subject))
    print("Label subject keys:", sorted(label_by_subject))
    raise RuntimeError(
        "No subjects matched across radar, IMU, and labels. "
        "Inspect filenames and update extract_subject_id()."
    )

sample_subject = common_subjects[0]
sample_radar = load_any(radar_by_subject[sample_subject])
sample_imu = load_any(imu_by_subject[sample_subject])
sample_label_obj = load_any(label_by_subject[sample_subject])

print("\nSample subject:", sample_subject)
print("Radar file:", radar_by_subject[sample_subject])
print("IMU file:", imu_by_subject[sample_subject])
print("Label file:", label_by_subject[sample_subject])
print("\nRadar object:\n", summarise_object(sample_radar))
print("\nIMU object:\n", summarise_object(sample_imu))
print("\nLabel object:\n", summarise_object(sample_label_obj))

Radar files: 20
IMU files: 20
Pose-label files: 20
Raw video-label files: 20
Matched subjects: 20
['subject01', 'subject02', 'subject03', 'subject04', 'subject05', 'subject06', 'subject07', 'subject08', 'subject09', 'subject10', 'subject11', 'subject12', 'subject13', 'subject14', 'subject15', 'subject16', 'subject17', 'subject18', 'subject19', 'subject20']

Sample subject: subject01
Radar file: /kaggle/input/datasets/hrithikmajumdar/dataset-mri/dataset_release/dataset_release/features/radar/subject1_featuremap.npy
IMU file: /kaggle/input/datasets/hrithikmajumdar/dataset-mri/dataset_release/dataset_release/features/imu/subject1/acc_ori.pt
Label file: /kaggle/input/datasets/hrithikmajumdar/dataset-mri/dataset_release/dataset_release/aligned_data/pose_labels/subject1_all_labels.cpl

Radar object:
 ndarray(shape=(6384, 14, 14, 5), dtype=float64, preview=[-0.86035, 2.582, 0.41699, 0.0, -0.6087622791159407])

IMU object:
 Tensor(shape=(6384, 6, 12), dtype=torch.float32, preview=[-0.010256187

## Annotation-coordinate correction

The release includes action boundaries expressed as timestamps. When those are
absolute wall/Unix timestamps, they must first be shifted to recording-relative
coordinates. Otherwise, a 30–60 second action divided by a timestamp near
`1.6e9` or `1.6e12` appears to have almost zero duration and is rejected.

The parser now detects this condition, subtracts the first annotation timestamp,
and prints the resulting duration-fraction audit.

In [7]:
START_NAMES = {
    "start", "begin", "onset", "start frame", "start time",
    "startframe", "starttime", "start index", "start idx"
}
END_NAMES = {
    "end", "stop", "offset", "end frame", "end time",
    "endframe", "endtime", "end index", "end idx"
}
LABEL_NAMES = {
    "label", "action", "activity", "class", "name", "value",
    "action label", "activity label", "class label"
}

def labels_equal(left, right):
    try:
        if pd.isna(left) and pd.isna(right):
            return True
    except Exception:
        pass
    try:
        return bool(left == right)
    except Exception:
        return False

def is_number(value):
    if isinstance(value, (int, float, np.integer, np.floating)):
        return not pd.isna(value)
    if isinstance(value, str):
        try:
            float(value.strip())
            return True
        except Exception:
            return False
    return False

def is_scalar_label(value):
    return isinstance(
        value,
        (str, int, float, np.integer, np.floating, bool, np.bool_)
    )

def get_video_label_object(label_object):
    if isinstance(label_object, dict):
        for key, value in label_object.items():
            if normalise_text(key) in {
                "video label", "video labels", "videolabel",
                "action label", "action labels", "activity labels"
            }:
                return value
    return label_object

def finalise_segment_scale(segments, explicit_scale=None):
    """
    Clean, deduplicate, and convert interval coordinates to a proportional
    recording-relative system.

    Important:
    mRI annotations may use absolute wall/Unix timestamps. Dividing an action
    duration directly by a Unix timestamp makes every action fraction nearly
    zero. When coordinates look absolute, this function subtracts the first
    annotation timestamp before calculating the scale.
    """
    clean = []

    for segment in segments:
        try:
            start = float(segment["start"])
            end = float(segment["end"])
        except Exception:
            continue

        if not np.isfinite(start) or not np.isfinite(end) or end <= start:
            continue

        clean.append({
            "raw_label": segment["raw_label"],
            "start": start,
            "end": end,
            "scale": segment.get("scale"),
            "source": segment.get("source", "unknown"),
        })

    if not clean:
        return []

    minimum_start = min(item["start"] for item in clean)
    maximum_end = max(item["end"] for item in clean)

    if explicit_scale is not None:
        try:
            explicit_scale = float(explicit_scale)
            if np.isfinite(explicit_scale):
                maximum_end = max(maximum_end, explicit_scale)
        except Exception:
            pass

    raw_span = max(maximum_end - minimum_start, 1e-9)
    absolute_reference = max(abs(maximum_end), 1.0)
    relative_span_ratio = raw_span / absolute_reference

    # Detect wall/Unix-like coordinates. Examples:
    # seconds:      ~1.6e9
    # milliseconds: ~1.6e12
    # microseconds: ~1.6e15
    #
    # A large coordinate with a duration that is tiny relative to the
    # coordinate magnitude is almost certainly an absolute timestamp.
    absolute_like = (
        abs(minimum_start) >= 1e6
        and relative_span_ratio < 0.25
    ) or (
        abs(minimum_start) > 1000.0 * raw_span
    )

    origin = minimum_start if absolute_like else 0.0
    scale = maximum_end - origin

    if scale <= 0 or not np.isfinite(scale):
        raise ValueError(
            f"Invalid annotation scale after normalization: "
            f"origin={origin}, maximum_end={maximum_end}, scale={scale}"
        )

    normalised = []

    for item in clean:
        shifted_start = item["start"] - origin
        shifted_end = item["end"] - origin

        shifted_start = max(0.0, shifted_start)
        shifted_end = min(scale, shifted_end)

        if shifted_end <= shifted_start:
            continue

        normalised.append({
            "raw_label": item["raw_label"],
            "start": shifted_start,
            "end": shifted_end,
            "scale": scale,
            "source": item["source"],
            "coordinate_origin": origin,
            "absolute_timestamp_detected": absolute_like,
        })

    deduplicated = {}

    for item in normalised:
        key = (
            str(item["raw_label"]),
            round(item["start"], 8),
            round(item["end"], 8),
        )
        deduplicated[key] = item

    output = sorted(
        deduplicated.values(),
        key=lambda item: (
            item["start"],
            item["end"],
            str(item["raw_label"]),
        ),
    )

    if output:
        durations = [
            (item["end"] - item["start"]) / item["scale"]
            for item in output
        ]

        print(
            "Annotation coordinate audit:",
            {
                "intervals": len(output),
                "absolute_timestamp_detected": absolute_like,
                "origin": origin,
                "scale": scale,
                "minimum_duration_fraction": float(min(durations)),
                "median_duration_fraction": float(np.median(durations)),
                "maximum_duration_fraction": float(max(durations)),
            },
        )

    return output


def framewise_labels_to_segments(labels, positions=None, source="framewise"):
    labels = list(labels)
    if not labels:
        return []

    if positions is None:
        positions = np.arange(len(labels), dtype=np.float64)
    else:
        positions = np.asarray(positions, dtype=np.float64)

    if len(positions) != len(labels):
        raise ValueError(
            f"Positions length {len(positions)} != labels length {len(labels)}"
        )

    order = np.argsort(positions)
    positions = positions[order]
    labels = [labels[index] for index in order]

    if len(positions) > 1:
        steps = np.diff(positions)
        steps = steps[steps > 0]
        step = float(np.median(steps)) if len(steps) else 1.0
    else:
        step = 1.0

    total_scale = float(positions[-1] + step)
    segments = []
    run_start = 0
    current = labels[0]

    for index in range(1, len(labels)):
        if not labels_equal(labels[index], current):
            segments.append({
                "raw_label": current,
                "start": float(positions[run_start]),
                "end": float(positions[index]),
                "scale": total_scale,
                "source": source,
            })
            run_start = index
            current = labels[index]

    segments.append({
        "raw_label": current,
        "start": float(positions[run_start]),
        "end": total_scale,
        "scale": total_scale,
        "source": source,
    })

    return finalise_segment_scale(segments, explicit_scale=total_scale)

def find_named_key(mapping, candidate_names):
    normalised = {normalise_text(key): key for key in mapping.keys()}
    return next(
        (
            original
            for cleaned, original in normalised.items()
            if cleaned in candidate_names
        ),
        None,
    )

def parse_interval_record(record, inherited_label=None, source="interval_record"):
    if not isinstance(record, dict):
        return None

    start_key = find_named_key(record, START_NAMES)
    end_key = find_named_key(record, END_NAMES)
    label_key = find_named_key(record, LABEL_NAMES)

    if start_key is None or end_key is None:
        return None

    label = record[label_key] if label_key is not None else inherited_label
    if label is None:
        return None

    try:
        start = float(record[start_key])
        end = float(record[end_key])
    except Exception:
        return None

    if end <= start:
        return None

    return {
        "raw_label": label,
        "start": start,
        "end": end,
        "scale": None,
        "source": source,
    }

def interval_pairs_for_label(raw_label, value, source="label_intervals"):
    segments = []

    if torch.is_tensor(value):
        value = value.detach().cpu().numpy()

    if isinstance(value, dict):
        start_key = find_named_key(value, START_NAMES)
        end_key = find_named_key(value, END_NAMES)

        if start_key is not None and end_key is not None:
            starts = value[start_key]
            ends = value[end_key]

            starts = [starts] if np.isscalar(starts) else list(starts)
            ends = [ends] if np.isscalar(ends) else list(ends)

            for start, end in zip(starts, ends):
                if is_number(start) and is_number(end) and float(end) > float(start):
                    segments.append({
                        "raw_label": raw_label,
                        "start": float(start),
                        "end": float(end),
                        "scale": None,
                        "source": source,
                    })
            return segments

        for nested in value.values():
            segments.extend(interval_pairs_for_label(raw_label, nested, source))
        return segments

    if isinstance(value, (list, tuple, np.ndarray)):
        array = np.asarray(value, dtype=object)

        if (
            array.ndim == 1
            and len(array) >= 2
            and is_number(array[0])
            and is_number(array[1])
        ):
            start, end = float(array[0]), float(array[1])
            if end > start:
                segments.append({
                    "raw_label": raw_label,
                    "start": start,
                    "end": end,
                    "scale": None,
                    "source": source,
                })
            return segments

        if array.ndim == 2 and array.shape[1] >= 2:
            for row in array:
                if is_number(row[0]) and is_number(row[1]):
                    start, end = float(row[0]), float(row[1])
                    if end > start:
                        segments.append({
                            "raw_label": raw_label,
                            "start": start,
                            "end": end,
                            "scale": None,
                            "source": source,
                        })
            return segments

        for nested in list(value):
            segments.extend(interval_pairs_for_label(raw_label, nested, source))

    return segments

def parse_interval_matrix(array, source="interval_matrix"):
    array = np.asarray(array, dtype=object)
    if array.ndim != 2 or array.shape[1] < 3:
        return []

    layouts = [(0, 1, 2), (2, 0, 1)]
    candidates = []

    for label_col, start_col, end_col in layouts:
        parsed = []
        for row in array:
            if (
                is_number(row[start_col])
                and is_number(row[end_col])
                and float(row[end_col]) > float(row[start_col])
            ):
                parsed.append({
                    "raw_label": row[label_col],
                    "start": float(row[start_col]),
                    "end": float(row[end_col]),
                    "scale": None,
                    "source": source,
                })

        unique_labels = len({str(item["raw_label"]) for item in parsed})
        score = len(parsed) + (10 if 1 <= unique_labels <= 20 else 0)
        candidates.append((score, parsed))

    return max(candidates, key=lambda item: item[0])[1] if candidates else []

def extract_video_label_segments(label_object):
    video_label = get_video_label_object(label_object)

    if torch.is_tensor(video_label):
        video_label = video_label.detach().cpu().numpy()

    if isinstance(video_label, pd.DataFrame):
        records = []
        for row in video_label.to_dict("records"):
            parsed = parse_interval_record(row, source="dataframe_interval")
            if parsed is not None:
                records.append(parsed)
        if records:
            return finalise_segment_scale(records)

        if video_label.shape[1] == 1:
            return framewise_labels_to_segments(
                video_label.iloc[:, 0].tolist(),
                source="dataframe_framewise",
            )

        video_label = video_label.to_numpy(dtype=object)

    if isinstance(video_label, dict):
        label_key = find_named_key(video_label, LABEL_NAMES)
        start_key = find_named_key(video_label, START_NAMES)
        end_key = find_named_key(video_label, END_NAMES)

        if label_key is not None and start_key is not None and end_key is not None:
            labels = video_label[label_key]
            starts = video_label[start_key]
            ends = video_label[end_key]
            segments = []
            for label, start, end in zip(labels, starts, ends):
                if is_number(start) and is_number(end) and float(end) > float(start):
                    segments.append({
                        "raw_label": label,
                        "start": float(start),
                        "end": float(end),
                        "scale": None,
                        "source": "parallel_arrays",
                    })
            if segments:
                return finalise_segment_scale(segments)

        if video_label and all(is_number(key) for key in video_label.keys()):
            if all(is_scalar_label(value) for value in video_label.values()):
                ordered = sorted(video_label.items(), key=lambda item: float(item[0]))
                return framewise_labels_to_segments(
                    labels=[value for _, value in ordered],
                    positions=[float(key) for key, _ in ordered],
                    source="dictionary_framewise",
                )

        interval_records = []
        for key, value in video_label.items():
            parsed = parse_interval_record(
                value,
                inherited_label=key,
                source="dictionary_interval_record",
            )
            if parsed is not None:
                interval_records.append(parsed)
        if interval_records:
            return finalise_segment_scale(interval_records)

        labelled_intervals = []
        for raw_label, value in video_label.items():
            labelled_intervals.extend(
                interval_pairs_for_label(
                    raw_label,
                    value,
                    source="label_to_intervals",
                )
            )
        if labelled_intervals:
            return finalise_segment_scale(labelled_intervals)

        recursive_segments = []
        for value in video_label.values():
            if isinstance(value, (dict, list, tuple, np.ndarray, pd.DataFrame)):
                try:
                    recursive_segments.extend(extract_video_label_segments(value))
                except Exception:
                    pass
        if recursive_segments:
            return finalise_segment_scale(recursive_segments)

        raise ValueError(
            "Unrecognised video_label dictionary schema. "
            f"Example keys: {list(video_label.keys())[:20]}"
        )

    if isinstance(video_label, (list, tuple)):
        if len(video_label) == 0:
            return []

        if all(is_scalar_label(value) for value in video_label):
            return framewise_labels_to_segments(
                video_label,
                source="list_framewise",
            )

        interval_records = []
        for item in video_label:
            parsed = parse_interval_record(
                item,
                source="list_interval_record",
            )
            if parsed is not None:
                interval_records.append(parsed)
        if interval_records:
            return finalise_segment_scale(interval_records)

        array = np.asarray(video_label, dtype=object)
        parsed_matrix = parse_interval_matrix(array, source="list_interval_matrix")
        if parsed_matrix:
            return finalise_segment_scale(parsed_matrix)

        recursive_segments = []
        for item in video_label:
            if isinstance(item, (dict, list, tuple, np.ndarray, pd.DataFrame)):
                try:
                    recursive_segments.extend(extract_video_label_segments(item))
                except Exception:
                    pass
        if recursive_segments:
            return finalise_segment_scale(recursive_segments)

    if isinstance(video_label, np.ndarray):
        if video_label.ndim == 0:
            return []

        if video_label.ndim == 1:
            return framewise_labels_to_segments(
                video_label.tolist(),
                source="array_framewise",
            )

        if video_label.ndim == 2:
            if 1 in video_label.shape:
                return framewise_labels_to_segments(
                    video_label.reshape(-1).tolist(),
                    source="array_framewise",
                )

            if video_label.shape[1] == 2:
                first_numeric = all(is_number(value) for value in video_label[:, 0])
                second_numeric = all(is_number(value) for value in video_label[:, 1])

                if first_numeric and not second_numeric:
                    return framewise_labels_to_segments(
                        labels=video_label[:, 1].tolist(),
                        positions=[float(value) for value in video_label[:, 0]],
                        source="array_time_label",
                    )
                if second_numeric and not first_numeric:
                    return framewise_labels_to_segments(
                        labels=video_label[:, 0].tolist(),
                        positions=[float(value) for value in video_label[:, 1]],
                        source="array_label_time",
                    )

            parsed = parse_interval_matrix(video_label, source="array_interval_matrix")
            if parsed:
                return finalise_segment_scale(parsed)

    raise ValueError(
        f"Unsupported video_label structure: {type(video_label)}"
    )

sample_segments = extract_video_label_segments(sample_label_obj)
print("Parsed sample intervals:", len(sample_segments))
display(pd.DataFrame(sample_segments).head(30))

if len(sample_segments) <= 1:
    print(
        "Warning: only one interval was parsed for the sample subject. "
        "Review the printed label-object schema before continuing."
    )

Annotation coordinate audit: {'intervals': 13, 'absolute_timestamp_detected': False, 'origin': 0.0, 'scale': 6481.0, 'minimum_duration_fraction': 0.006326184230828576, 'median_duration_fraction': 0.0793087486498997, 'maximum_duration_fraction': 0.11124826415676593}
Parsed sample intervals: 13


,raw_label,start,end,scale,source,coordinate_origin,absolute_timestamp_detected
0,T pose,98.0,139.0,6481.0,label_to_intervals,0.0,False
1,pose_1,169.0,640.0,6481.0,label_to_intervals,0.0,False
2,pose_2,640.0,1122.0,6481.0,label_to_intervals,0.0,False
3,pose_3,1122.0,1661.0,6481.0,label_to_intervals,0.0,False
4,pose_4,1661.0,2173.0,6481.0,label_to_intervals,0.0,False
5,pose_5,2173.0,2692.0,6481.0,label_to_intervals,0.0,False
6,pose_6,2692.0,3206.0,6481.0,label_to_intervals,0.0,False
7,pose_7,3206.0,3723.0,6481.0,label_to_intervals,0.0,False
8,pose_8,3723.0,4253.0,6481.0,label_to_intervals,0.0,False
9,pose_9,4253.0,4744.0,6481.0,label_to_intervals,0.0,False


In [8]:
def maybe_numeric(value):
    if isinstance(value, (int, float, np.integer, np.floating)) and not pd.isna(value):
        return float(value)
    if isinstance(value, str):
        try:
            return float(value.strip())
        except Exception:
            return None
    return None

all_raw_segments = {}
parse_failures = {}
label_duration = defaultdict(float)
label_occurrences = Counter()

for subject in common_subjects:
    try:
        label_object = load_any(label_by_subject[subject])
        segments = extract_video_label_segments(label_object)
        all_raw_segments[subject] = segments

        for segment in segments:
            scale = float(segment.get("scale") or max(segment["end"], 1.0))
            duration = max(
                0.0,
                float(segment["end"]) - float(segment["start"])
            ) / max(scale, 1e-9)

            label_duration[segment["raw_label"]] += duration
            label_occurrences[segment["raw_label"]] += 1

        print(f"{subject}: {len(segments)} raw intervals")

    except Exception as error:
        parse_failures[subject] = repr(error)
        print(f"{subject}: PARSE FAILED -> {error}")

if parse_failures:
    print("\nParse failures:")
    for subject, error in parse_failures.items():
        print(subject, error)

total_raw_intervals = sum(len(items) for items in all_raw_segments.values())
print("\nTotal raw intervals:", total_raw_intervals)

raw_label_table = pd.DataFrame([
    {
        "raw_label": raw_label,
        "normalised": normalise_text(raw_label),
        "occurrences": label_occurrences[raw_label],
        "duration_fraction": duration,
        "numeric_value": maybe_numeric(raw_label),
    }
    for raw_label, duration in label_duration.items()
]).sort_values(["numeric_value", "raw_label"], na_position="last")

display(raw_label_table)

numeric_values = sorted({
    int(value)
    for value in raw_label_table["numeric_value"].dropna().tolist()
    if float(value).is_integer()
})

NUMERIC_TO_CANONICAL = {}
BACKGROUND_NUMERIC = set(int(x) for x in cfg.EXTRA_BACKGROUND_NUMERIC_IDS)

def configure_numeric_mapping(mode, values):
    values = set(values)
    mapping = {}
    background = set(BACKGROUND_NUMERIC)

    if mode == "zero_background":
        background.add(0)
        mapping = {value: value - 1 for value in range(1, 13)}
    elif mode == "one_based":
        mapping = {value: value - 1 for value in range(1, 13)}
    elif mode == "direct":
        mapping = {value: value for value in range(12)}
    elif mode == "auto":
        if set(range(13)).issubset(values):
            background.add(0)
            mapping = {value: value - 1 for value in range(1, 13)}
        elif set(range(1, 13)).issubset(values):
            mapping = {value: value - 1 for value in range(1, 13)}
        elif set(range(12)).issubset(values):
            mapping = {value: value for value in range(12)}
        elif values and min(values) >= 1 and max(values) <= 12:
            mapping = {value: value - 1 for value in values}
        elif values and min(values) >= 0 and max(values) <= 11:
            mapping = {value: value for value in values}
        else:
            print(
                "Warning: numeric label scheme is ambiguous. "
                "Set cfg.NUMERIC_LABEL_MODE manually if mapping diagnostics look wrong."
            )
            non_background = sorted(value for value in values if value not in background)
            mapping = {
                value: index
                for index, value in enumerate(non_background[:12])
            }
    else:
        raise ValueError(
            "NUMERIC_LABEL_MODE must be auto, direct, one_based, or zero_background."
        )

    return mapping, background

NUMERIC_TO_CANONICAL, BACKGROUND_NUMERIC = configure_numeric_mapping(
    cfg.NUMERIC_LABEL_MODE,
    numeric_values,
)

print("Numeric values found:", numeric_values)
print("Numeric mapping:", NUMERIC_TO_CANONICAL)
print("Numeric background IDs:", sorted(BACKGROUND_NUMERIC))

def map_raw_label(raw_label):
    numeric = maybe_numeric(raw_label)

    if numeric is not None and float(numeric).is_integer():
        numeric = int(numeric)
        if numeric in BACKGROUND_NUMERIC:
            return None
        canonical = NUMERIC_TO_CANONICAL.get(numeric)
        if canonical is None or canonical >= NUM_CLASSES:
            return None
        return canonical

    text = normalise_text(raw_label)
    if text in BACKGROUND_ALIASES:
        return None

    canonical = ALIASES.get(text)

    if canonical is None:
        matches = {
            class_id
            for alias, class_id in ALIASES.items()
            if alias and (alias in text or text in alias)
        }
        canonical = next(iter(matches)) if len(matches) == 1 else None

    if canonical is None or canonical >= NUM_CLASSES:
        return None

    return canonical

mapping_diagnostic = pd.DataFrame([
    {
        "raw_label": raw_label,
        "normalised": normalise_text(raw_label),
        "mapped_class_id": map_raw_label(raw_label),
        "mapped_class_name": (
            CLASS_NAMES[map_raw_label(raw_label)]
            if map_raw_label(raw_label) is not None
            else "IGNORED"
        ),
        "occurrences": label_occurrences[raw_label],
        "duration_fraction": label_duration[raw_label],
    }
    for raw_label in label_duration.keys()
]).sort_values(["mapped_class_id", "raw_label"], na_position="last")

display(mapping_diagnostic)
mapping_diagnostic.to_csv(
    Path(cfg.WORK_DIR) / "label_mapping_diagnostic.csv",
    index=False,
)

Annotation coordinate audit: {'intervals': 13, 'absolute_timestamp_detected': False, 'origin': 0.0, 'scale': 6481.0, 'minimum_duration_fraction': 0.006326184230828576, 'median_duration_fraction': 0.0793087486498997, 'maximum_duration_fraction': 0.11124826415676593}
subject01: 13 raw intervals
Annotation coordinate audit: {'intervals': 13, 'absolute_timestamp_detected': False, 'origin': 0.0, 'scale': 7193.0, 'minimum_duration_fraction': 0.005282913944112331, 'median_duration_fraction': 0.07562908383150285, 'maximum_duration_fraction': 0.10843875990546364}
subject02: 13 raw intervals
Annotation coordinate audit: {'intervals': 13, 'absolute_timestamp_detected': False, 'origin': 0.0, 'scale': 7039.0, 'minimum_duration_fraction': 0.0056826253729222904, 'median_duration_fraction': 0.07998295212388123, 'maximum_duration_fraction': 0.08992754652649525}
subject03: 13 raw intervals
Annotation coordinate audit: {'intervals': 13, 'absolute_timestamp_detected': False, 'origin': 0.0, 'scale': 6930.0

,raw_label,normalised,occurrences,duration_fraction,numeric_value
0,T pose,t pose,20,0.105875,None
13,T pose 2,t pose 2,13,0.036090,None
11,free_form,free form,20,1.639437,None
1,pose_1,pose 1,20,1.539689,None
10,pose_10,pose 10,20,1.585161,None
2,pose_2,pose 2,20,1.566915,None
3,pose_3,pose 3,20,1.569462,None
4,pose_4,pose 4,20,1.532832,None
5,pose_5,pose 5,20,1.586671,None
6,pose_6,pose 6,20,1.579234,None


Numeric values found: []
Numeric mapping: {}
Numeric background IDs: [-1]


,raw_label,normalised,mapped_class_id,mapped_class_name,occurrences,duration_fraction
0,T pose,t pose,None,IGNORED,20,0.105875
13,T pose 2,t pose 2,None,IGNORED,13,0.036090
11,free_form,free form,None,IGNORED,20,1.639437
1,pose_1,pose 1,None,IGNORED,20,1.539689
10,pose_10,pose 10,None,IGNORED,20,1.585161
2,pose_2,pose 2,None,IGNORED,20,1.566915
3,pose_3,pose 3,None,IGNORED,20,1.569462
4,pose_4,pose 4,None,IGNORED,20,1.532832
5,pose_5,pose 5,None,IGNORED,20,1.586671
6,pose_6,pose 6,None,IGNORED,20,1.579234


In [9]:
# ============================================================
# Build manifest with mRI-specific label mapping
# ============================================================

import re
from collections import Counter
from pathlib import Path

# ------------------------------------------------------------
# mRI label mapping
# ------------------------------------------------------------
#
# pose_1  -> left upper-limb extension
# pose_2  -> right upper-limb extension
# pose_3  -> both upper-limb extension
# pose_4  -> left front lunge
# pose_5  -> right front lunge
# pose_6  -> squat
# pose_7  -> left side lunge
# pose_8  -> right side lunge
# pose_9  -> left limb extension
# pose_10 -> right limb extension
#
# P1 additionally contains:
# free_form -> class 10
# walk      -> class 11
#
# T pose and T pose 2 are calibration/background intervals.


def normalise_mri_label(label):
    """
    Convert labels such as:
        pose_1    -> pose 1
        free_form -> free form
        T pose 2  -> t pose 2
    """
    text = str(label).strip().lower()
    text = text.replace("_", " ").replace("-", " ")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


BACKGROUND_MRI_LABELS = {
    "",
    "none",
    "nan",
    "background",
    "bg",
    "idle",
    "transition",
    "calibration",
    "t pose",
    "t pose 1",
    "t pose 2",
    "tpose",
    "tpose 1",
    "tpose 2",
}


def map_mri_label(raw_label):
    """
    Return:
        class_id, status

    status is one of:
        mapped
        background
        protocol_excluded
        unmapped
    """

    text = normalise_mri_label(raw_label)

    # --------------------------------------------------------
    # Calibration/background
    # --------------------------------------------------------
    if text in BACKGROUND_MRI_LABELS:
        return None, "background"

    # --------------------------------------------------------
    # pose_1 ... pose_10
    # --------------------------------------------------------
    pose_match = re.fullmatch(
        r"pose\s*0*(10|[1-9])",
        text,
    )

    if pose_match:
        pose_number = int(pose_match.group(1))
        class_id = pose_number - 1

        if 0 <= class_id < NUM_CLASSES:
            return class_id, "mapped"

        return None, "protocol_excluded"

    # --------------------------------------------------------
    # P1-only free-form class
    # --------------------------------------------------------
    if text in {
        "free form",
        "freeform",
        "free form stretching",
        "free form stretching relaxing",
        "stretching relaxing",
    }:
        if NUM_CLASSES > 10:
            return 10, "mapped"

        return None, "protocol_excluded"

    # --------------------------------------------------------
    # P1-only walking class
    # --------------------------------------------------------
    if text in {
        "walk",
        "walking",
        "straight line walk",
        "straight line walking",
    }:
        if NUM_CLASSES > 11:
            return 11, "mapped"

        return None, "protocol_excluded"

    # --------------------------------------------------------
    # Fall back to the earlier descriptive-name mapper
    # --------------------------------------------------------
    existing_mapper = globals().get("map_raw_label")

    if callable(existing_mapper):
        try:
            mapped_id = existing_mapper(raw_label)

            if mapped_id is not None:
                mapped_id = int(mapped_id)

                if 0 <= mapped_id < NUM_CLASSES:
                    return mapped_id, "mapped"

                return None, "protocol_excluded"

        except Exception:
            pass

    return None, "unmapped"


# ------------------------------------------------------------
# Verify the mapping before building the manifest
# ------------------------------------------------------------

mapping_preview_labels = [
    "T pose",
    "pose_1",
    "pose_2",
    "pose_3",
    "pose_4",
    "pose_5",
    "pose_6",
    "pose_7",
    "pose_8",
    "pose_9",
    "pose_10",
    "free_form",
    "walk",
    "T pose 2",
]

mapping_preview = []

for raw_label in mapping_preview_labels:
    class_id, status = map_mri_label(raw_label)

    mapping_preview.append({
        "raw_label": raw_label,
        "normalised_label": normalise_mri_label(raw_label),
        "status": status,
        "class_id": class_id,
        "class_name": (
            CLASS_NAMES[class_id]
            if class_id is not None
            else None
        ),
    })

print("mRI label-mapping preview:")
display(pd.DataFrame(mapping_preview))


# ------------------------------------------------------------
# Build manifest
# ------------------------------------------------------------

manifest_rows = []

rejection_counts = Counter()
background_label_counts = Counter()
protocol_excluded_counts = Counter()
unmapped_label_counts = Counter()

short_segment_examples = []
valid_fraction_examples = []

for subject in common_subjects:
    segments = all_raw_segments.get(subject, [])

    rejection_counts["subjects_seen"] += 1
    rejection_counts["raw_segments_seen"] += len(segments)

    per_class_counter = Counter()

    for segment_index, segment in enumerate(segments):
        raw_label = segment.get("raw_label")

        class_id, label_status = map_mri_label(
            raw_label
        )

        # ----------------------------------------------------
        # Ignore calibration/background intervals
        # ----------------------------------------------------
        if label_status == "background":
            rejection_counts["background_or_calibration"] += 1
            background_label_counts[
                repr(raw_label)
            ] += 1
            continue

        # ----------------------------------------------------
        # Ignore P1-only classes when running P2_10
        # ----------------------------------------------------
        if label_status == "protocol_excluded":
            rejection_counts["excluded_by_protocol"] += 1
            protocol_excluded_counts[
                repr(raw_label)
            ] += 1
            continue

        # ----------------------------------------------------
        # Truly unknown label
        # ----------------------------------------------------
        if label_status == "unmapped":
            rejection_counts["unmapped_label"] += 1
            unmapped_label_counts[
                repr(raw_label)
            ] += 1
            continue

        if class_id is None:
            rejection_counts["unexpected_null_class"] += 1
            continue

        # ----------------------------------------------------
        # Read interval coordinates
        # ----------------------------------------------------
        try:
            start_value = float(
                segment["start"]
            )

            end_value = float(
                segment["end"]
            )

            scale = float(
                segment.get("scale")
                or max(end_value, 1.0)
            )

        except Exception:
            rejection_counts[
                "invalid_numeric_boundary"
            ] += 1
            continue

        if (
            not np.isfinite(start_value)
            or not np.isfinite(end_value)
        ):
            rejection_counts[
                "non_finite_boundary"
            ] += 1
            continue

        if (
            not np.isfinite(scale)
            or scale <= 0
        ):
            rejection_counts["invalid_scale"] += 1
            continue

        # ----------------------------------------------------
        # Convert to proportional recording coordinates
        # ----------------------------------------------------
        start_fraction = float(
            np.clip(
                start_value / scale,
                0.0,
                1.0,
            )
        )

        end_fraction = float(
            np.clip(
                end_value / scale,
                0.0,
                1.0,
            )
        )

        duration_fraction = (
            end_fraction
            - start_fraction
        )

        if end_fraction <= start_fraction:
            rejection_counts[
                "non_positive_interval"
            ] += 1
            continue

        # ----------------------------------------------------
        # Remove implausibly short segments
        # ----------------------------------------------------
        if (
            duration_fraction
            < cfg.MIN_SEGMENT_FRACTION
        ):
            rejection_counts[
                "below_minimum_fraction"
            ] += 1

            if len(short_segment_examples) < 20:
                short_segment_examples.append({
                    "subject": subject,
                    "raw_label": repr(raw_label),
                    "class_id": class_id,
                    "class_name": CLASS_NAMES[class_id],
                    "start": start_value,
                    "end": end_value,
                    "scale": scale,
                    "duration_fraction": (
                        duration_fraction
                    ),
                    "absolute_timestamp_detected": (
                        segment.get(
                            "absolute_timestamp_detected"
                        )
                    ),
                    "coordinate_origin": (
                        segment.get(
                            "coordinate_origin"
                        )
                    ),
                })

            continue

        # ----------------------------------------------------
        # Optional per-subject/class limit
        # ----------------------------------------------------
        if (
            cfg.MAX_SEGMENTS_PER_CLASS_PER_SUBJECT
            is not None
            and per_class_counter[class_id]
            >= cfg.MAX_SEGMENTS_PER_CLASS_PER_SUBJECT
        ):
            rejection_counts[
                "class_subject_cap"
            ] += 1
            continue

        rejection_counts["accepted"] += 1

        if len(valid_fraction_examples) < 20:
            valid_fraction_examples.append({
                "subject": subject,
                "raw_label": repr(raw_label),
                "class_id": class_id,
                "class_name": (
                    CLASS_NAMES[class_id]
                ),
                "start_fraction": (
                    start_fraction
                ),
                "end_fraction": (
                    end_fraction
                ),
                "duration_fraction": (
                    duration_fraction
                ),
            })

        manifest_rows.append({
            "sample_id": (
                f"{subject}_"
                f"{segment_index:05d}"
            ),
            "subject": subject,
            "class_id": class_id,
            "class_name": (
                CLASS_NAMES[class_id]
            ),
            "start_frac": start_fraction,
            "end_frac": end_fraction,
            "duration_frac": (
                duration_fraction
            ),
            "source_start": start_value,
            "source_end": end_value,
            "source_scale": scale,
            "coordinate_origin": (
                segment.get(
                    "coordinate_origin",
                    0.0,
                )
            ),
            "absolute_timestamp_detected": (
                segment.get(
                    "absolute_timestamp_detected",
                    False,
                )
            ),
            "raw_label": str(raw_label),
            "normalised_raw_label": (
                normalise_mri_label(
                    raw_label
                )
            ),
            "label_source": (
                segment.get(
                    "source",
                    "unknown",
                )
            ),
            "radar_path": str(
                radar_by_subject[subject]
            ),
            "imu_path": str(
                imu_by_subject[subject]
            ),
            "label_path": str(
                label_by_subject[subject]
            ),
        })

        per_class_counter[
            class_id
        ] += 1


# ------------------------------------------------------------
# Manifest-construction audit
# ------------------------------------------------------------

print("\nManifest construction audit:")

rejection_table = pd.DataFrame(
    sorted(
        rejection_counts.items()
    ),
    columns=[
        "stage",
        "count",
    ],
)

display(rejection_table)


if background_label_counts:
    print(
        "\nIgnored calibration/background labels:"
    )

    display(
        pd.DataFrame(
            background_label_counts.most_common(),
            columns=[
                "raw_label",
                "count",
            ],
        )
    )


if protocol_excluded_counts:
    print(
        "\nLabels excluded by the selected protocol:"
    )

    display(
        pd.DataFrame(
            protocol_excluded_counts.most_common(),
            columns=[
                "raw_label",
                "count",
            ],
        )
    )


if unmapped_label_counts:
    print(
        "\nUnrecognised labels that still require mapping:"
    )

    display(
        pd.DataFrame(
            unmapped_label_counts.most_common(30),
            columns=[
                "raw_label",
                "count",
            ],
        )
    )


if short_segment_examples:
    print(
        "\nExamples rejected as too short:"
    )

    display(
        pd.DataFrame(
            short_segment_examples
        )
    )


if valid_fraction_examples:
    print(
        "\nExamples accepted into the manifest:"
    )

    display(
        pd.DataFrame(
            valid_fraction_examples
        )
    )


# ------------------------------------------------------------
# Construct DataFrame
# ------------------------------------------------------------

manifest = pd.DataFrame(
    manifest_rows
)


if manifest.empty:
    raise RuntimeError(
        "No usable action segments were created. "
        "Review the manifest audit above. "
        "The mRI pose labels are now mapped explicitly, "
        "so any remaining failure is likely caused by "
        "interval coordinates or filtering."
    )


# ------------------------------------------------------------
# Complete-manifest statistics
# ------------------------------------------------------------

class_counts_complete = (
    manifest["class_id"]
    .value_counts()
    .reindex(
        range(NUM_CLASSES),
        fill_value=0,
    )
    .sort_index()
)


class_subject_counts = (
    manifest
    .groupby("class_id")["subject"]
    .nunique()
    .reindex(
        range(NUM_CLASSES),
        fill_value=0,
    )
    .sort_index()
)


complete_audit = pd.DataFrame({
    "class_id": range(NUM_CLASSES),
    "class_name": CLASS_NAMES,
    "segments": (
        class_counts_complete.values
    ),
    "subjects": (
        class_subject_counts.values
    ),
})


print(
    "\nTotal manifest segments:",
    len(manifest),
)

print(
    "Manifest subjects:",
    manifest["subject"].nunique(),
)

display(
    complete_audit
)


subject_class_table = pd.crosstab(
    manifest["subject"],
    manifest["class_id"],
).reindex(
    columns=range(NUM_CLASSES),
    fill_value=0,
)


subject_class_table.columns = [
    (
        f"{class_id}: "
        f"{CLASS_NAMES[class_id]}"
    )
    for class_id
    in subject_class_table.columns
]


display(
    subject_class_table
)


# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

missing_classes = (
    class_counts_complete[
        class_counts_complete == 0
    ]
    .index
    .tolist()
)


insufficient_subject_classes = (
    class_subject_counts[
        class_subject_counts
        < cfg.MIN_SUBJECTS_PER_CLASS
    ]
    .index
    .tolist()
)


if missing_classes:
    missing_details = {
        class_id: CLASS_NAMES[class_id]
        for class_id in missing_classes
    }

    raise RuntimeError(
        "The complete manifest is missing classes: "
        f"{missing_details}."
    )


if (
    len(manifest)
    < cfg.MIN_TOTAL_MANIFEST_SEGMENTS
):
    raise RuntimeError(
        f"Only {len(manifest)} manifest segments "
        "were extracted. "
        f"At least "
        f"{cfg.MIN_TOTAL_MANIFEST_SEGMENTS} "
        "are required."
    )


if insufficient_subject_classes:
    details = {
        class_id: {
            "class_name": CLASS_NAMES[
                class_id
            ],
            "subjects": int(
                class_subject_counts.loc[
                    class_id
                ]
            ),
        }
        for class_id
        in insufficient_subject_classes
    }

    raise RuntimeError(
        "Some classes occur in too few subjects "
        "for subject-independent evaluation: "
        f"{details}."
    )


# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

manifest.to_csv(
    Path(cfg.WORK_DIR)
    / "full_manifest.csv",
    index=False,
)


subject_class_table.to_csv(
    Path(cfg.WORK_DIR)
    / "subject_class_distribution.csv"
)


complete_audit.to_csv(
    Path(cfg.WORK_DIR)
    / "complete_manifest_audit.csv",
    index=False,
)


print(
    "\nManifest integrity checks passed."
)

mRI label-mapping preview:


,raw_label,normalised_label,status,class_id,class_name
0,T pose,t pose,background,NaN,None
1,pose_1,pose 1,mapped,0.0,left upper-limb extension
2,pose_2,pose 2,mapped,1.0,right upper-limb extension
3,pose_3,pose 3,mapped,2.0,both upper-limb extension
4,pose_4,pose 4,mapped,3.0,left front lunge
5,pose_5,pose 5,mapped,4.0,right front lunge
6,pose_6,pose 6,mapped,5.0,squat
7,pose_7,pose 7,mapped,6.0,left side lunge
8,pose_8,pose 8,mapped,7.0,right side lunge
9,pose_9,pose 9,mapped,8.0,left limb extension



Manifest construction audit:


,stage,count
0,accepted,200
1,background_or_calibration,33
2,excluded_by_protocol,40
3,raw_segments_seen,273
4,subjects_seen,20



Ignored calibration/background labels:


...

,subject,raw_label,class_id,class_name,start_fraction,end_fraction,duration_fraction
0,subject01,'pose_1',0,left upper-limb extension,0.026076,0.098750,0.072674
1,subject01,'pose_2',1,right upper-limb extension,0.098750,0.173121,0.074371
2,subject01,'pose_3',2,both upper-limb extension,0.173121,0.256288,0.083166
3,subject01,'pose_4',3,left front lunge,0.256288,0.335288,0.079000
4,subject01,'pose_5',4,right front lunge,0.335288,0.415368,0.080080
5,subject01,'pose_6',5,squat,0.415368,0.494677,0.079309
6,subject01,'pose_7',6,left side lunge,0.494677,0.574448,0.079772
7,subject01,'pose_8',7,right side lunge,0.574448,0.656226,0.081778
8,subject01,'pose_9',8,left limb extension,0.656226,0.731986,0.075760
9,subject01,'pose_10',9,right limb extension,0.731986,0.818084,0.086098



Total manifest segments: 200
Manifest subjects: 20


,class_id,class_name,segments,subjects
0,0,left upper-limb extension,20,20
1,1,right upper-limb extension,20,20
2,2,both upper-limb extension,20,20
3,3,left front lunge,20,20
4,4,right front lunge,20,20
5,5,squat,20,20
6,6,left side lunge,20,20
7,7,right side lunge,20,20
8,8,left limb extension,20,20
9,9,right limb extension,20,20


,0: left upper-limb extension,1: right upper-limb extension,2: both upper-limb extension,3: left front lunge,4: right front lunge,5: squat,6: left side lunge,7: right side lunge,8: left limb extension,9: right limb extension
subject,,,,,,,,,,
subject01,1,1,1,1,1,1,1,1,1,1
subject02,1,1,1,1,1,1,1,1,1,1
subject03,1,1,1,1,1,1,1,1,1,1
subject04,1,1,1,1,1,1,1,1,1,1
subject05,1,1,1,1,1,1,1,1,1,1
subject06,1,1,1,1,1,1,1,1,1,1
subject07,1,1,1,1,1,1,1,1,1,1
subject08,1,1,1,1,1,1,1,1,1,1
subject09,1,1,1,1,1,1,1,1,1,1



Manifest integrity checks passed.


In [10]:
def unwrap_imu_tensor(obj):
    if isinstance(obj, dict):
        for key in ["acc_ori", "data", "features", "x", "imu"]:
            if key in obj and (
                torch.is_tensor(obj[key])
                or isinstance(obj[key], np.ndarray)
            ):
                obj = obj[key]
                break
        else:
            values = [
                value
                for value in obj.values()
                if torch.is_tensor(value) or isinstance(value, np.ndarray)
            ]
            if not values:
                raise TypeError("No tensor-like IMU value found.")
            obj = values[0]

    if torch.is_tensor(obj):
        obj = obj.detach().cpu().numpy()

    return np.asarray(obj)

def canonicalise_imu_shape(array):
    array = np.asarray(array)

    if array.ndim == 3 and array.shape[1:] == (6, 12):
        return array
    if array.ndim == 3 and array.shape[:2] == (6, 12):
        return np.moveaxis(array, -1, 0)
    if array.ndim == 2 and array.shape[1] == 72:
        return array.reshape(array.shape[0], 6, 12)
    if array.ndim == 2 and array.shape[0] == 72:
        return array.T.reshape(array.shape[1], 6, 12)

    raise ValueError(f"Unexpected IMU shape: {array.shape}")

def canonicalise_radar_shape(array):
    array = np.asarray(array)

    if array.ndim != 4:
        raise ValueError(f"Unexpected radar shape: {array.shape}")
    if array.shape[-1] == 5:
        return array
    if array.shape[1] == 5:
        return np.moveaxis(array, 1, -1)

    raise ValueError(
        f"Could not identify five radar channels in shape {array.shape}"
    )

audit_rows = []

for subject in common_subjects:
    radar = canonicalise_radar_shape(
        np.load(radar_by_subject[subject], mmap_mode="r")
    )
    imu = canonicalise_imu_shape(
        unwrap_imu_tensor(load_any(imu_by_subject[subject]))
    )

    audit_rows.append({
        "subject": subject,
        "radar_frames": int(radar.shape[0]),
        "radar_shape": str(tuple(radar.shape)),
        "imu_frames": int(imu.shape[0]),
        "imu_shape": str(tuple(imu.shape)),
        "manifest_segments": int(
            (manifest["subject"] == subject).sum()
        ),
        "manifest_classes": int(
            manifest.loc[
                manifest["subject"] == subject,
                "class_id"
            ].nunique()
        ),
    })

audit_df = pd.DataFrame(audit_rows)
display(audit_df)

audit_df.to_csv(
    Path(cfg.WORK_DIR) / "dataset_audit.csv",
    index=False,
)

if (audit_df["manifest_segments"] <= 1).any():
    print(
        "Warning: one or more subjects have at most one parsed action segment. "
        "Inspect those subjects before relying on the results."
    )

,subject,radar_frames,radar_shape,imu_frames,imu_shape,manifest_segments,manifest_classes
0,subject01,6384,"(6384, 14, 14, 5)",6384,"(6384, 6, 12)",10,10
1,subject02,7150,"(7150, 14, 14, 5)",7150,"(7150, 6, 12)",10,10
2,subject03,6982,"(6982, 14, 14, 5)",6982,"(6982, 6, 12)",10,10
3,subject04,6893,"(6893, 14, 14, 5)",6893,"(6893, 6, 12)",10,10
4,subject05,7219,"(7219, 14, 14, 5)",7219,"(7219, 6, 12)",10,10
5,subject06,6906,"(6906, 14, 14, 5)",6906,"(6906, 6, 12)",10,10
6,subject07,6743,"(6743, 14, 14, 5)",6743,"(6743, 6, 12)",10,10
7,subject08,6817,"(6817, 14, 14, 5)",6817,"(6817, 6, 12)",10,10
8,subject09,6704,"(6704, 14, 14, 5)",6704,"(6704, 6, 12)",10,10
9,subject10,6911,"(6911, 14, 14, 5)",6911,"(6911, 6, 12)",10,10


In [11]:
def counts_for_subjects(frame, selected_subjects):
    return (
        frame.loc[
            frame["subject"].isin(selected_subjects),
            "class_id"
        ]
        .value_counts()
        .reindex(range(NUM_CLASSES), fill_value=0)
        .sort_index()
    )

def find_coverage_aware_split(
    frame,
    seed,
    max_attempts,
    test_fraction,
    val_fraction_of_remainder,
):
    subjects = np.array(sorted(frame["subject"].unique()))
    number_of_subjects = len(subjects)

    if number_of_subjects < 5:
        raise RuntimeError(
            f"Only {number_of_subjects} subjects are available. "
            "At least five are required."
        )

    number_test = max(1, int(round(number_of_subjects * test_fraction)))
    number_remaining = number_of_subjects - number_test
    number_val = max(
        1,
        int(round(number_remaining * val_fraction_of_remainder))
    )
    number_train = number_of_subjects - number_test - number_val

    if number_train < 1:
        raise RuntimeError("The requested fractions leave no training subjects.")

    print(
        f"Searching for {number_train} train, {number_val} validation, "
        f"and {number_test} test subjects."
    )

    global_distribution = (
        frame["class_id"]
        .value_counts(normalize=True)
        .reindex(range(NUM_CLASSES), fill_value=0)
        .values
    )

    rng = np.random.default_rng(seed)
    best = None
    best_score = np.inf

    for attempt in range(max_attempts):
        shuffled = rng.permutation(subjects)

        test_subjects = shuffled[:number_test]
        val_subjects = shuffled[
            number_test:number_test + number_val
        ]
        train_subjects = shuffled[
            number_test + number_val:
        ]

        train_counts = counts_for_subjects(frame, train_subjects)
        val_counts = counts_for_subjects(frame, val_subjects)
        test_counts = counts_for_subjects(frame, test_subjects)

        missing_train = int((train_counts == 0).sum())
        missing_val = int((val_counts == 0).sum())
        missing_test = int((test_counts == 0).sum())

        def distribution_distance(counts):
            if counts.sum() == 0:
                return np.inf
            return float(
                np.abs(counts.values / counts.sum() - global_distribution).sum()
            )

        score = (
            1_000_000 * missing_train
            + 100_000 * missing_val
            + 100_000 * missing_test
            + distribution_distance(train_counts)
            + distribution_distance(val_counts)
            + distribution_distance(test_counts)
        )

        candidate = {
            "train_subjects": train_subjects,
            "val_subjects": val_subjects,
            "test_subjects": test_subjects,
            "train_counts": train_counts,
            "val_counts": val_counts,
            "test_counts": test_counts,
            "attempt": attempt + 1,
            "score": score,
        }

        if score < best_score:
            best_score = score
            best = candidate

        if missing_train == 0 and missing_val == 0 and missing_test == 0:
            print(f"Full-coverage split found after {attempt + 1} attempts.")
            return candidate

    raise RuntimeError(
        "No subject-independent split with complete class coverage was found "
        f"after {max_attempts} attempts. Best split had missing counts: "
        f"train={(best['train_counts'] == 0).sum()}, "
        f"validation={(best['val_counts'] == 0).sum()}, "
        f"test={(best['test_counts'] == 0).sum()}. "
        "Inspect subject_class_distribution.csv."
    )

split_result = find_coverage_aware_split(
    frame=manifest,
    seed=cfg.SPLIT_SEED,
    max_attempts=cfg.SPLIT_SEARCH_ATTEMPTS,
    test_fraction=cfg.TEST_SUBJECT_FRACTION,
    val_fraction_of_remainder=cfg.VAL_SUBJECT_FRACTION_OF_REMAINDER,
)

train_subjects = np.array(split_result["train_subjects"])
val_subjects = np.array(split_result["val_subjects"])
test_subjects = np.array(split_result["test_subjects"])

split_map = {subject: "train" for subject in train_subjects}
split_map.update({subject: "val" for subject in val_subjects})
split_map.update({subject: "test" for subject in test_subjects})

manifest["split"] = manifest["subject"].map(split_map)

train_df = manifest.loc[
    manifest["split"] == "train"
].reset_index(drop=True)

val_df = manifest.loc[
    manifest["split"] == "val"
].reset_index(drop=True)

test_df = manifest.loc[
    manifest["split"] == "test"
].reset_index(drop=True)

assert set(train_subjects).isdisjoint(val_subjects)
assert set(train_subjects).isdisjoint(test_subjects)
assert set(val_subjects).isdisjoint(test_subjects)

print("\nTrain subjects:", sorted(train_subjects.tolist()))
print("Validation subjects:", sorted(val_subjects.tolist()))
print("Test subjects:", sorted(test_subjects.tolist()))
print(
    "Samples:",
    {
        "train": len(train_df),
        "validation": len(val_df),
        "test": len(test_df),
    },
)

split_audit_rows = []

for split_name, split_frame in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df),
]:
    counts = (
        split_frame["class_id"]
        .value_counts()
        .reindex(range(NUM_CLASSES), fill_value=0)
        .sort_index()
    )

    for class_id in range(NUM_CLASSES):
        split_audit_rows.append({
            "split": split_name,
            "class_id": class_id,
            "class_name": CLASS_NAMES[class_id],
            "segments": int(counts.loc[class_id]),
            "subjects": int(
                split_frame.loc[
                    split_frame["class_id"] == class_id,
                    "subject"
                ].nunique()
            ),
        })

split_audit = pd.DataFrame(split_audit_rows)
display(split_audit)

if (split_audit["segments"] == 0).any():
    raise RuntimeError(
        "A split is missing at least one class despite the coverage search."
    )

manifest.to_csv(
    Path(cfg.WORK_DIR) / "manifest_with_splits.csv",
    index=False,
)
split_audit.to_csv(
    Path(cfg.WORK_DIR) / "split_class_audit.csv",
    index=False,
)

Searching for 12 train, 4 validation, and 4 test subjects.
Full-coverage split found after 1 attempts.

Train subjects: ['subject03', 'subject04', 'subject05', 'subject06', 'subject08', 'subject09', 'subject11', 'subject12', 'subject13', 'subject17', 'subject18', 'subject20']
Validation subjects: ['subject02', 'subject07', 'subject14', 'subject16']
Test subjects: ['subject01', 'subject10', 'subject15', 'subject19']
Samples: {'train': 120, 'validation': 40, 'test': 40}


,split,class_id,class_name,segments,subjects
0,train,0,left upper-limb extension,12,12
1,train,1,right upper-limb extension,12,12
2,train,2,both upper-limb extension,12,12
3,train,3,left front lunge,12,12
4,train,4,right front lunge,12,12
5,train,5,squat,12,12
6,train,6,left side lunge,12,12
7,train,7,right side lunge,12,12
8,train,8,left limb extension,12,12
9,train,9,right limb extension,12,12


In [12]:
def compute_training_stats(subject_list):
    radar_sum = np.zeros(5, dtype=np.float64)
    radar_sumsq = np.zeros(5, dtype=np.float64)
    radar_count = 0

    imu_sum = np.zeros(72, dtype=np.float64)
    imu_sumsq = np.zeros(72, dtype=np.float64)
    imu_count = 0

    for subject in subject_list:
        radar = canonicalise_radar_shape(np.load(radar_by_subject[subject], mmap_mode="r"))
        n = radar.shape[0]
        if n > cfg.STAT_MAX_FRAMES_PER_SUBJECT:
            idx = np.linspace(0, n - 1, cfg.STAT_MAX_FRAMES_PER_SUBJECT).astype(int)
            radar_sample = np.asarray(radar[idx], dtype=np.float32)
        else:
            radar_sample = np.asarray(radar, dtype=np.float32)
        flat_r = radar_sample.reshape(-1, 5).astype(np.float64)
        radar_sum += flat_r.sum(axis=0)
        radar_sumsq += np.square(flat_r).sum(axis=0)
        radar_count += flat_r.shape[0]

        imu = canonicalise_imu_shape(unwrap_imu_tensor(load_any(imu_by_subject[subject])))
        n_i = imu.shape[0]
        if n_i > cfg.STAT_MAX_FRAMES_PER_SUBJECT:
            idx_i = np.linspace(0, n_i - 1, cfg.STAT_MAX_FRAMES_PER_SUBJECT).astype(int)
            imu_sample = imu[idx_i]
        else:
            imu_sample = imu
        flat_i = imu_sample.reshape(-1, 72).astype(np.float64)
        imu_sum += flat_i.sum(axis=0)
        imu_sumsq += np.square(flat_i).sum(axis=0)
        imu_count += flat_i.shape[0]

    radar_mean = radar_sum / radar_count
    radar_var = np.maximum(radar_sumsq / radar_count - radar_mean**2, 1e-8)
    imu_mean = imu_sum / imu_count
    imu_var = np.maximum(imu_sumsq / imu_count - imu_mean**2, 1e-8)

    return {
        "radar_mean": radar_mean.astype(np.float32),
        "radar_std": np.sqrt(radar_var).astype(np.float32),
        "imu_mean": imu_mean.astype(np.float32),
        "imu_std": np.sqrt(imu_var).astype(np.float32),
    }

stats = compute_training_stats(train_subjects)
print("Radar mean:", stats["radar_mean"])
print("Radar std:", stats["radar_std"])
np.savez(Path(cfg.WORK_DIR) / "normalisation_stats.npz", **stats)

Radar mean: [-3.9595038e-02  1.0351279e+00  4.3106000e-03 -1.7172944e-04
 -2.7271398e-04]
Radar std: [0.59321713 1.1836222  0.55033183 0.33115694 0.68407977]


In [13]:
def temporal_resample_np(array, target_length):
    array = np.asarray(array, dtype=np.float32)

    if array.shape[0] == target_length:
        return array
    if array.shape[0] < 2:
        return np.repeat(array, target_length, axis=0)

    flattened = torch.from_numpy(
        array.reshape(array.shape[0], -1).T
    ).unsqueeze(0)

    resized = F.interpolate(
        flattened,
        size=target_length,
        mode="linear",
        align_corners=False,
    )

    return (
        resized.squeeze(0)
        .T.numpy()
        .reshape((target_length,) + array.shape[1:])
    )

class MRISegmentDataset(Dataset):
    def __init__(self, frame, stats, sequence_length, cache_size=3):
        self.frame = frame.reset_index(drop=True).copy()
        self.sequence_length = sequence_length

        self.radar_mean = stats["radar_mean"].reshape(1, 1, 1, 5)
        self.radar_std = np.maximum(
            stats["radar_std"].reshape(1, 1, 1, 5),
            1e-6,
        )

        self.imu_mean = stats["imu_mean"].reshape(1, 72)
        self.imu_std = np.maximum(
            stats["imu_std"].reshape(1, 72),
            1e-6,
        )

        self.cache_size = cache_size
        self._cache = OrderedDict()

    def __len__(self):
        return len(self.frame)

    def _load_subject(self, radar_path, imu_path):
        key = (radar_path, imu_path)

        if key in self._cache:
            self._cache.move_to_end(key)
            return self._cache[key]

        radar = canonicalise_radar_shape(
            np.load(radar_path, mmap_mode="r")
        )

        imu = canonicalise_imu_shape(
            unwrap_imu_tensor(load_any(imu_path))
        )

        self._cache[key] = (radar, imu)
        self._cache.move_to_end(key)

        while len(self._cache) > self.cache_size:
            self._cache.popitem(last=False)

        return radar, imu

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        radar_full, imu_full = self._load_subject(
            row.radar_path,
            row.imu_path,
        )

        radar_start = int(round(row.start_frac * len(radar_full)))
        radar_end = int(round(row.end_frac * len(radar_full)))
        imu_start = int(round(row.start_frac * len(imu_full)))
        imu_end = int(round(row.end_frac * len(imu_full)))

        radar_start = min(max(radar_start, 0), len(radar_full) - 1)
        radar_end = min(max(radar_end, radar_start + 1), len(radar_full))
        imu_start = min(max(imu_start, 0), len(imu_full) - 1)
        imu_end = min(max(imu_end, imu_start + 1), len(imu_full))

        radar = np.asarray(
            radar_full[radar_start:radar_end],
            dtype=np.float32,
        )

        imu = np.asarray(
            imu_full[imu_start:imu_end],
            dtype=np.float32,
        ).reshape(-1, 72)

        if radar.shape[0] == 0 or imu.shape[0] == 0:
            raise RuntimeError(
                f"Empty modality slice for {row.sample_id}: "
                f"radar={radar.shape}, imu={imu.shape}"
            )

        radar = temporal_resample_np(
            radar,
            self.sequence_length,
        )

        imu = temporal_resample_np(
            imu,
            self.sequence_length,
        )

        radar = np.nan_to_num(
            (radar - self.radar_mean) / self.radar_std,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        imu = np.nan_to_num(
            (imu - self.imu_mean) / self.imu_std,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        return {
            "radar": torch.from_numpy(radar).float(),
            "imu": torch.from_numpy(imu).float(),
            "label": torch.tensor(
                int(row.class_id),
                dtype=torch.long,
            ),
            "subject": row.subject,
            "sample_id": row.sample_id,
        }

train_dataset = MRISegmentDataset(
    train_df,
    stats,
    cfg.SEQ_LEN,
    cfg.CACHE_SUBJECTS_PER_WORKER,
)

val_dataset = MRISegmentDataset(
    val_df,
    stats,
    cfg.SEQ_LEN,
    cfg.CACHE_SUBJECTS_PER_WORKER,
)

test_dataset = MRISegmentDataset(
    test_df,
    stats,
    cfg.SEQ_LEN,
    cfg.CACHE_SUBJECTS_PER_WORKER,
)

class_counts = (
    train_df["class_id"]
    .value_counts()
    .reindex(range(NUM_CLASSES), fill_value=0)
    .sort_index()
)

if (class_counts == 0).any():
    raise RuntimeError(
        "Training split is missing classes "
        f"{class_counts[class_counts == 0].index.tolist()}."
    )

class_weights_np = (
    len(train_df)
    / (NUM_CLASSES * class_counts.values.astype(np.float64))
)

class_weights = torch.tensor(
    class_weights_np,
    dtype=torch.float32,
    device=DEVICE,
)

print(
    "Training class weights:",
    dict(zip(CLASS_NAMES, class_weights_np.round(3))),
)

sampler = None
shuffle = True

if cfg.USE_WEIGHTED_SAMPLER:
    weight_map = dict(zip(range(NUM_CLASSES), class_weights_np))
    sample_weights = train_df["class_id"].map(weight_map).values

    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(
            sample_weights,
            dtype=torch.double,
        ),
        num_samples=len(sample_weights),
        replacement=True,
    )

    shuffle = False

pin_memory = DEVICE.type == "cuda"
persistent_workers = cfg.NUM_WORKERS > 0

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.BATCH_SIZE,
    sampler=sampler,
    shuffle=shuffle,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
    drop_last=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
)

batch = next(iter(train_loader))

print("Radar batch:", tuple(batch["radar"].shape))
print("IMU batch:", tuple(batch["imu"].shape))
print("Labels:", tuple(batch["label"].shape))
print("Label IDs in batch:", sorted(batch["label"].unique().tolist()))

Training class weights: {'left upper-limb extension': np.float64(1.0), 'right upper-limb extension': np.float64(1.0), 'both upper-limb extension': np.float64(1.0), 'left front lunge': np.float64(1.0), 'right front lunge': np.float64(1.0), 'squat': np.float64(1.0), 'left side lunge': np.float64(1.0), 'right side lunge': np.float64(1.0), 'left limb extension': np.float64(1.0), 'right limb extension': np.float64(1.0)}
Radar batch: (24, 96, 14, 14, 5)
IMU batch: (24, 96, 72)
Labels: (24,)
Label IDs in batch: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


## Proposed RTPD-Net architecture

The radar and IMU branches preserve a sequence of temporal tokens instead of immediately reducing each recording to one pooled vector. The teacher aligns IMU tokens to the radar timeline using cross-attention, estimates modality reliability at every time step, and performs reliability-weighted fusion.

The student receives only radar. During training, it learns from teacher predictions, temporally aligned teacher tokens, batch-level relations, class prototypes, and an adversarial subject head. The subject head is removed at deployment.

In [14]:
# ============================================================
# Token encoders, teacher, student, and adversarial components
# ============================================================

class ResidualTemporalBlock(nn.Module):
    def __init__(self, in_dim, out_dim, dilation, dropout):
        super().__init__()
        padding = dilation
        self.main = nn.Sequential(
            nn.Conv1d(in_dim, out_dim, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_dim, out_dim, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.skip = nn.Identity() if in_dim == out_dim else nn.Conv1d(in_dim, out_dim, 1)
        self.activation = nn.GELU()

    def forward(self, x):
        return self.activation(self.main(x) + self.skip(x))


class TemporalTokenRefiner(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        blocks = []
        current = input_dim
        for output_dim, dilation in zip(cfg.TCN_CHANNELS, cfg.TCN_DILATIONS):
            blocks.append(
                ResidualTemporalBlock(current, output_dim, dilation, cfg.DROPOUT)
            )
            current = output_dim
        self.blocks = nn.Sequential(*blocks)
        self.output_dim = current

    def forward(self, tokens):
        x = tokens.transpose(1, 2)
        x = self.blocks(x)
        return x.transpose(1, 2)


class TemporalAttentionPool(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim // 2),
            nn.Tanh(),
            nn.Linear(dim // 2, 1),
        )

    def forward(self, tokens):
        weights = torch.softmax(self.score(tokens).squeeze(-1), dim=1)
        pooled = torch.sum(tokens * weights.unsqueeze(-1), dim=1)
        return pooled, weights


class RadarTokenEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.frame_cnn = nn.Sequential(
            nn.Conv2d(5, 24, 3, padding=1),
            nn.BatchNorm2d(24),
            nn.GELU(),
            nn.MaxPool2d(2),
            nn.Conv2d(24, 48, 3, padding=1),
            nn.BatchNorm2d(48),
            nn.GELU(),
            nn.Conv2d(48, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.frame_projection = nn.Sequential(
            nn.Linear(64, cfg.EMBED_DIM),
            nn.LayerNorm(cfg.EMBED_DIM),
            nn.GELU(),
            nn.Dropout(cfg.DROPOUT),
        )
        self.temporal = TemporalTokenRefiner(cfg.EMBED_DIM)
        self.output_dim = self.temporal.output_dim

    def forward(self, radar):
        if radar.ndim != 5 or radar.shape[-1] != 5:
            raise ValueError(
                f"Expected radar [B,T,H,W,5], received {tuple(radar.shape)}"
            )
        batch_size, time_steps, height, width, channels = radar.shape
        frames = (
            radar.permute(0, 1, 4, 2, 3)
            .contiguous()
            .reshape(batch_size * time_steps, channels, height, width)
        )
        frame_features = self.frame_cnn(frames).flatten(1)
        frame_features = self.frame_projection(frame_features)
        tokens = frame_features.reshape(batch_size, time_steps, -1)
        return self.temporal(tokens)


class IMUTokenEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.input_projection = nn.Sequential(
            nn.Linear(72, cfg.EMBED_DIM),
            nn.LayerNorm(cfg.EMBED_DIM),
            nn.GELU(),
            nn.Dropout(cfg.DROPOUT),
        )
        self.temporal = TemporalTokenRefiner(cfg.EMBED_DIM)
        self.output_dim = self.temporal.output_dim

    def forward(self, imu):
        if imu.ndim != 3 or imu.shape[-1] != 72:
            raise ValueError(f"Expected IMU [B,T,72], received {tuple(imu.shape)}")
        return self.temporal(self.input_projection(imu))


class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, coefficient):
        ctx.coefficient = coefficient
        return x.view_as(x)

    @staticmethod
    def backward(ctx, gradient):
        return -ctx.coefficient * gradient, None


def gradient_reverse(x, coefficient=1.0):
    return GradientReversalFunction.apply(x, coefficient)


class RadarOnlyModel(nn.Module):
    def __init__(self, num_classes, num_train_subjects=0):
        super().__init__()
        self.encoder = RadarTokenEncoder()
        dim = self.encoder.output_dim
        self.pool = TemporalAttentionPool(dim)
        self.classifier = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Dropout(cfg.DROPOUT),
            nn.Linear(dim, num_classes),
        )
        self.subject_head = (
            nn.Sequential(
                nn.LayerNorm(dim),
                nn.Linear(dim, dim // 2),
                nn.GELU(),
                nn.Dropout(cfg.DROPOUT),
                nn.Linear(dim // 2, num_train_subjects),
            )
            if num_train_subjects > 1
            else None
        )

    def forward(self, radar, grl_coefficient=0.0):
        tokens = self.encoder(radar)
        pooled, temporal_attention = self.pool(tokens)
        logits = self.classifier(pooled)
        subject_logits = None
        if self.subject_head is not None:
            subject_logits = self.subject_head(
                gradient_reverse(pooled, grl_coefficient)
            )
        return {
            "logits": logits,
            "tokens": tokens,
            "pooled": pooled,
            "temporal_attention": temporal_attention,
            "subject_logits": subject_logits,
        }


class IMUOnlyModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.encoder = IMUTokenEncoder()
        dim = self.encoder.output_dim
        self.pool = TemporalAttentionPool(dim)
        self.classifier = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Dropout(cfg.DROPOUT),
            nn.Linear(dim, num_classes),
        )

    def forward(self, imu):
        tokens = self.encoder(imu)
        pooled, temporal_attention = self.pool(tokens)
        return {
            "logits": self.classifier(pooled),
            "tokens": tokens,
            "pooled": pooled,
            "temporal_attention": temporal_attention,
        }


class ReliabilityAwareTeacher(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.radar_encoder = RadarTokenEncoder()
        self.imu_encoder = IMUTokenEncoder()
        dim = self.radar_encoder.output_dim

        self.radar_projection = nn.Sequential(
            nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU()
        )
        self.imu_projection = nn.Sequential(
            nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU()
        )

        self.imu_to_radar_attention = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=cfg.NUM_ATTENTION_HEADS,
            dropout=cfg.DROPOUT,
            batch_first=True,
        )

        self.reliability_head = nn.Sequential(
            nn.LayerNorm(dim * 4),
            nn.Linear(dim * 4, dim),
            nn.GELU(),
            nn.Dropout(cfg.DROPOUT),
            nn.Linear(dim, 2),
        )

        self.fused_refiner = TemporalTokenRefiner(dim)
        self.pool = TemporalAttentionPool(dim)

        self.classifier = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Dropout(cfg.DROPOUT),
            nn.Linear(dim, num_classes),
        )
        self.radar_aux_classifier = nn.Linear(dim, num_classes)
        self.imu_aux_classifier = nn.Linear(dim, num_classes)

    def forward(self, radar, imu):
        radar_tokens = self.radar_projection(self.radar_encoder(radar))
        imu_tokens = self.imu_projection(self.imu_encoder(imu))

        aligned_imu, cross_attention = self.imu_to_radar_attention(
            query=radar_tokens,
            key=imu_tokens,
            value=imu_tokens,
            need_weights=True,
            average_attn_weights=True,
        )

        reliability_input = torch.cat(
            [
                radar_tokens,
                aligned_imu,
                torch.abs(radar_tokens - aligned_imu),
                radar_tokens * aligned_imu,
            ],
            dim=-1,
        )
        reliability_logits = self.reliability_head(reliability_input)
        reliability = torch.softmax(reliability_logits, dim=-1)

        fused_tokens = (
            reliability[..., 0:1] * radar_tokens
            + reliability[..., 1:2] * aligned_imu
        )
        fused_tokens = self.fused_refiner(fused_tokens)
        pooled, temporal_attention = self.pool(fused_tokens)

        radar_pooled = radar_tokens.mean(dim=1)
        imu_pooled = aligned_imu.mean(dim=1)

        return {
            "logits": self.classifier(pooled),
            "fused_tokens": fused_tokens,
            "pooled": pooled,
            "reliability": reliability,
            "reliability_logits": reliability_logits,
            "cross_attention": cross_attention,
            "temporal_attention": temporal_attention,
            "radar_aux_logits": self.radar_aux_classifier(radar_pooled),
            "imu_aux_logits": self.imu_aux_classifier(imu_pooled),
            "radar_tokens": radar_tokens,
            "imu_tokens": aligned_imu,
        }


def count_trainable_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


SUBJECT_TO_TRAIN_ID = {
    subject: index for index, subject in enumerate(sorted(train_subjects.tolist()))
}

scratch_radar_model = RadarOnlyModel(NUM_CLASSES).to(DEVICE)
teacher_model = ReliabilityAwareTeacher(NUM_CLASSES).to(DEVICE)
student_model = RadarOnlyModel(
    NUM_CLASSES,
    num_train_subjects=len(SUBJECT_TO_TRAIN_ID),
).to(DEVICE)

print("Scratch radar parameters:", f"{count_trainable_parameters(scratch_radar_model):,}")
print("Teacher parameters:", f"{count_trainable_parameters(teacher_model):,}")
print("Student parameters:", f"{count_trainable_parameters(student_model):,}")

Scratch radar parameters: 454,491
Teacher parameters: 1,425,777
Student parameters: 463,783


In [15]:
# ============================================================
# Architecture smoke test
# ============================================================

scratch_radar_model.eval()
teacher_model.eval()
student_model.eval()

with torch.inference_mode():
    radar_probe = batch["radar"][:2].to(DEVICE)
    imu_probe = batch["imu"][:2].to(DEVICE)

    radar_output = scratch_radar_model(radar_probe)
    teacher_output = teacher_model(radar_probe, imu_probe)
    student_output = student_model(radar_probe)

print("Radar input:", tuple(radar_probe.shape))
print("IMU input:", tuple(imu_probe.shape))
print("Radar logits:", tuple(radar_output["logits"].shape))
print("Teacher logits:", tuple(teacher_output["logits"].shape))
print("Teacher fused tokens:", tuple(teacher_output["fused_tokens"].shape))
print("Reliability weights:", tuple(teacher_output["reliability"].shape))
print("Student tokens:", tuple(student_output["tokens"].shape))

assert radar_output["logits"].shape == (2, NUM_CLASSES)
assert teacher_output["logits"].shape == (2, NUM_CLASSES)
assert teacher_output["reliability"].shape == (2, cfg.SEQ_LEN, 2)
assert student_output["tokens"].shape[:2] == (2, cfg.SEQ_LEN)

del radar_probe, imu_probe, radar_output, teacher_output, student_output
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print("Architecture smoke test passed.")

Radar input: (2, 96, 14, 14, 5)
IMU input: (2, 96, 72)
Radar logits: (2, 10)
Teacher logits: (2, 10)
Teacher fused tokens: (2, 96, 128)
Reliability weights: (2, 96, 2)
Student tokens: (2, 96, 128)
Architecture smoke test passed.


## Stage A — radar self-supervised pretraining

Two augmented radar views are generated from each sequence. The encoder learns through:

- **masked temporal reconstruction:** reconstruct per-frame radar channel statistics at masked time steps;
- **cross-view contrastive learning:** keep two augmented views of the same exercise close while separating different sequences.

No class labels or IMU features are used in this stage.

In [16]:
# ============================================================
# Radar SSL augmentations and losses
# ============================================================

def radar_frame_statistics(radar):
    channel_mean = radar.mean(dim=(2, 3))
    channel_std = radar.std(dim=(2, 3), unbiased=False)
    return torch.cat([channel_mean, channel_std], dim=-1)


def augment_radar_for_ssl(radar, mask_probability=None):
    mask_probability = (
        cfg.SSL_MASK_PROB if mask_probability is None else mask_probability
    )
    augmented = radar.clone()
    batch_size, time_steps, height, width, channels = augmented.shape

    temporal_mask = (
        torch.rand(batch_size, time_steps, device=augmented.device)
        < mask_probability
    )

    # Guarantee at least one masked frame per sample.
    for sample_index in range(batch_size):
        if not temporal_mask[sample_index].any():
            temporal_mask[
                sample_index,
                torch.randint(time_steps, (1,), device=augmented.device),
            ] = True

    augmented = augmented.masked_fill(
        temporal_mask[:, :, None, None, None],
        0.0,
    )

    # Spatial-cell masking.
    spatial_mask = (
        torch.rand(batch_size, time_steps, height, width, 1, device=augmented.device)
        < 0.08
    )
    augmented = augmented.masked_fill(spatial_mask, 0.0)

    # Per-sample amplitude jitter and low Gaussian noise.
    amplitude = torch.empty(
        batch_size, 1, 1, 1, 1, device=augmented.device
    ).uniform_(0.90, 1.10)
    augmented = augmented * amplitude
    augmented = augmented + 0.03 * torch.randn_like(augmented)

    return augmented, temporal_mask


def nt_xent_loss(first, second, temperature):
    first = F.normalize(first, dim=-1)
    second = F.normalize(second, dim=-1)
    representations = torch.cat([first, second], dim=0)
    similarity = representations @ representations.T / temperature

    number = first.shape[0]
    diagonal_mask = torch.eye(2 * number, device=first.device, dtype=torch.bool)
    similarity = similarity.masked_fill(diagonal_mask, -1e9)

    positive_indices = torch.cat(
        [
            torch.arange(number, 2 * number, device=first.device),
            torch.arange(0, number, device=first.device),
        ]
    )
    return F.cross_entropy(similarity, positive_indices)


class RadarSSLModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = RadarTokenEncoder()
        dim = self.encoder.output_dim
        self.pool = TemporalAttentionPool(dim)
        self.projection = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Linear(dim, cfg.PROJECTION_DIM),
        )
        self.reconstruction = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim // 2),
            nn.GELU(),
            nn.Linear(dim // 2, 10),
        )

    def forward(self, radar):
        tokens = self.encoder(radar)
        pooled, _ = self.pool(tokens)
        return {
            "tokens": tokens,
            "projection": self.projection(pooled),
            "reconstruction": self.reconstruction(tokens),
        }


def make_grad_scaler():
    enabled = cfg.AMP and DEVICE.type == "cuda"
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=enabled)


def autocast_context():
    if cfg.AMP and DEVICE.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def pretrain_radar_ssl(model, loader, epochs):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.SSL_LEARNING_RATE,
        weight_decay=cfg.WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(epochs, 1)
    )
    scaler = make_grad_scaler()
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_total = 0.0
        running_reconstruction = 0.0
        running_contrastive = 0.0
        seen = 0

        for batch_data in loader:
            radar = batch_data["radar"].to(
                DEVICE, non_blocking=DEVICE.type == "cuda"
            )
            target_statistics = radar_frame_statistics(radar).detach()
            first_view, first_mask = augment_radar_for_ssl(radar)
            second_view, second_mask = augment_radar_for_ssl(radar)

            optimizer.zero_grad(set_to_none=True)

            with autocast_context():
                first_output = model(first_view)
                second_output = model(second_view)

                first_reconstruction_error = F.smooth_l1_loss(
                    first_output["reconstruction"],
                    target_statistics,
                    reduction="none",
                ).mean(dim=-1)
                second_reconstruction_error = F.smooth_l1_loss(
                    second_output["reconstruction"],
                    target_statistics,
                    reduction="none",
                ).mean(dim=-1)

                reconstruction_loss = 0.5 * (
                    first_reconstruction_error[first_mask].mean()
                    + second_reconstruction_error[second_mask].mean()
                )
                contrastive_loss = nt_xent_loss(
                    first_output["projection"],
                    second_output["projection"],
                    cfg.SSL_CONTRASTIVE_TEMP,
                )
                loss = (
                    cfg.SSL_RECON_WEIGHT * reconstruction_loss
                    + cfg.SSL_CONTRASTIVE_WEIGHT * contrastive_loss
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            batch_size = radar.shape[0]
            seen += batch_size
            running_total += float(loss.detach()) * batch_size
            running_reconstruction += float(reconstruction_loss.detach()) * batch_size
            running_contrastive += float(contrastive_loss.detach()) * batch_size

        scheduler.step()

        row = {
            "epoch": epoch,
            "ssl_loss": running_total / max(seen, 1),
            "reconstruction_loss": running_reconstruction / max(seen, 1),
            "contrastive_loss": running_contrastive / max(seen, 1),
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(row)
        print(
            f"SSL {epoch:02d}/{epochs} | "
            f"loss={row['ssl_loss']:.4f} | "
            f"recon={row['reconstruction_loss']:.4f} | "
            f"contrast={row['contrastive_loss']:.4f}"
        )

    return pd.DataFrame(history)

In [18]:
# ============================================================
# Run or skip radar SSL pretraining
# Fix: stable FP32 NT-Xent loss for mixed precision
# ============================================================

from contextlib import nullcontext

def nt_xent_loss(first, second, temperature):
    """
    Stable NT-Xent loss.

    The previous version used:
        similarity.masked_fill(diagonal_mask, -1e9)

    Under AMP, similarity can be float16, and -1e9 overflows float16.
    This version computes the contrastive similarity matrix in float32.
    """

    context = (
        torch.autocast(device_type="cuda", enabled=False)
        if first.is_cuda
        else nullcontext()
    )

    with context:
        first = F.normalize(first.float(), dim=-1)
        second = F.normalize(second.float(), dim=-1)

        number = first.shape[0]

        if number < 2:
            raise RuntimeError(
                "NT-Xent loss requires at least two samples in the batch."
            )

        representations = torch.cat(
            [first, second],
            dim=0,
        )

        similarity = (
            representations
            @ representations.T
        ) / float(temperature)

        diagonal_mask = torch.eye(
            2 * number,
            device=similarity.device,
            dtype=torch.bool,
        )

        # Safe for float32 and avoids FP16 overflow.
        similarity = similarity.masked_fill(
            diagonal_mask,
            torch.finfo(similarity.dtype).min,
        )

        positive_indices = torch.cat(
            [
                torch.arange(
                    number,
                    2 * number,
                    device=similarity.device,
                ),
                torch.arange(
                    0,
                    number,
                    device=similarity.device,
                ),
            ]
        ).long()

        loss = F.cross_entropy(
            similarity,
            positive_indices,
        )

    return loss


ssl_model = RadarSSLModel().to(DEVICE)
ssl_history = pd.DataFrame()

if cfg.RUN_SSL:
    ssl_history = pretrain_radar_ssl(
        ssl_model,
        train_loader,
        cfg.SSL_EPOCHS,
    )

    ssl_history.to_csv(
        Path(cfg.WORK_DIR) / "ssl_history.csv",
        index=False,
    )

    torch.save(
        ssl_model.encoder.state_dict(),
        Path(cfg.WORK_DIR) / "ssl_radar_encoder.pt",
    )

    print("Saved SSL radar encoder.")

else:
    print("SSL pretraining skipped.")

SSL 01/10 | loss=1.2279 | recon=0.1545 | contrast=3.0668
SSL 02/10 | loss=0.8974 | recon=0.1090 | contrast=2.2523
SSL 03/10 | loss=0.7037 | recon=0.0910 | contrast=1.7504
SSL 04/10 | loss=0.5785 | recon=0.0742 | contrast=1.4409
SSL 05/10 | loss=0.5213 | recon=0.0675 | contrast=1.2966
SSL 06/10 | loss=0.4966 | recon=0.0633 | contrast=1.2379
SSL 07/10 | loss=0.4822 | recon=0.0620 | contrast=1.2005
SSL 08/10 | loss=0.4547 | recon=0.0608 | contrast=1.1257
SSL 09/10 | loss=0.4488 | recon=0.0605 | contrast=1.1093
SSL 10/10 | loss=0.4612 | recon=0.0598 | contrast=1.1469
Saved SSL radar encoder.


## Shared training and evaluation utilities

The following functions keep model, labels, and class weights on the same device, use validation macro-F1 for early stopping, and return logits for calibration and uncertainty analysis.

In [19]:
# ============================================================
# Metrics, prediction, class weights, and supervised radar training
# ============================================================

def class_weights_for_frame(frame, device):
    counts = (
        frame["class_id"]
        .value_counts()
        .reindex(range(NUM_CLASSES), fill_value=0)
        .sort_index()
    )
    if (counts == 0).any():
        raise RuntimeError(
            f"Missing training classes: {counts[counts == 0].index.tolist()}"
        )
    weights = len(frame) / (NUM_CLASSES * counts.values.astype(np.float32))
    return torch.tensor(weights, dtype=torch.float32, device=device)


def metric_dictionary(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


@torch.inference_mode()
def predict_radar_model(model, loader, radar_corruption=None):
    model.eval()
    logits_list = []
    labels_list = []
    sample_ids = []
    subjects = []
    attentions = []

    for batch_data in loader:
        radar = batch_data["radar"].to(
            DEVICE, non_blocking=DEVICE.type == "cuda"
        )
        if radar_corruption is not None:
            radar = apply_radar_corruption(radar, radar_corruption)

        output = model(radar)
        logits_list.append(output["logits"].detach().cpu())
        labels_list.append(batch_data["label"].cpu())
        sample_ids.extend(batch_data["sample_id"])
        subjects.extend(batch_data["subject"])
        attentions.append(output["temporal_attention"].detach().cpu())

    logits = torch.cat(logits_list).numpy()
    labels = torch.cat(labels_list).numpy()
    probabilities = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()
    predictions = probabilities.argmax(axis=1)

    return {
        "logits": logits,
        "probs": probabilities,
        "y_true": labels,
        "y_pred": predictions,
        "sample_ids": np.asarray(sample_ids),
        "subjects": np.asarray(subjects),
        "temporal_attention": torch.cat(attentions).numpy(),
    }


@torch.inference_mode()
def predict_imu_model(model, loader):
    model.eval()
    logits_list = []
    labels_list = []
    sample_ids = []
    subjects = []

    for batch_data in loader:
        imu = batch_data["imu"].to(
            DEVICE, non_blocking=DEVICE.type == "cuda"
        )
        output = model(imu)
        logits_list.append(output["logits"].detach().cpu())
        labels_list.append(batch_data["label"].cpu())
        sample_ids.extend(batch_data["sample_id"])
        subjects.extend(batch_data["subject"])

    logits = torch.cat(logits_list).numpy()
    labels = torch.cat(labels_list).numpy()
    probabilities = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()

    return {
        "logits": logits,
        "probs": probabilities,
        "y_true": labels,
        "y_pred": probabilities.argmax(axis=1),
        "sample_ids": np.asarray(sample_ids),
        "subjects": np.asarray(subjects),
    }


def train_imu_supervised(
    model,
    train_loader_local,
    val_loader_local,
    train_frame,
    epochs,
    name,
):
    model = model.to(DEVICE)
    class_weights_local = class_weights_for_frame(train_frame, DEVICE)
    criterion = nn.CrossEntropyLoss(
        weight=None if cfg.USE_WEIGHTED_SAMPLER else class_weights_local,
        label_smoothing=cfg.LABEL_SMOOTHING,
    )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.LEARNING_RATE,
        weight_decay=cfg.WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=2, factor=0.5
    )
    scaler = make_grad_scaler()

    best_state = copy.deepcopy(model.state_dict())
    best_score = -np.inf
    stale_epochs = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        seen = 0

        for batch_data in train_loader_local:
            imu = batch_data["imu"].to(
                DEVICE, non_blocking=DEVICE.type == "cuda"
            )
            labels = batch_data["label"].to(
                DEVICE, non_blocking=DEVICE.type == "cuda"
            )

            optimizer.zero_grad(set_to_none=True)
            with autocast_context():
                output = model(imu)
                loss = criterion(output["logits"], labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            batch_size = labels.shape[0]
            seen += batch_size
            running_loss += float(loss.detach()) * batch_size

        validation_prediction = predict_imu_model(model, val_loader_local)
        validation = metric_dictionary(
            validation_prediction["y_true"],
            validation_prediction["y_pred"],
        )
        scheduler.step(validation["macro_f1"])

        row = {
            "epoch": epoch,
            "train_loss": running_loss / max(seen, 1),
            **{f"val_{key}": value for key, value in validation.items()},
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(row)

        print(
            f"{name} {epoch:02d}/{epochs} | "
            f"loss={row['train_loss']:.4f} | "
            f"val_macro_f1={validation['macro_f1']:.4f}"
        )

        if validation["macro_f1"] > best_score + 1e-6:
            best_score = validation["macro_f1"]
            best_state = copy.deepcopy(model.state_dict())
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= cfg.PATIENCE:
                print(f"{name}: early stopping.")
                break

    model.load_state_dict(best_state)
    return pd.DataFrame(history)


@torch.inference_mode()
def predict_teacher_model(model, loader, corruption=None):
    model.eval()
    logits_list = []
    labels_list = []
    reliability_list = []
    sample_ids = []
    subjects = []

    for batch_data in loader:
        radar = batch_data["radar"].to(
            DEVICE, non_blocking=DEVICE.type == "cuda"
        )
        imu = batch_data["imu"].to(
            DEVICE, non_blocking=DEVICE.type == "cuda"
        )

        if corruption is not None:
            radar, imu, _ = corrupt_modalities(
                radar,
                imu,
                forced_condition=corruption,
            )

        output = model(radar, imu)
        logits_list.append(output["logits"].detach().cpu())
        labels_list.append(batch_data["label"].cpu())
        reliability_list.append(output["reliability"].detach().cpu())
        sample_ids.extend(batch_data["sample_id"])
        subjects.extend(batch_data["subject"])

    logits = torch.cat(logits_list).numpy()
    labels = torch.cat(labels_list).numpy()
    probabilities = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()

    return {
        "logits": logits,
        "probs": probabilities,
        "y_true": labels,
        "y_pred": probabilities.argmax(axis=1),
        "sample_ids": np.asarray(sample_ids),
        "subjects": np.asarray(subjects),
        "reliability": torch.cat(reliability_list).numpy(),
    }


def evaluate_validation_radar(model, loader):
    prediction = predict_radar_model(model, loader)
    return metric_dictionary(prediction["y_true"], prediction["y_pred"])


def train_radar_supervised(
    model,
    train_loader_local,
    val_loader_local,
    train_frame,
    epochs,
    name,
):
    model = model.to(DEVICE)
    class_weights_local = class_weights_for_frame(train_frame, DEVICE)
    criterion = nn.CrossEntropyLoss(
        weight=None if cfg.USE_WEIGHTED_SAMPLER else class_weights_local,
        label_smoothing=cfg.LABEL_SMOOTHING,
    )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.LEARNING_RATE,
        weight_decay=cfg.WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=2, factor=0.5
    )
    scaler = make_grad_scaler()

    best_state = copy.deepcopy(model.state_dict())
    best_score = -np.inf
    stale_epochs = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        seen = 0

        for batch_data in train_loader_local:
            radar = batch_data["radar"].to(
                DEVICE, non_blocking=DEVICE.type == "cuda"
            )
            labels = batch_data["label"].to(
                DEVICE, non_blocking=DEVICE.type == "cuda"
            )

            optimizer.zero_grad(set_to_none=True)
            with autocast_context():
                output = model(radar)
                loss = criterion(output["logits"], labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            batch_size = labels.shape[0]
            seen += batch_size
            running_loss += float(loss.detach()) * batch_size

        validation = evaluate_validation_radar(model, val_loader_local)
        scheduler.step(validation["macro_f1"])

        row = {
            "epoch": epoch,
            "train_loss": running_loss / max(seen, 1),
            **{f"val_{key}": value for key, value in validation.items()},
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(row)

        print(
            f"{name} {epoch:02d}/{epochs} | "
            f"loss={row['train_loss']:.4f} | "
            f"val_macro_f1={validation['macro_f1']:.4f}"
        )

        if validation["macro_f1"] > best_score + 1e-6:
            best_score = validation["macro_f1"]
            best_state = copy.deepcopy(model.state_dict())
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= cfg.PATIENCE:
                print(f"{name}: early stopping.")
                break

    model.load_state_dict(best_state)
    return pd.DataFrame(history)

In [20]:
# ============================================================
# Train scratch and SSL-initialised radar baselines
# ============================================================

scratch_radar_model = RadarOnlyModel(NUM_CLASSES).to(DEVICE)
scratch_history = train_radar_supervised(
    scratch_radar_model,
    train_loader,
    val_loader,
    train_df,
    cfg.BASELINE_EPOCHS,
    "scratch_radar",
)
scratch_history.to_csv(
    Path(cfg.WORK_DIR) / "scratch_radar_history.csv",
    index=False,
)

ssl_radar_model = RadarOnlyModel(NUM_CLASSES).to(DEVICE)
if cfg.RUN_SSL:
    ssl_radar_model.encoder.load_state_dict(
        ssl_model.encoder.state_dict(),
        strict=True,
    )

ssl_radar_history = train_radar_supervised(
    ssl_radar_model,
    train_loader,
    val_loader,
    train_df,
    cfg.BASELINE_EPOCHS,
    "ssl_radar",
)
ssl_radar_history.to_csv(
    Path(cfg.WORK_DIR) / "ssl_radar_history.csv",
    index=False,
)

torch.save(
    scratch_radar_model.state_dict(),
    Path(cfg.WORK_DIR) / "scratch_radar_model.pt",
)
torch.save(
    ssl_radar_model.state_dict(),
    Path(cfg.WORK_DIR) / "ssl_radar_model.pt",
)

scratch_radar 01/14 | loss=2.2912 | val_macro_f1=0.0182
scratch_radar 02/14 | loss=1.8107 | val_macro_f1=0.0722
scratch_radar 03/14 | loss=1.5425 | val_macro_f1=0.2339
scratch_radar 04/14 | loss=1.2615 | val_macro_f1=0.3219
scratch_radar 05/14 | loss=1.1546 | val_macro_f1=0.4884
scratch_radar 06/14 | loss=0.9744 | val_macro_f1=0.5793
scratch_radar 07/14 | loss=0.9242 | val_macro_f1=0.5947
scratch_radar 08/14 | loss=0.7753 | val_macro_f1=0.5709
scratch_radar 09/14 | loss=0.7643 | val_macro_f1=0.6534
scratch_radar 10/14 | loss=0.6401 | val_macro_f1=0.6075
scratch_radar 11/14 | loss=0.6018 | val_macro_f1=0.6088
scratch_radar 12/14 | loss=0.6289 | val_macro_f1=0.6534
scratch_radar 13/14 | loss=0.5251 | val_macro_f1=0.5719
scratch_radar 14/14 | loss=0.4643 | val_macro_f1=0.6352
scratch_radar: early stopping.
ssl_radar 01/14 | loss=2.1565 | val_macro_f1=0.2470
ssl_radar 02/14 | loss=1.7324 | val_macro_f1=0.4669
ssl_radar 03/14 | loss=1.4438 | val_macro_f1=0.5382
ssl_radar 04/14 | loss=1.2367

## Strong unimodal and late-fusion baselines

The previous experiment found IMU-only and probability-level late fusion to be strong competitors. This notebook therefore trains a dedicated IMU-only model and later evaluates an equal-weight radar–IMU late-fusion baseline.

In [21]:
# ============================================================
# Train the strong IMU-only baseline
# ============================================================

imu_model = None
imu_history = pd.DataFrame()

if cfg.RUN_STRONG_BASELINES:
    imu_model = IMUOnlyModel(NUM_CLASSES).to(DEVICE)
    imu_history = train_imu_supervised(
        imu_model,
        train_loader,
        val_loader,
        train_df,
        cfg.STRONG_BASELINE_EPOCHS,
        "imu_only",
    )
    imu_history.to_csv(
        Path(cfg.WORK_DIR) / "imu_only_history.csv",
        index=False,
    )
    torch.save(
        imu_model.state_dict(),
        Path(cfg.WORK_DIR) / "imu_only_model.pt",
    )
else:
    print("Strong IMU and late-fusion baselines disabled.")

imu_only 01/14 | loss=2.2206 | val_macro_f1=0.2157
imu_only 02/14 | loss=1.3316 | val_macro_f1=0.3562
imu_only 03/14 | loss=0.8398 | val_macro_f1=0.7000
imu_only 04/14 | loss=0.5577 | val_macro_f1=0.7229
imu_only 05/14 | loss=0.4923 | val_macro_f1=0.7286
imu_only 06/14 | loss=0.3738 | val_macro_f1=0.7594
imu_only 07/14 | loss=0.3073 | val_macro_f1=0.7530
imu_only 08/14 | loss=0.2670 | val_macro_f1=0.7725
imu_only 09/14 | loss=0.2548 | val_macro_f1=0.8003
imu_only 10/14 | loss=0.2374 | val_macro_f1=0.8226
imu_only 11/14 | loss=0.2274 | val_macro_f1=0.8226
imu_only 12/14 | loss=0.2156 | val_macro_f1=0.8226
imu_only 13/14 | loss=0.2156 | val_macro_f1=0.8226
imu_only 14/14 | loss=0.2220 | val_macro_f1=0.8226


## Stage B — corruption-supervised reliability teacher

The teacher is trained on both clean and corrupted modality pairs. Synthetic corruption provides an explicit quality target for the reliability head:

- temporal frame masking;
- missing IMU sensors;
- channel dropout;
- spatial radar masking;
- Gaussian noise;
- amplitude attenuation.

The teacher must classify correctly, preserve consistency with its clean prediction, and lower the reliability assigned to a degraded modality.

In [22]:
# ============================================================
# Corruption curriculum with reliability targets
# ============================================================

def _normalise_quality(radar_quality, imu_quality):
    quality = torch.stack([radar_quality, imu_quality], dim=-1)
    quality = quality.clamp_min(0.02)
    return quality / quality.sum(dim=-1, keepdim=True)


def corrupt_modalities(radar, imu, forced_condition=None):
    radar_corrupted = radar.clone()
    imu_corrupted = imu.clone()

    batch_size, time_steps = radar.shape[:2]
    radar_quality = torch.ones(batch_size, time_steps, device=radar.device)
    imu_quality = torch.ones(batch_size, time_steps, device=radar.device)

    conditions = [
        "clean",
        "imu_frame_mask",
        "imu_sensor_mask",
        "imu_noise",
        "radar_frame_mask",
        "radar_spatial_mask",
        "radar_channel_drop",
        "radar_noise",
        "both_mild",
    ]

    for sample_index in range(batch_size):
        if forced_condition is None:
            if random.random() > cfg.CORRUPTION_PROB:
                condition = "clean"
            else:
                condition = random.choice(conditions[1:])
        else:
            condition = forced_condition

        if condition == "clean":
            continue

        if condition in {"imu_frame_mask", "both_mild"}:
            severity = 0.20 if condition == "both_mild" else random.uniform(0.15, 0.55)
            count = max(1, int(round(time_steps * severity)))
            indices = torch.randperm(time_steps, device=imu.device)[:count]
            imu_corrupted[sample_index, indices] = 0.0
            imu_quality[sample_index, indices] = 0.05

        if condition == "imu_sensor_mask":
            severity = random.uniform(1 / 6, 0.50)
            number_sensors = max(1, int(round(6 * severity)))
            sensors = torch.randperm(6, device=imu.device)[:number_sensors]
            for sensor in sensors:
                start = int(sensor.item()) * 12
                imu_corrupted[sample_index, :, start:start + 12] = 0.0
            imu_quality[sample_index] *= max(0.05, 1.0 - number_sensors / 6.0)

        if condition in {"imu_noise", "both_mild"}:
            sigma = 0.15 if condition == "both_mild" else random.uniform(0.15, 0.65)
            imu_corrupted[sample_index] += sigma * torch.randn_like(
                imu_corrupted[sample_index]
            )
            imu_quality[sample_index] *= math.exp(-sigma)

        if condition in {"radar_frame_mask", "both_mild"}:
            severity = 0.12 if condition == "both_mild" else random.uniform(0.10, 0.45)
            count = max(1, int(round(time_steps * severity)))
            indices = torch.randperm(time_steps, device=radar.device)[:count]
            radar_corrupted[sample_index, indices] = 0.0
            radar_quality[sample_index, indices] = 0.05

        if condition == "radar_spatial_mask":
            height, width = radar.shape[2], radar.shape[3]
            box_h = max(1, int(round(height * random.uniform(0.20, 0.50))))
            box_w = max(1, int(round(width * random.uniform(0.20, 0.50))))
            top = random.randint(0, max(height - box_h, 0))
            left = random.randint(0, max(width - box_w, 0))
            radar_corrupted[
                sample_index, :, top:top + box_h, left:left + box_w, :
            ] = 0.0
            removed_fraction = (box_h * box_w) / (height * width)
            radar_quality[sample_index] *= max(0.05, 1.0 - removed_fraction)

        if condition == "radar_channel_drop":
            channel = random.randrange(radar.shape[-1])
            radar_corrupted[sample_index, :, :, :, channel] = 0.0
            radar_quality[sample_index] *= 0.80

        if condition == "radar_noise":
            sigma = random.uniform(0.10, 0.50)
            radar_corrupted[sample_index] += sigma * torch.randn_like(
                radar_corrupted[sample_index]
            )
            radar_quality[sample_index] *= math.exp(-sigma)

    reliability_target = _normalise_quality(radar_quality, imu_quality)
    return radar_corrupted, imu_corrupted, reliability_target


def apply_radar_corruption(radar, condition):
    corrupted = radar.clone()
    batch_size, time_steps, height, width, channels = corrupted.shape

    if condition == "clean":
        return corrupted
    if condition == "frame_mask_10":
        probability = 0.10
        mask = torch.rand(batch_size, time_steps, device=radar.device) < probability
        return corrupted.masked_fill(mask[:, :, None, None, None], 0.0)
    if condition == "frame_mask_30":
        probability = 0.30
        mask = torch.rand(batch_size, time_steps, device=radar.device) < probability
        return corrupted.masked_fill(mask[:, :, None, None, None], 0.0)
    if condition == "frame_mask_50":
        probability = 0.50
        mask = torch.rand(batch_size, time_steps, device=radar.device) < probability
        return corrupted.masked_fill(mask[:, :, None, None, None], 0.0)
    if condition == "channel_drop":
        channel = random.randrange(channels)
        corrupted[..., channel] = 0.0
        return corrupted
    if condition == "gaussian_025":
        return corrupted + 0.25 * torch.randn_like(corrupted)
    if condition == "gaussian_050":
        return corrupted + 0.50 * torch.randn_like(corrupted)
    if condition == "spatial_mask":
        top = height // 4
        left = width // 4
        corrupted[:, :, top:top + height // 2, left:left + width // 2, :] = 0.0
        return corrupted
    raise ValueError(f"Unknown radar corruption: {condition}")

In [23]:
# ============================================================
# Reliability-teacher training
# ============================================================

@torch.inference_mode()
def evaluate_teacher(model, loader):
    prediction = predict_teacher_model(model, loader)
    return metric_dictionary(prediction["y_true"], prediction["y_pred"])


def teacher_consistency_loss(clean_logits, corrupted_logits):
    clean_probability = torch.softmax(clean_logits.detach(), dim=-1)
    return F.kl_div(
        F.log_softmax(corrupted_logits, dim=-1),
        clean_probability,
        reduction="batchmean",
    )


def train_reliability_teacher(
    model,
    train_loader_local,
    val_loader_local,
    train_frame,
    epochs,
):
    model = model.to(DEVICE)
    class_weights_local = class_weights_for_frame(train_frame, DEVICE)
    criterion = nn.CrossEntropyLoss(
        weight=None if cfg.USE_WEIGHTED_SAMPLER else class_weights_local,
        label_smoothing=cfg.LABEL_SMOOTHING,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.LEARNING_RATE,
        weight_decay=cfg.WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=2, factor=0.5
    )
    scaler = make_grad_scaler()

    best_state = copy.deepcopy(model.state_dict())
    best_score = -np.inf
    stale_epochs = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        totals = defaultdict(float)
        seen = 0

        for batch_data in train_loader_local:
            radar = batch_data["radar"].to(
                DEVICE, non_blocking=DEVICE.type == "cuda"
            )
            imu = batch_data["imu"].to(
                DEVICE, non_blocking=DEVICE.type == "cuda"
            )
            labels = batch_data["label"].to(
                DEVICE, non_blocking=DEVICE.type == "cuda"
            )

            corrupted_radar, corrupted_imu, reliability_target = corrupt_modalities(
                radar, imu
            )

            optimizer.zero_grad(set_to_none=True)

            with autocast_context():
                clean_output = model(radar, imu)
                corrupted_output = model(corrupted_radar, corrupted_imu)

                clean_classification = criterion(clean_output["logits"], labels)
                corrupted_classification = criterion(
                    corrupted_output["logits"], labels
                )
                auxiliary_loss = 0.25 * (
                    criterion(clean_output["radar_aux_logits"], labels)
                    + criterion(clean_output["imu_aux_logits"], labels)
                    + criterion(corrupted_output["radar_aux_logits"], labels)
                    + criterion(corrupted_output["imu_aux_logits"], labels)
                )
                reliability_loss = F.smooth_l1_loss(
                    corrupted_output["reliability"],
                    reliability_target,
                )
                consistency_loss = teacher_consistency_loss(
                    clean_output["logits"],
                    corrupted_output["logits"],
                )

                loss = (
                    0.50 * clean_classification
                    + 0.50 * corrupted_classification
                    + cfg.TEACHER_AUX_WEIGHT * auxiliary_loss
                    + cfg.RELIABILITY_WEIGHT * reliability_loss
                    + cfg.CONSISTENCY_WEIGHT * consistency_loss
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            batch_size = labels.shape[0]
            seen += batch_size
            totals["loss"] += float(loss.detach()) * batch_size
            totals["clean_ce"] += float(clean_classification.detach()) * batch_size
            totals["corrupt_ce"] += float(corrupted_classification.detach()) * batch_size
            totals["reliability"] += float(reliability_loss.detach()) * batch_size
            totals["consistency"] += float(consistency_loss.detach()) * batch_size

        validation = evaluate_teacher(model, val_loader_local)
        scheduler.step(validation["macro_f1"])

        row = {
            "epoch": epoch,
            **{key: value / max(seen, 1) for key, value in totals.items()},
            **{f"val_{key}": value for key, value in validation.items()},
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(row)

        print(
            f"teacher {epoch:02d}/{epochs} | "
            f"loss={row['loss']:.4f} | "
            f"rel={row['reliability']:.4f} | "
            f"val_macro_f1={validation['macro_f1']:.4f}"
        )

        if validation["macro_f1"] > best_score + 1e-6:
            best_score = validation["macro_f1"]
            best_state = copy.deepcopy(model.state_dict())
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= cfg.PATIENCE:
                print("Teacher: early stopping.")
                break

    model.load_state_dict(best_state)
    return pd.DataFrame(history)


teacher_model = ReliabilityAwareTeacher(NUM_CLASSES).to(DEVICE)

if cfg.RUN_SSL:
    teacher_model.radar_encoder.load_state_dict(
        ssl_model.encoder.state_dict(),
        strict=True,
    )

teacher_history = train_reliability_teacher(
    teacher_model,
    train_loader,
    val_loader,
    train_df,
    cfg.TEACHER_EPOCHS,
)

teacher_history.to_csv(
    Path(cfg.WORK_DIR) / "reliability_teacher_history.csv",
    index=False,
)
torch.save(
    teacher_model.state_dict(),
    Path(cfg.WORK_DIR) / "reliability_teacher.pt",
)

teacher 01/20 | loss=2.5569 | rel=0.0161 | val_macro_f1=0.0642
teacher 02/20 | loss=1.5154 | rel=0.0344 | val_macro_f1=0.0550
teacher 03/20 | loss=1.0963 | rel=0.0648 | val_macro_f1=0.2036
teacher 04/20 | loss=0.8860 | rel=0.0573 | val_macro_f1=0.9006
teacher 05/20 | loss=0.7539 | rel=0.0535 | val_macro_f1=0.9029
teacher 06/20 | loss=0.7429 | rel=0.0399 | val_macro_f1=0.9029
teacher 07/20 | loss=0.6749 | rel=0.0386 | val_macro_f1=0.9006
teacher 08/20 | loss=0.7014 | rel=0.0332 | val_macro_f1=0.9006
teacher 09/20 | loss=0.7303 | rel=0.0325 | val_macro_f1=0.8997
teacher 10/20 | loss=0.7066 | rel=0.0313 | val_macro_f1=0.9260
teacher 11/20 | loss=0.6436 | rel=0.0321 | val_macro_f1=0.9260
teacher 12/20 | loss=0.6283 | rel=0.0409 | val_macro_f1=0.9260
teacher 13/20 | loss=0.6668 | rel=0.0341 | val_macro_f1=0.9260
teacher 14/20 | loss=0.6303 | rel=0.0300 | val_macro_f1=0.9260
teacher 15/20 | loss=0.6306 | rel=0.0282 | val_macro_f1=0.9260
Teacher: early stopping.


In [24]:
# ============================================================
# Compute multimodal class prototypes from the trained teacher
# ============================================================

@torch.inference_mode()
def compute_teacher_prototypes(model, loader, num_classes):
    model.eval()
    feature_sums = None
    counts = torch.zeros(num_classes, device=DEVICE)

    for batch_data in loader:
        radar = batch_data["radar"].to(
            DEVICE, non_blocking=DEVICE.type == "cuda"
        )
        imu = batch_data["imu"].to(
            DEVICE, non_blocking=DEVICE.type == "cuda"
        )
        labels = batch_data["label"].to(
            DEVICE, non_blocking=DEVICE.type == "cuda"
        )
        features = F.normalize(model(radar, imu)["pooled"], dim=-1)

        if feature_sums is None:
            feature_sums = torch.zeros(
                num_classes,
                features.shape[-1],
                device=DEVICE,
            )

        feature_sums.index_add_(0, labels, features)
        counts.index_add_(0, labels, torch.ones_like(labels, dtype=torch.float32))

    if (counts == 0).any():
        raise RuntimeError(
            f"Prototype classes without samples: "
            f"{torch.where(counts == 0)[0].tolist()}"
        )

    prototypes = feature_sums / counts.unsqueeze(-1)
    return F.normalize(prototypes, dim=-1)


teacher_prototypes = compute_teacher_prototypes(
    teacher_model,
    train_loader,
    NUM_CLASSES,
)

torch.save(
    teacher_prototypes.detach().cpu(),
    Path(cfg.WORK_DIR) / "teacher_class_prototypes.pt",
)

print("Teacher prototypes:", tuple(teacher_prototypes.shape))

Teacher prototypes: (10, 128)


## Stage C — confidence-adaptive temporal prototype distillation

For each training sample, teacher entropy determines how strongly its soft prediction should influence the student. The student also matches:

- teacher tokens through soft temporal alignment;
- teacher sample-to-sample relations;
- the correct multimodal class prototype;
- exercise labels while suppressing subject identity.

In [25]:
# ============================================================
# Distillation losses and student training
# ============================================================

@dataclass
class StudentLossConfiguration:
    use_logit: bool = True
    use_temporal: bool = True
    use_relation: bool = True
    use_prototype: bool = True
    use_subject_adversary: bool = True
    name: str = "full_rt_pd"


def teacher_confidence(logits):
    probability = torch.softmax(logits, dim=-1)
    entropy = -torch.sum(
        probability * torch.log(probability.clamp_min(1e-8)),
        dim=-1,
    )
    return (1.0 - entropy / math.log(logits.shape[-1])).clamp(0.0, 1.0)


def confidence_weighted_kd(student_logits, teacher_logits, confidence):
    temperature = cfg.KD_TEMPERATURE
    per_sample = F.kl_div(
        F.log_softmax(student_logits / temperature, dim=-1),
        F.softmax(teacher_logits / temperature, dim=-1),
        reduction="none",
    ).sum(dim=-1) * (temperature ** 2)

    normalised_weight = confidence / confidence.mean().clamp_min(1e-6)
    return torch.mean(per_sample * normalised_weight.detach())


def soft_temporal_alignment_loss(student_tokens, teacher_tokens, confidence):
    student_normalised = F.normalize(student_tokens, dim=-1)
    teacher_normalised = F.normalize(teacher_tokens, dim=-1)

    similarity = torch.matmul(
        student_normalised,
        teacher_normalised.transpose(1, 2),
    ) / cfg.TEMPORAL_ALIGNMENT_TEMP

    alignment = torch.softmax(similarity, dim=-1)
    aligned_teacher = torch.matmul(alignment, teacher_normalised)

    per_time = 1.0 - F.cosine_similarity(
        student_normalised,
        aligned_teacher.detach(),
        dim=-1,
    )
    per_sample = per_time.mean(dim=-1)
    return torch.mean(per_sample * confidence.detach())


def relation_distillation_loss(student_features, teacher_features):
    student_features = F.normalize(student_features, dim=-1)
    teacher_features = F.normalize(teacher_features, dim=-1)

    student_relation = student_features @ student_features.T
    teacher_relation = teacher_features @ teacher_features.T

    return F.mse_loss(student_relation, teacher_relation.detach())


def prototype_distillation_loss(student_features, labels, prototypes):
    student_features = F.normalize(student_features, dim=-1)
    prototype_logits = (
        student_features @ prototypes.detach().T
    ) / cfg.PROTOTYPE_TEMP
    classification = F.cross_entropy(prototype_logits, labels)
    positive_prototypes = prototypes.detach()[labels]
    attraction = (
        1.0 - F.cosine_similarity(student_features, positive_prototypes, dim=-1)
    ).mean()
    return classification + 0.25 * attraction


def subject_ids_from_batch(subjects):
    ids = []
    for subject in subjects:
        if subject not in SUBJECT_TO_TRAIN_ID:
            raise KeyError(f"Unknown training subject: {subject}")
        ids.append(SUBJECT_TO_TRAIN_ID[subject])
    return torch.tensor(ids, dtype=torch.long, device=DEVICE)


def adversarial_schedule(epoch, epochs):
    progress = (epoch - 1) / max(epochs - 1, 1)
    return float(2.0 / (1.0 + math.exp(-10.0 * progress)) - 1.0)


@torch.inference_mode()
def evaluate_student(model, loader):
    prediction = predict_radar_model(model, loader)
    return metric_dictionary(prediction["y_true"], prediction["y_pred"])


def train_distilled_student(
    student,
    teacher,
    prototypes,
    train_loader_local,
    val_loader_local,
    train_frame,
    epochs,
    loss_configuration,
):
    student = student.to(DEVICE)
    teacher = teacher.to(DEVICE).eval()

    for parameter in teacher.parameters():
        parameter.requires_grad_(False)

    class_weights_local = class_weights_for_frame(train_frame, DEVICE)
    hard_criterion = nn.CrossEntropyLoss(
        weight=None if cfg.USE_WEIGHTED_SAMPLER else class_weights_local,
        label_smoothing=cfg.LABEL_SMOOTHING,
    )

    optimizer = torch.optim.AdamW(
        student.parameters(),
        lr=cfg.LEARNING_RATE,
        weight_decay=cfg.WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=2, factor=0.5
    )
    scaler = make_grad_scaler()

    best_state = copy.deepcopy(student.state_dict())
    best_score = -np.inf
    stale_epochs = 0
    history = []

    for epoch in range(1, epochs + 1):
        student.train()
        totals = defaultdict(float)
        seen = 0
        grl_coefficient = adversarial_schedule(epoch, epochs)

        # Gradually introduce the more structured losses.
        distillation_ramp = min(1.0, epoch / max(epochs * 0.30, 1.0))

        for batch_data in train_loader_local:
            radar = batch_data["radar"].to(
                DEVICE, non_blocking=DEVICE.type == "cuda"
            )
            imu = batch_data["imu"].to(
                DEVICE, non_blocking=DEVICE.type == "cuda"
            )
            labels = batch_data["label"].to(
                DEVICE, non_blocking=DEVICE.type == "cuda"
            )

            subject_labels = (
                subject_ids_from_batch(batch_data["subject"])
                if loss_configuration.use_subject_adversary
                else None
            )

            with torch.inference_mode():
                teacher_output = teacher(radar, imu)
                confidence = teacher_confidence(teacher_output["logits"])

            optimizer.zero_grad(set_to_none=True)

            with autocast_context():
                student_output = student(
                    radar,
                    grl_coefficient=(
                        grl_coefficient
                        if loss_configuration.use_subject_adversary
                        else 0.0
                    ),
                )

                hard_loss = hard_criterion(student_output["logits"], labels)
                total_loss = hard_loss

                logit_loss = torch.zeros((), device=DEVICE)
                temporal_loss = torch.zeros((), device=DEVICE)
                relation_loss = torch.zeros((), device=DEVICE)
                prototype_loss = torch.zeros((), device=DEVICE)
                subject_loss = torch.zeros((), device=DEVICE)

                if loss_configuration.use_logit:
                    logit_loss = confidence_weighted_kd(
                        student_output["logits"],
                        teacher_output["logits"],
                        confidence,
                    )
                    total_loss = (
                        total_loss
                        + distillation_ramp * cfg.KD_LOGIT_WEIGHT * logit_loss
                    )

                if loss_configuration.use_temporal:
                    temporal_loss = soft_temporal_alignment_loss(
                        student_output["tokens"],
                        teacher_output["fused_tokens"],
                        confidence,
                    )
                    total_loss = (
                        total_loss
                        + distillation_ramp
                        * cfg.KD_TEMPORAL_WEIGHT
                        * temporal_loss
                    )

                if loss_configuration.use_relation:
                    relation_loss = relation_distillation_loss(
                        student_output["pooled"],
                        teacher_output["pooled"],
                    )
                    total_loss = (
                        total_loss
                        + distillation_ramp
                        * cfg.KD_RELATION_WEIGHT
                        * relation_loss
                    )

                if loss_configuration.use_prototype:
                    prototype_loss = prototype_distillation_loss(
                        student_output["pooled"],
                        labels,
                        prototypes,
                    )
                    total_loss = (
                        total_loss
                        + distillation_ramp
                        * cfg.KD_PROTOTYPE_WEIGHT
                        * prototype_loss
                    )

                if (
                    loss_configuration.use_subject_adversary
                    and student_output["subject_logits"] is not None
                ):
                    subject_loss = F.cross_entropy(
                        student_output["subject_logits"],
                        subject_labels,
                    )
                    total_loss = (
                        total_loss
                        + cfg.SUBJECT_ADV_WEIGHT * subject_loss
                    )

            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(student.parameters(), cfg.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            batch_size = labels.shape[0]
            seen += batch_size
            totals["loss"] += float(total_loss.detach()) * batch_size
            totals["hard"] += float(hard_loss.detach()) * batch_size
            totals["logit"] += float(logit_loss.detach()) * batch_size
            totals["temporal"] += float(temporal_loss.detach()) * batch_size
            totals["relation"] += float(relation_loss.detach()) * batch_size
            totals["prototype"] += float(prototype_loss.detach()) * batch_size
            totals["subject"] += float(subject_loss.detach()) * batch_size
            totals["teacher_confidence"] += float(confidence.mean()) * batch_size

        validation = evaluate_student(student, val_loader_local)
        scheduler.step(validation["macro_f1"])

        row = {
            "epoch": epoch,
            **{key: value / max(seen, 1) for key, value in totals.items()},
            **{f"val_{key}": value for key, value in validation.items()},
            "grl_coefficient": grl_coefficient,
            "distillation_ramp": distillation_ramp,
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(row)

        print(
            f"{loss_configuration.name} {epoch:02d}/{epochs} | "
            f"loss={row['loss']:.4f} | "
            f"conf={row['teacher_confidence']:.3f} | "
            f"val_macro_f1={validation['macro_f1']:.4f}"
        )

        if validation["macro_f1"] > best_score + 1e-6:
            best_score = validation["macro_f1"]
            best_state = copy.deepcopy(student.state_dict())
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= cfg.PATIENCE:
                print(f"{loss_configuration.name}: early stopping.")
                break

    student.load_state_dict(best_state)
    return pd.DataFrame(history)

In [27]:
# ============================================================
# Train the full RTPD-Net radar-only student
# Fix: clone teacher inference tensors before distillation losses
# ============================================================

from contextlib import nullcontext


# ------------------------------------------------------------
# Helper: convert inference/no_grad teacher tensors into normal
# detached tensors that can safely be used inside autograd losses.
# ------------------------------------------------------------

def _safe_teacher_tensor(tensor, dtype=torch.float32):
    if tensor is None:
        return None

    return (
        tensor
        .detach()
        .clone()
        .to(dtype=dtype)
    )


# ------------------------------------------------------------
# Safe replacement: teacher confidence
# ------------------------------------------------------------

def teacher_confidence(teacher_logits):
    with torch.autocast(
        device_type="cuda",
        enabled=False,
    ) if teacher_logits.is_cuda else nullcontext():

        logits = _safe_teacher_tensor(
            teacher_logits,
            dtype=torch.float32,
        )

        probability = torch.softmax(
            logits,
            dim=-1,
        )

        entropy = -(
            probability
            * probability.clamp_min(1e-8).log()
        ).sum(dim=-1)

        confidence = (
            1.0
            - entropy
            / math.log(
                logits.shape[-1]
            )
        )

        return confidence.clamp(
            0.0,
            1.0,
        )


# ------------------------------------------------------------
# Safe replacement: logit KD
# ------------------------------------------------------------

def confidence_weighted_kd(
    student_logits,
    teacher_logits,
    confidence,
):
    temperature = float(
        getattr(
            cfg,
            "KD_TEMPERATURE",
            4.0,
        )
    )

    with torch.autocast(
        device_type="cuda",
        enabled=False,
    ) if student_logits.is_cuda else nullcontext():

        student_logits = student_logits.float()

        teacher_logits = _safe_teacher_tensor(
            teacher_logits,
            dtype=torch.float32,
        )

        confidence = _safe_teacher_tensor(
            confidence,
            dtype=torch.float32,
        ).to(student_logits.device)

        confidence = confidence / confidence.mean().clamp_min(
            1e-6
        )

        per_sample_kd = F.kl_div(
            F.log_softmax(
                student_logits / temperature,
                dim=-1,
            ),
            F.softmax(
                teacher_logits / temperature,
                dim=-1,
            ),
            reduction="none",
        ).sum(dim=-1) * (
            temperature ** 2
        )

        return torch.mean(
            per_sample_kd
            * confidence
        )


# ------------------------------------------------------------
# Safe replacement: temporal alignment loss
# ------------------------------------------------------------

def soft_temporal_alignment_loss(
    student_tokens,
    teacher_tokens,
    confidence,
):
    """
    Safe temporal distillation.

    The original error happened because teacher_tokens were created
    under torch.inference_mode(). This version clones teacher_tokens
    into a normal detached tensor before computing any autograd-tracked
    operation with student_tokens.
    """

    temperature = float(
        getattr(
            cfg,
            "TEMPORAL_ALIGNMENT_TEMP",
            0.20,
        )
    )

    with torch.autocast(
        device_type="cuda",
        enabled=False,
    ) if student_tokens.is_cuda else nullcontext():

        student_tokens = F.normalize(
            student_tokens.float(),
            dim=-1,
        )

        teacher_tokens = F.normalize(
            _safe_teacher_tensor(
                teacher_tokens,
                dtype=torch.float32,
            ).to(student_tokens.device),
            dim=-1,
        )

        confidence = _safe_teacher_tensor(
            confidence,
            dtype=torch.float32,
        ).to(student_tokens.device)

        confidence = confidence / confidence.mean().clamp_min(
            1e-6
        )

        batch_size = student_tokens.shape[0]
        student_time = student_tokens.shape[1]
        teacher_time = teacher_tokens.shape[1]

        similarity = torch.bmm(
            student_tokens,
            teacher_tokens.transpose(1, 2),
        ) / temperature

        if student_time == teacher_time:
            target = torch.arange(
                student_time,
                device=student_tokens.device,
            )
        else:
            target = torch.linspace(
                0,
                teacher_time - 1,
                steps=student_time,
                device=student_tokens.device,
            ).round().long()

        target = target.unsqueeze(0).repeat(
            batch_size,
            1,
        )

        per_time = F.cross_entropy(
            similarity.reshape(
                batch_size * student_time,
                teacher_time,
            ),
            target.reshape(
                batch_size * student_time,
            ),
            reduction="none",
        ).reshape(
            batch_size,
            student_time,
        )

        per_sample = per_time.mean(dim=1)

        return torch.mean(
            per_sample
            * confidence
        )


# ------------------------------------------------------------
# Safe replacement: relation distillation
# ------------------------------------------------------------

def relation_distillation_loss(
    student_features,
    teacher_features,
):
    with torch.autocast(
        device_type="cuda",
        enabled=False,
    ) if student_features.is_cuda else nullcontext():

        student_features = F.normalize(
            student_features.float(),
            dim=-1,
        )

        teacher_features = F.normalize(
            _safe_teacher_tensor(
                teacher_features,
                dtype=torch.float32,
            ).to(student_features.device),
            dim=-1,
        )

        student_relation = torch.cdist(
            student_features,
            student_features,
            p=2,
        )

        teacher_relation = torch.cdist(
            teacher_features,
            teacher_features,
            p=2,
        )

        student_relation = student_relation / student_relation.mean().clamp_min(
            1e-6
        )

        teacher_relation = teacher_relation / teacher_relation.mean().clamp_min(
            1e-6
        )

        return F.smooth_l1_loss(
            student_relation,
            teacher_relation,
        )


# ------------------------------------------------------------
# Safe replacement: prototype distillation
# ------------------------------------------------------------

def prototype_distillation_loss(
    student_features,
    labels,
    prototypes,
):
    temperature = float(
        getattr(
            cfg,
            "PROTOTYPE_TEMP",
            0.20,
        )
    )

    with torch.autocast(
        device_type="cuda",
        enabled=False,
    ) if student_features.is_cuda else nullcontext():

        student_features = F.normalize(
            student_features.float(),
            dim=-1,
        )

        prototypes = F.normalize(
            _safe_teacher_tensor(
                prototypes,
                dtype=torch.float32,
            ).to(student_features.device),
            dim=-1,
        )

        labels = labels.long().to(
            student_features.device
        )

        prototype_logits = (
            student_features
            @ prototypes.T
        ) / temperature

        return F.cross_entropy(
            prototype_logits,
            labels,
        )


# ------------------------------------------------------------
# Make teacher prototypes safe before passing them to training
# ------------------------------------------------------------

teacher_prototypes = _safe_teacher_tensor(
    teacher_prototypes,
    dtype=torch.float32,
).to(DEVICE)


# ------------------------------------------------------------
# Recreate the student cleanly
# ------------------------------------------------------------

try:
    del student_model
    del student_history
except Exception:
    pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


student_model = RadarOnlyModel(
    NUM_CLASSES,
    num_train_subjects=len(SUBJECT_TO_TRAIN_ID),
).to(DEVICE)


# ------------------------------------------------------------
# Initialise from the teacher's corruption-trained radar branch
# ------------------------------------------------------------

student_model.encoder.load_state_dict(
    teacher_model.radar_encoder.state_dict(),
    strict=True,
)


full_loss_configuration = StudentLossConfiguration(
    use_logit=True,
    use_temporal=True,
    use_relation=True,
    use_prototype=True,
    use_subject_adversary=True,
    name="full_rt_pd",
)


# ------------------------------------------------------------
# Train RTPD-Net student
# ------------------------------------------------------------

student_history = train_distilled_student(
    student_model,
    teacher_model,
    teacher_prototypes,
    train_loader,
    val_loader,
    train_df,
    cfg.STUDENT_EPOCHS,
    full_loss_configuration,
)


student_history.to_csv(
    Path(cfg.WORK_DIR)
    / "full_student_history.csv",
    index=False,
)


torch.save(
    student_model.state_dict(),
    Path(cfg.WORK_DIR)
    / "rtpdnet_radar_student.pt",
)


print("Saved full RTPD-Net radar-only student.")
display(student_history.tail())

full_rt_pd 01/22 | loss=3.3428 | conf=0.848 | val_macro_f1=0.3416
full_rt_pd 02/22 | loss=3.3584 | conf=0.848 | val_macro_f1=0.4663
full_rt_pd 03/22 | loss=3.6679 | conf=0.848 | val_macro_f1=0.4958
full_rt_pd 04/22 | loss=4.0683 | conf=0.848 | val_macro_f1=0.5144
full_rt_pd 05/22 | loss=4.5394 | conf=0.848 | val_macro_f1=0.5756
full_rt_pd 06/22 | loss=4.8981 | conf=0.848 | val_macro_f1=0.5749
full_rt_pd 07/22 | loss=5.2374 | conf=0.848 | val_macro_f1=0.6282
full_rt_pd 08/22 | loss=4.9327 | conf=0.848 | val_macro_f1=0.6133
full_rt_pd 09/22 | loss=4.7993 | conf=0.848 | val_macro_f1=0.5052
full_rt_pd 10/22 | loss=4.6383 | conf=0.848 | val_macro_f1=0.5194
full_rt_pd 11/22 | loss=4.4853 | conf=0.848 | val_macro_f1=0.6460
full_rt_pd 12/22 | loss=4.4988 | conf=0.848 | val_macro_f1=0.7316
full_rt_pd 13/22 | loss=4.4193 | conf=0.848 | val_macro_f1=0.6545
full_rt_pd 14/22 | loss=4.3905 | conf=0.848 | val_macro_f1=0.7595
full_rt_pd 15/22 | loss=4.2659 | conf=0.848 | val_macro_f1=0.7161
full_rt_pd

,epoch,loss,hard,logit,temporal,relation,prototype,subject,teacher_confidence,val_accuracy,val_balanced_accuracy,val_macro_f1,val_weighted_f1,grl_coefficient,distillation_ramp,lr
17,18,4.136908,0.560236,0.755812,4.538204,0.018052,1.460208,2.419312,0.847725,0.775,0.775,0.757222,0.757222,0.999390,1.0,0.000075
18,19,4.073837,0.540774,0.711729,4.537192,0.017421,1.448617,2.442611,0.847725,0.775,0.775,0.765556,0.765556,0.999621,1.0,0.000075
19,20,3.946210,0.482289,0.660761,4.533076,0.016624,1.398861,2.456738,0.847725,0.775,0.775,0.765556,0.765556,0.999765,1.0,0.000075
20,21,3.886397,0.455659,0.636114,4.530278,0.015767,1.379965,2.443416,0.847725,0.750,0.750,0.736032,0.736032,0.999854,1.0,0.000075
21,22,3.898476,0.457941,0.635759,4.532673,0.017342,1.398348,2.449048,0.847725,0.775,0.775,0.757698,0.757698,0.999909,1.0,0.000037


# Final Experiment: Constrained Robustness Fine-Tuning

This section starts after the full RTPD-Net student has been trained.

The architecture is unchanged:

```text
RadarOnlyModel → same RTPD-Net radar-only student
```

The fine-tuning is constrained by a frozen copy of the trained RTPD student. The frozen model acts as the **clean-performance anchor**. The trainable model sees corrupted radar views, but it is penalised if it moves too far away from the original RTPD clean predictions and original parameter region.

This is intentionally different from the earlier attempts:

- no new model architecture;
- no simultaneous temporal/prototype/relation/adversarial objectives;
- no new multimodal teacher;
- no ensemble as the main solution;
- no test-set model selection.

In [28]:
# ============================================================
# Constrained fine-tuning configuration
# ============================================================

from dataclasses import dataclass
from pathlib import Path
import copy
import json
import math
import random
import time
from collections import defaultdict

@dataclass
class ConstrainedFTConfig:
    # Keep the clean RTPD model almost fixed.
    CLEAN_F1_TOLERANCE: float = 0.005
    CLEAN_ACC_TOLERANCE: float = 0.010
    MAX_ECE_INCREASE: float = 0.030

    # Fine-tuning is deliberately small.
    EPOCHS: int = 10
    LEARNING_RATE: float = 4e-5
    WEIGHT_DECAY: float = 5e-5
    PATIENCE: int = 4
    GRAD_CLIP: float = 3.0

    # Constrained robustness loss weights.
    CLEAN_CE_WEIGHT: float = 0.40
    ROBUST_CE_WEIGHT: float = 0.45
    CLEAN_ANCHOR_KL_WEIGHT: float = 0.85
    CORRUPT_ANCHOR_KL_WEIGHT: float = 0.35
    CONSISTENCY_WEIGHT: float = 0.18
    L2SP_WEIGHT: float = 2e-5

    # KD temperature for the frozen clean RTPD anchor.
    ANCHOR_TEMPERATURE: float = 3.0

    # Corruption curriculum.
    MIN_SEVERITY: float = 0.08
    MAX_SEVERITY: float = 0.45

    # Candidate recipes: no architecture changes, only fine-tuning schedules.
    RECIPES: tuple = (
        "preserve_clean",
        "balanced_robust",
        "strong_robust",
    )

    # Validation robustness conditions used for checkpoint selection.
    VALIDATION_CORRUPTIONS: tuple = (
        ("frame_mask", 0.20),
        ("frame_mask", 0.35),
        ("temporal_block", 0.30),
        ("channel_drop", 0.20),
        ("gaussian", 0.30),
        ("spatial_mask", 0.25),
        ("amplitude", 0.35),
    )

    # Final test robustness conditions.
    TEST_CORRUPTIONS: tuple = (
        ("clean", None),
        ("frame_mask_10", ("frame_mask", 0.10)),
        ("frame_mask_30", ("frame_mask", 0.30)),
        ("frame_mask_50", ("frame_mask", 0.50)),
        ("temporal_block_30", ("temporal_block", 0.30)),
        ("channel_drop_20", ("channel_drop", 0.20)),
        ("channel_drop_40", ("channel_drop", 0.40)),
        ("gaussian_25", ("gaussian", 0.25)),
        ("gaussian_50", ("gaussian", 0.50)),
        ("spatial_mask_30", ("spatial_mask", 0.30)),
        ("amplitude_50", ("amplitude", 0.50)),
        ("temporal_roll_50", ("temporal_roll", 0.50)),
    )

    # Statistical testing.
    BOOTSTRAP_SAMPLES: int = 2000
    PERMUTATION_SAMPLES: int = 5000

    # Output.
    OUTPUT_SUBDIR: str = "sensors_constrained_ft"

cft = ConstrainedFTConfig()

CFT_WORK_DIR = Path(cfg.WORK_DIR) / cft.OUTPUT_SUBDIR
CFT_WORK_DIR.mkdir(parents=True, exist_ok=True)

print(cft)
print("Constrained fine-tuning output directory:", CFT_WORK_DIR)

ConstrainedFTConfig(CLEAN_F1_TOLERANCE=0.005, CLEAN_ACC_TOLERANCE=0.01, MAX_ECE_INCREASE=0.03, EPOCHS=10, LEARNING_RATE=4e-05, WEIGHT_DECAY=5e-05, PATIENCE=4, GRAD_CLIP=3.0, CLEAN_CE_WEIGHT=0.4, ROBUST_CE_WEIGHT=0.45, CLEAN_ANCHOR_KL_WEIGHT=0.85, CORRUPT_ANCHOR_KL_WEIGHT=0.35, CONSISTENCY_WEIGHT=0.18, L2SP_WEIGHT=2e-05, ANCHOR_TEMPERATURE=3.0, MIN_SEVERITY=0.08, MAX_SEVERITY=0.45, RECIPES=('preserve_clean', 'balanced_robust', 'strong_robust'), VALIDATION_CORRUPTIONS=(('frame_mask', 0.2), ('frame_mask', 0.35), ('temporal_block', 0.3), ('channel_drop', 0.2), ('gaussian', 0.3), ('spatial_mask', 0.25), ('amplitude', 0.35)), TEST_CORRUPTIONS=(('clean', None), ('frame_mask_10', ('frame_mask', 0.1)), ('frame_mask_30', ('frame_mask', 0.3)), ('frame_mask_50', ('frame_mask', 0.5)), ('temporal_block_30', ('temporal_block', 0.3)), ('channel_drop_20', ('channel_drop', 0.2)), ('channel_drop_40', ('channel_drop', 0.4)), ('gaussian_25', ('gaussian', 0.25)), ('gaussian_50', ('gaussian', 0.5)), ('spatia

In [29]:
# ============================================================
# Radar corruption functions for constrained fine-tuning
# ============================================================

RADAR_CORRUPTION_KINDS = (
    "frame_mask",
    "temporal_block",
    "spatial_mask",
    "channel_drop",
    "gaussian",
    "amplitude",
    "temporal_roll",
)


def constrained_corruption_severity(epoch, epochs, recipe):
    progress = min(max((epoch - 1) / max(epochs - 1, 1), 0.0), 1.0)

    if recipe == "preserve_clean":
        max_severity = min(cft.MAX_SEVERITY, 0.30)
    elif recipe == "balanced_robust":
        max_severity = min(cft.MAX_SEVERITY, 0.40)
    elif recipe == "strong_robust":
        max_severity = cft.MAX_SEVERITY
    else:
        raise ValueError(f"Unknown recipe: {recipe}")

    return cft.MIN_SEVERITY + progress * (max_severity - cft.MIN_SEVERITY)


def corrupt_radar_only(radar, severity, forced_kind=None):
    corrupted = radar.clone()
    batch_size, time_steps, height, width, channels = corrupted.shape
    kinds = []

    for index in range(batch_size):
        kind = forced_kind or random.choice(RADAR_CORRUPTION_KINDS)
        kinds.append(kind)

        if kind == "frame_mask":
            count = max(1, int(round(time_steps * severity)))
            selected = torch.randperm(time_steps, device=radar.device)[:count]
            corrupted[index, selected] = 0.0

        elif kind == "temporal_block":
            count = max(1, int(round(time_steps * severity)))
            start = random.randint(0, max(time_steps - count, 0))
            corrupted[index, start:start + count] = 0.0

        elif kind == "spatial_mask":
            box_h = max(1, int(round(height * math.sqrt(severity))))
            box_w = max(1, int(round(width * math.sqrt(severity))))
            top = random.randint(0, max(height - box_h, 0))
            left = random.randint(0, max(width - box_w, 0))
            corrupted[index, :, top:top + box_h, left:left + box_w, :] = 0.0

        elif kind == "channel_drop":
            number = max(1, min(channels, int(math.ceil(channels * severity))))
            selected = torch.randperm(channels, device=radar.device)[:number]
            corrupted[index, :, :, :, selected] = 0.0

        elif kind == "gaussian":
            sigma = 0.04 + 0.50 * severity
            corrupted[index] += sigma * torch.randn_like(corrupted[index])

        elif kind == "amplitude":
            scale = max(0.15, 1.0 - 0.90 * severity)
            corrupted[index] *= scale

        elif kind == "temporal_roll":
            max_shift = max(1, int(round(time_steps * 0.25 * severity)))
            shift = random.randint(-max_shift, max_shift)
            corrupted[index] = torch.roll(corrupted[index], shifts=shift, dims=0)

        else:
            raise ValueError(f"Unknown radar corruption: {kind}")

    return corrupted, kinds


def make_two_corrupted_views(radar, severity):
    first, first_kinds = corrupt_radar_only(radar, severity)
    second, second_kinds = corrupt_radar_only(radar, severity)
    return first, second, first_kinds, second_kinds

In [30]:
# ============================================================
# Calibration, prediction, and multi-objective evaluation helpers
# ============================================================

def cft_multiclass_brier(probabilities, labels, num_classes):
    one_hot = np.eye(num_classes)[labels]
    return float(np.mean(np.sum((probabilities - one_hot) ** 2, axis=1)))


def cft_expected_calibration_error(probabilities, labels, bins=10):
    confidence = probabilities.max(axis=1)
    prediction = probabilities.argmax(axis=1)
    correctness = prediction == labels
    edges = np.linspace(0.0, 1.0, bins + 1)
    error = 0.0

    for lower, upper in zip(edges[:-1], edges[1:]):
        if upper == 1.0:
            mask = (confidence >= lower) & (confidence <= upper)
        else:
            mask = (confidence >= lower) & (confidence < upper)

        if not mask.any():
            continue

        error += mask.mean() * abs(correctness[mask].mean() - confidence[mask].mean())

    return float(error)


class CFTTemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_temperature = nn.Parameter(torch.zeros(1))

    @property
    def temperature(self):
        return self.log_temperature.exp().clamp(0.05, 20.0)

    def forward(self, logits):
        return logits / self.temperature

    def fit(self, logits, labels):
        logits_tensor = torch.tensor(logits, dtype=torch.float32, device=DEVICE)
        labels_tensor = torch.tensor(labels, dtype=torch.long, device=DEVICE)

        optimizer = torch.optim.LBFGS(
            [self.log_temperature],
            lr=0.10,
            max_iter=100,
            line_search_fn="strong_wolfe",
        )

        def closure():
            optimizer.zero_grad()
            loss = F.cross_entropy(self(logits_tensor), labels_tensor)
            loss.backward()
            return loss

        optimizer.step(closure)
        return float(self.temperature.detach().cpu())


def cft_probabilities_from_logits(logits, temperature=1.0):
    tensor = torch.tensor(logits, dtype=torch.float32)
    return torch.softmax(tensor / float(temperature), dim=-1).numpy()


@torch.inference_mode()
def predict_rtpd_radar(model, loader, corruption=None, seed=2026):
    model.eval()
    logits_list = []
    labels_list = []
    sample_ids = []
    subjects = []

    previous_python = random.getstate()
    previous_torch = torch.random.get_rng_state()
    previous_cuda = torch.cuda.get_rng_state_all() if DEVICE.type == "cuda" else None

    random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(seed)

    for batch_data in loader:
        radar = batch_data["radar"].to(DEVICE, non_blocking=DEVICE.type == "cuda")

        if corruption is not None:
            kind, severity = corruption
            radar, _ = corrupt_radar_only(radar, severity=float(severity), forced_kind=kind)

        output = model(radar)
        logits_list.append(output["logits"].detach().cpu())
        labels_list.append(batch_data["label"].cpu())
        sample_ids.extend(batch_data["sample_id"])
        subjects.extend(batch_data["subject"])

    random.setstate(previous_python)
    torch.random.set_rng_state(previous_torch)
    if DEVICE.type == "cuda" and previous_cuda is not None:
        torch.cuda.set_rng_state_all(previous_cuda)

    logits = torch.cat(logits_list).numpy()
    labels = torch.cat(labels_list).numpy()

    return {
        "logits": logits,
        "y_true": labels,
        "y_pred": logits.argmax(axis=1),
        "sample_ids": np.asarray(sample_ids),
        "subjects": np.asarray(subjects),
    }


def cft_metric_row(name, logits, labels, temperature=1.0):
    probabilities = cft_probabilities_from_logits(logits, temperature)
    predictions = probabilities.argmax(axis=1)

    return {
        "model": name,
        "accuracy": accuracy_score(labels, predictions),
        "balanced_accuracy": balanced_accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, predictions, average="weighted", zero_division=0),
        "nll": log_loss(labels, probabilities, labels=list(range(NUM_CLASSES))),
        "brier": cft_multiclass_brier(probabilities, labels, NUM_CLASSES),
        "ece": cft_expected_calibration_error(probabilities, labels, bins=10),
        "temperature": temperature,
    }, probabilities


def fit_temperature_on_validation(model, loader):
    prediction = predict_rtpd_radar(model, loader, seed=cfg.SEED)
    scaler = CFTTemperatureScaler().to(DEVICE)
    temperature = scaler.fit(prediction["logits"], prediction["y_true"])
    return temperature


def evaluate_clean_model(model, loader, temperature=1.0, name="model"):
    prediction = predict_rtpd_radar(model, loader, seed=cfg.SEED)
    row, probabilities = cft_metric_row(
        name,
        prediction["logits"],
        prediction["y_true"],
        temperature,
    )
    return row, prediction, probabilities


def evaluate_validation_robustness(model, loader, temperature=1.0, seed_offset=0):
    rows = []

    for condition_index, corruption in enumerate(cft.VALIDATION_CORRUPTIONS):
        prediction = predict_rtpd_radar(
            model,
            loader,
            corruption=corruption,
            seed=cfg.SEED + seed_offset + condition_index,
        )
        probabilities = cft_probabilities_from_logits(
            prediction["logits"],
            temperature,
        )
        predictions = probabilities.argmax(axis=1)
        rows.append({
            "corruption": f"{corruption[0]}_{corruption[1]:.2f}",
            "accuracy": accuracy_score(prediction["y_true"], predictions),
            "macro_f1": f1_score(
                prediction["y_true"],
                predictions,
                average="macro",
                zero_division=0,
            ),
            "ece": cft_expected_calibration_error(
                probabilities,
                prediction["y_true"],
                bins=10,
            ),
        })

    robust_df = pd.DataFrame(rows)

    return {
        "mean_corrupted_macro_f1": float(robust_df["macro_f1"].mean()),
        "worst_corrupted_macro_f1": float(robust_df["macro_f1"].min()),
        "mean_corrupted_accuracy": float(robust_df["accuracy"].mean()),
        "mean_corrupted_ece": float(robust_df["ece"].mean()),
    }, robust_df

In [31]:
# ============================================================
# Baseline RTPD validation and test measurements
# ============================================================

rtpd_anchor_model = copy.deepcopy(student_model).to(DEVICE)
rtpd_anchor_model.eval()

for parameter in rtpd_anchor_model.parameters():
    parameter.requires_grad_(False)

base_val_temperature = fit_temperature_on_validation(rtpd_anchor_model, val_loader)
base_test_temperature = base_val_temperature

base_val_row, base_val_prediction, base_val_probabilities = evaluate_clean_model(
    rtpd_anchor_model,
    val_loader,
    temperature=base_val_temperature,
    name="RTPD-Net clean reference validation",
)

base_test_row, base_test_prediction, base_test_probabilities = evaluate_clean_model(
    rtpd_anchor_model,
    test_loader,
    temperature=base_test_temperature,
    name="RTPD-Net clean reference test",
)

base_val_robust_summary, base_val_robust_df = evaluate_validation_robustness(
    rtpd_anchor_model,
    val_loader,
    temperature=base_val_temperature,
    seed_offset=1000,
)

print("Base RTPD validation clean metrics:")
display(pd.DataFrame([base_val_row]).style.format({
    col: "{:.4f}" for col in pd.DataFrame([base_val_row]).select_dtypes(include=[np.number]).columns
}))

print("Base RTPD validation robustness summary:")
display(pd.DataFrame([base_val_robust_summary]).style.format("{:.4f}"))

print("Base RTPD test clean metrics:")
display(pd.DataFrame([base_test_row]).style.format({
    col: "{:.4f}" for col in pd.DataFrame([base_test_row]).select_dtypes(include=[np.number]).columns
}))

base_val_robust_df.to_csv(CFT_WORK_DIR / "base_validation_robustness.csv", index=False)
pd.DataFrame([base_val_row]).to_csv(CFT_WORK_DIR / "base_validation_clean.csv", index=False)
pd.DataFrame([base_test_row]).to_csv(CFT_WORK_DIR / "base_test_clean.csv", index=False)

Base RTPD validation clean metrics:


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1,nll,brier,ece,temperature
0,RTPD-Net clean reference validation,0.7750,0.7750,0.7656,0.7656,0.8054,0.3280,0.0751,0.7897


Base RTPD validation robustness summary:


,mean_corrupted_macro_f1,worst_corrupted_macro_f1,mean_corrupted_accuracy,mean_corrupted_ece
0,0.5206,0.3741,0.5643,0.1788


Base RTPD test clean metrics:


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1,nll,brier,ece,temperature
0,RTPD-Net clean reference test,0.7500,0.7500,0.7496,0.7496,0.7963,0.3424,0.1264,0.7897


In [32]:
# ============================================================
# Constrained fine-tuning losses
# ============================================================

def anchor_kl_loss(student_logits, anchor_logits):
    temperature = float(cft.ANCHOR_TEMPERATURE)

    return F.kl_div(
        F.log_softmax(student_logits.float() / temperature, dim=-1),
        F.softmax(anchor_logits.detach().float() / temperature, dim=-1),
        reduction="batchmean",
    ) * (temperature ** 2)


def l2sp_penalty(model, anchor_state):
    penalty = torch.zeros((), device=DEVICE)

    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if name not in anchor_state:
            continue

        reference = anchor_state[name].to(parameter.device, dtype=parameter.dtype)
        penalty = penalty + torch.sum((parameter - reference) ** 2)

    return penalty


def recipe_weights(recipe):
    if recipe == "preserve_clean":
        return {
            "clean_ce": cft.CLEAN_CE_WEIGHT * 1.25,
            "robust_ce": cft.ROBUST_CE_WEIGHT * 0.65,
            "clean_anchor": cft.CLEAN_ANCHOR_KL_WEIGHT * 1.35,
            "corrupt_anchor": cft.CORRUPT_ANCHOR_KL_WEIGHT * 0.80,
            "consistency": cft.CONSISTENCY_WEIGHT * 0.75,
            "l2sp": cft.L2SP_WEIGHT * 1.50,
        }

    if recipe == "balanced_robust":
        return {
            "clean_ce": cft.CLEAN_CE_WEIGHT,
            "robust_ce": cft.ROBUST_CE_WEIGHT,
            "clean_anchor": cft.CLEAN_ANCHOR_KL_WEIGHT,
            "corrupt_anchor": cft.CORRUPT_ANCHOR_KL_WEIGHT,
            "consistency": cft.CONSISTENCY_WEIGHT,
            "l2sp": cft.L2SP_WEIGHT,
        }

    if recipe == "strong_robust":
        return {
            "clean_ce": cft.CLEAN_CE_WEIGHT * 0.80,
            "robust_ce": cft.ROBUST_CE_WEIGHT * 1.35,
            "clean_anchor": cft.CLEAN_ANCHOR_KL_WEIGHT * 0.90,
            "corrupt_anchor": cft.CORRUPT_ANCHOR_KL_WEIGHT * 1.20,
            "consistency": cft.CONSISTENCY_WEIGHT * 1.20,
            "l2sp": cft.L2SP_WEIGHT * 0.80,
        }

    raise ValueError(f"Unknown constrained fine-tuning recipe: {recipe}")


def constrained_selection_score(clean_row, robustness_summary):
    clean_margin = clean_row["macro_f1"] - (
        base_val_row["macro_f1"] - cft.CLEAN_F1_TOLERANCE
    )
    robust_gain = (
        robustness_summary["mean_corrupted_macro_f1"]
        - base_val_robust_summary["mean_corrupted_macro_f1"]
    )

    # Reward robustness only when the clean constraint is satisfied.
    if clean_margin < 0:
        return -10.0 + clean_margin

    return (
        0.45 * clean_row["macro_f1"]
        + 0.45 * robustness_summary["mean_corrupted_macro_f1"]
        + 0.10 * robustness_summary["worst_corrupted_macro_f1"]
        - 0.05 * clean_row["ece"]
        + 0.10 * max(0.0, robust_gain)
    )

In [33]:
# ============================================================
# Constrained RTPD fine-tuning trainer
# ============================================================

def train_constrained_finetune(recipe):
    seed_everything(cfg.SEED + abs(hash(recipe)) % 10000)

    model = copy.deepcopy(student_model).to(DEVICE)
    anchor = copy.deepcopy(rtpd_anchor_model).to(DEVICE).eval()

    for parameter in anchor.parameters():
        parameter.requires_grad_(False)

    anchor_state = {
        name: value.detach().clone().to(DEVICE)
        for name, value in anchor.state_dict().items()
        if torch.is_floating_point(value)
    }

    class_weights = class_weights_for_frame(train_df, DEVICE)
    supervised_criterion = nn.CrossEntropyLoss(
        weight=None if cfg.USE_WEIGHTED_SAMPLER else class_weights,
        label_smoothing=cfg.LABEL_SMOOTHING,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cft.LEARNING_RATE,
        weight_decay=cft.WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
    )

    scaler = make_grad_scaler()
    weights = recipe_weights(recipe)

    best_state = copy.deepcopy(model.state_dict())
    best_score = -np.inf
    best_row = None
    best_robust_summary = None
    best_epoch = 0
    stale_epochs = 0
    history = []

    for epoch in range(1, cft.EPOCHS + 1):
        model.train()
        anchor.eval()

        severity = constrained_corruption_severity(epoch, cft.EPOCHS, recipe)
        totals = defaultdict(float)
        seen = 0

        for batch_data in train_loader:
            radar = batch_data["radar"].to(DEVICE, non_blocking=DEVICE.type == "cuda")
            labels = batch_data["label"].to(DEVICE, dtype=torch.long, non_blocking=DEVICE.type == "cuda")

            corrupted_1, corrupted_2, _, _ = make_two_corrupted_views(radar, severity)

            with torch.no_grad():
                anchor_clean_logits = anchor(radar)["logits"].detach().clone()

            optimizer.zero_grad(set_to_none=True)

            with autocast_context():
                clean_output = model(radar)
                corrupt_output_1 = model(corrupted_1)
                corrupt_output_2 = model(corrupted_2)

                clean_ce = supervised_criterion(clean_output["logits"], labels)
                robust_ce = 0.5 * (
                    supervised_criterion(corrupt_output_1["logits"], labels)
                    + supervised_criterion(corrupt_output_2["logits"], labels)
                )

            clean_anchor = anchor_kl_loss(clean_output["logits"], anchor_clean_logits)
            corrupt_anchor = 0.5 * (
                anchor_kl_loss(corrupt_output_1["logits"], anchor_clean_logits)
                + anchor_kl_loss(corrupt_output_2["logits"], anchor_clean_logits)
            )

            consistency = 0.5 * (
                F.kl_div(
                    F.log_softmax(corrupt_output_1["logits"].float(), dim=-1),
                    F.softmax(clean_output["logits"].detach().float(), dim=-1),
                    reduction="batchmean",
                )
                + F.kl_div(
                    F.log_softmax(corrupt_output_2["logits"].float(), dim=-1),
                    F.softmax(clean_output["logits"].detach().float(), dim=-1),
                    reduction="batchmean",
                )
            )

            l2sp = l2sp_penalty(model, anchor_state)

            loss = (
                weights["clean_ce"] * clean_ce.float()
                + weights["robust_ce"] * robust_ce.float()
                + weights["clean_anchor"] * clean_anchor
                + weights["corrupt_anchor"] * corrupt_anchor
                + weights["consistency"] * consistency
                + weights["l2sp"] * l2sp
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite constrained fine-tuning loss in recipe {recipe}."
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cft.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            batch_size = labels.shape[0]
            seen += batch_size

            for key, value in {
                "loss": loss,
                "clean_ce": clean_ce,
                "robust_ce": robust_ce,
                "clean_anchor": clean_anchor,
                "corrupt_anchor": corrupt_anchor,
                "consistency": consistency,
                "l2sp": l2sp,
            }.items():
                totals[key] += float(value.detach()) * batch_size

        # Validation evaluation is the actual constraint mechanism.
        val_temperature = fit_temperature_on_validation(model, val_loader)
        clean_row, _, _ = evaluate_clean_model(
            model,
            val_loader,
            temperature=val_temperature,
            name=f"{recipe}_validation_epoch_{epoch}",
        )
        robust_summary, _ = evaluate_validation_robustness(
            model,
            val_loader,
            temperature=val_temperature,
            seed_offset=2000 + epoch,
        )

        score = constrained_selection_score(clean_row, robust_summary)

        clean_f1_pass = clean_row["macro_f1"] >= base_val_row["macro_f1"] - cft.CLEAN_F1_TOLERANCE
        clean_acc_pass = clean_row["accuracy"] >= base_val_row["accuracy"] - cft.CLEAN_ACC_TOLERANCE
        ece_pass = clean_row["ece"] <= base_val_row["ece"] + cft.MAX_ECE_INCREASE
        robust_pass = robust_summary["mean_corrupted_macro_f1"] > base_val_robust_summary["mean_corrupted_macro_f1"]

        constraints_pass = clean_f1_pass and clean_acc_pass and ece_pass and robust_pass

        scheduler.step(score)

        row = {
            "recipe": recipe,
            "epoch": epoch,
            "severity": severity,
            "selection_score": score,
            "constraints_pass": constraints_pass,
            "clean_f1_pass": clean_f1_pass,
            "clean_acc_pass": clean_acc_pass,
            "ece_pass": ece_pass,
            "robust_pass": robust_pass,
            "val_temperature": val_temperature,
            **{f"train_{key}": value / max(seen, 1) for key, value in totals.items()},
            **{f"val_clean_{key}": value for key, value in clean_row.items() if key != "model"},
            **{f"val_robust_{key}": value for key, value in robust_summary.items()},
            "lr": optimizer.param_groups[0]["lr"],
        }

        history.append(row)

        print(
            f"{recipe} {epoch:02d}/{cft.EPOCHS} | "
            f"clean_f1={clean_row['macro_f1']:.4f} | "
            f"robust_mean_f1={robust_summary['mean_corrupted_macro_f1']:.4f} | "
            f"constraints={constraints_pass} | score={score:.4f}"
        )

        # Prefer constraint-satisfying checkpoints. If none exists,
        # the final selection cell will safely fall back to the base RTPD model.
        if constraints_pass and score > best_score + 1e-9:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            best_row = row
            best_robust_summary = robust_summary
            best_epoch = epoch
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= cft.PATIENCE and epoch >= 4:
                print(f"{recipe}: early stopping.")
                break

    history_df = pd.DataFrame(history)

    if best_row is not None:
        model.load_state_dict(best_state)
        accepted = True
    else:
        # Load final state for inspection, but mark as not accepted.
        accepted = False

    return model, history_df, accepted, best_epoch, best_score, best_row, best_robust_summary

In [34]:
# ============================================================
# Run constrained fine-tuning recipes
# ============================================================

constrained_candidates = []
constrained_histories = []

for recipe in cft.RECIPES:
    print("\n" + "=" * 100)
    print("Running constrained fine-tuning recipe:", recipe)

    candidate_model, history_df, accepted, best_epoch, best_score, best_row, best_robust_summary = train_constrained_finetune(recipe)

    history_path = CFT_WORK_DIR / f"history_{recipe}.csv"
    checkpoint_path = CFT_WORK_DIR / f"checkpoint_{recipe}.pt"

    history_df.to_csv(history_path, index=False)
    torch.save(candidate_model.state_dict(), checkpoint_path)

    constrained_histories.append(history_df)

    constrained_candidates.append({
        "recipe": recipe,
        "accepted_by_constraints": bool(accepted),
        "best_epoch": int(best_epoch),
        "best_score": float(best_score) if np.isfinite(best_score) else np.nan,
        "checkpoint": str(checkpoint_path),
        "history": str(history_path),
        "best_row": best_row,
        "best_robust_summary": best_robust_summary,
    })

    del candidate_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

constrained_registry_df = pd.DataFrame([
    {
        "recipe": item["recipe"],
        "accepted_by_constraints": item["accepted_by_constraints"],
        "best_epoch": item["best_epoch"],
        "best_score": item["best_score"],
        "checkpoint": item["checkpoint"],
        "history": item["history"],
    }
    for item in constrained_candidates
])

display(constrained_registry_df)
constrained_registry_df.to_csv(CFT_WORK_DIR / "constrained_candidate_registry.csv", index=False)

all_history_df = pd.concat(constrained_histories, ignore_index=True)
all_history_df.to_csv(CFT_WORK_DIR / "all_constrained_histories.csv", index=False)

display(
    all_history_df[
        [
            "recipe",
            "epoch",
            "constraints_pass",
            "val_clean_macro_f1",
            "val_clean_accuracy",
            "val_clean_ece",
            "val_robust_mean_corrupted_macro_f1",
            "val_robust_worst_corrupted_macro_f1",
            "selection_score",
        ]
    ].sort_values(["constraints_pass", "selection_score"], ascending=[False, False])
)


Running constrained fine-tuning recipe: preserve_clean
preserve_clean 01/10 | clean_f1=0.7572 | robust_mean_f1=0.5124 | constraints=False | score=-10.0033
preserve_clean 02/10 | clean_f1=0.8139 | robust_mean_f1=0.5547 | constraints=True | score=0.6599
preserve_clean 03/10 | clean_f1=0.8139 | robust_mean_f1=0.5673 | constraints=False | score=0.6638
preserve_clean 04/10 | clean_f1=0.8111 | robust_mean_f1=0.5601 | constraints=False | score=0.6560
preserve_clean 05/10 | clean_f1=0.7885 | robust_mean_f1=0.5702 | constraints=False | score=0.6514
preserve_clean 06/10 | clean_f1=0.8111 | robust_mean_f1=0.6478 | constraints=True | score=0.7216
preserve_clean 07/10 | clean_f1=0.7401 | robust_mean_f1=0.6299 | constraints=False | score=-10.0205
preserve_clean 08/10 | clean_f1=0.7849 | robust_mean_f1=0.6514 | constraints=False | score=0.7056
preserve_clean 09/10 | clean_f1=0.7617 | robust_mean_f1=0.6185 | constraints=False | score=0.6703
preserve_clean 10/10 | clean_f1=0.7367 | robust_mean_f1=0.68

,recipe,accepted_by_constraints,best_epoch,best_score,checkpoint,history
0,preserve_clean,True,6,0.721647,/kaggle/working/rtpdnet_mri/sensors_constraine...,/kaggle/working/rtpdnet_mri/sensors_constraine...
1,balanced_robust,True,5,0.693270,/kaggle/working/rtpdnet_mri/sensors_constraine...,/kaggle/working/rtpdnet_mri/sensors_constraine...
2,strong_robust,False,0,NaN,/kaggle/working/rtpdnet_mri/sensors_constraine...,/kaggle/working/rtpdnet_mri/sensors_constraine...


,recipe,epoch,constraints_pass,val_clean_macro_f1,val_clean_accuracy,val_clean_ece,val_robust_mean_corrupted_macro_f1,val_robust_worst_corrupted_macro_f1,selection_score
5,preserve_clean,6,True,0.811111,0.825,0.093520,0.647785,0.571023,0.721647
14,balanced_robust,5,True,0.788492,0.800,0.099047,0.629613,0.491746,0.693270
1,preserve_clean,2,True,0.813889,0.825,0.092109,0.554695,0.452778,0.659944
11,balanced_robust,2,True,0.784921,0.800,0.085467,0.575894,0.443534,0.657976
12,balanced_robust,3,True,0.782143,0.800,0.082269,0.561054,0.457463,0.650116
18,balanced_robust,9,False,0.784365,0.800,0.148557,0.681421,0.524158,0.720673
15,balanced_robust,6,False,0.784921,0.800,0.148004,0.645085,0.572967,0.705847
7,preserve_clean,8,False,0.784921,0.800,0.176539,0.651389,0.549870,0.705578
20,strong_robust,2,False,0.811111,0.825,0.136479,0.596831,0.507253,0.685098
13,balanced_robust,4,False,0.793254,0.800,0.106005,0.605380,0.485022,0.681064


In [35]:
# ============================================================
# Select final constrained checkpoint or safely fall back
# ============================================================

accepted_candidates = [
    candidate
    for candidate in constrained_candidates
    if candidate["accepted_by_constraints"]
]

if accepted_candidates:
    selected_candidate = sorted(
        accepted_candidates,
        key=lambda item: item["best_score"],
        reverse=True,
    )[0]

    selected_model = RadarOnlyModel(
        NUM_CLASSES,
        num_train_subjects=len(SUBJECT_TO_TRAIN_ID),
    ).to(DEVICE)

    selected_model.load_state_dict(
        torch.load(selected_candidate["checkpoint"], map_location=DEVICE)
    )

    selected_model_name = f"Constrained RTPD-FT ({selected_candidate['recipe']})"
    selected_status = "accepted_constrained_finetune"

else:
    selected_candidate = None
    selected_model = copy.deepcopy(rtpd_anchor_model).to(DEVICE)
    selected_model_name = "RTPD-Net clean reference fallback"
    selected_status = "fallback_to_rtpd_clean_reference"

selected_model.eval()

print("Selected final model:", selected_model_name)
print("Selection status:", selected_status)
if selected_candidate is not None:
    print("Best validation epoch:", selected_candidate["best_epoch"])
    print("Best validation score:", selected_candidate["best_score"])

Selected final model: Constrained RTPD-FT (preserve_clean)
Selection status: accepted_constrained_finetune
Best validation epoch: 6
Best validation score: 0.7216474796181775


# Final Evaluation: RTPD Reference vs Constrained Fine-Tuned Model

The final model is now evaluated on the test split. The test split is not used for fine-tuning, temperature selection, checkpoint selection, or recipe selection.

In [36]:
# ============================================================
# Final clean test evaluation
# ============================================================

selected_temperature = fit_temperature_on_validation(selected_model, val_loader)

selected_test_row, selected_test_prediction, selected_test_probabilities = evaluate_clean_model(
    selected_model,
    test_loader,
    temperature=selected_temperature,
    name=selected_model_name,
)

base_test_row, base_test_prediction, base_test_probabilities = evaluate_clean_model(
    rtpd_anchor_model,
    test_loader,
    temperature=base_test_temperature,
    name="RTPD-Net clean reference",
)

final_clean_df = pd.DataFrame([
    base_test_row,
    selected_test_row,
])

display(
    final_clean_df.style.format({
        column: "{:.4f}"
        for column in final_clean_df.select_dtypes(include=[np.number]).columns
    })
)

final_clean_df.to_csv(CFT_WORK_DIR / "final_clean_test_comparison.csv", index=False)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1,nll,brier,ece,temperature
0,RTPD-Net clean reference,0.7500,0.7500,0.7496,0.7496,0.7963,0.3424,0.1264,0.7897
1,Constrained RTPD-FT (preserve_clean),0.7750,0.7750,0.7746,0.7746,0.8302,0.3617,0.1230,0.7944


In [37]:
# ============================================================
# Final robustness test evaluation
# ============================================================

def evaluate_full_test_robustness(model, temperature, model_name):
    rows = []

    for condition_index, (condition_name, corruption) in enumerate(cft.TEST_CORRUPTIONS):
        prediction = predict_rtpd_radar(
            model,
            test_loader,
            corruption=corruption,
            seed=cfg.SEED + 5000 + condition_index,
        )
        probabilities = cft_probabilities_from_logits(
            prediction["logits"],
            temperature,
        )
        predictions = probabilities.argmax(axis=1)

        rows.append({
            "model": model_name,
            "condition": condition_name,
            "severity": 0.0 if corruption is None else float(corruption[1]),
            "accuracy": accuracy_score(prediction["y_true"], predictions),
            "balanced_accuracy": balanced_accuracy_score(prediction["y_true"], predictions),
            "macro_f1": f1_score(
                prediction["y_true"],
                predictions,
                average="macro",
                zero_division=0,
            ),
            "weighted_f1": f1_score(
                prediction["y_true"],
                predictions,
                average="weighted",
                zero_division=0,
            ),
            "ece": cft_expected_calibration_error(
                probabilities,
                prediction["y_true"],
                bins=10,
            ),
            "nll": log_loss(
                prediction["y_true"],
                probabilities,
                labels=list(range(NUM_CLASSES)),
            ),
            "brier": cft_multiclass_brier(
                probabilities,
                prediction["y_true"],
                NUM_CLASSES,
            ),
        })

    result = pd.DataFrame(rows)
    corrupted = result[result["condition"] != "clean"]

    summary = {
        "model": model_name,
        "clean_accuracy": float(result.loc[result["condition"] == "clean", "accuracy"].iloc[0]),
        "clean_macro_f1": float(result.loc[result["condition"] == "clean", "macro_f1"].iloc[0]),
        "clean_ece": float(result.loc[result["condition"] == "clean", "ece"].iloc[0]),
        "mean_corrupted_accuracy": float(corrupted["accuracy"].mean()),
        "mean_corrupted_macro_f1": float(corrupted["macro_f1"].mean()),
        "worst_corrupted_macro_f1": float(corrupted["macro_f1"].min()),
        "worst_condition": str(corrupted.loc[corrupted["macro_f1"].idxmin(), "condition"]),
        "relative_robustness": float(corrupted["macro_f1"].mean() / max(result.loc[result["condition"] == "clean", "macro_f1"].iloc[0], 1e-8)),
        "mean_corrupted_ece": float(corrupted["ece"].mean()),
        "mean_corrupted_nll": float(corrupted["nll"].mean()),
        "mean_corrupted_brier": float(corrupted["brier"].mean()),
    }

    return result, summary


base_robust_df, base_robust_summary = evaluate_full_test_robustness(
    rtpd_anchor_model,
    base_test_temperature,
    "RTPD-Net clean reference",
)

selected_robust_df, selected_robust_summary = evaluate_full_test_robustness(
    selected_model,
    selected_temperature,
    selected_model_name,
)

final_robustness_df = pd.concat([base_robust_df, selected_robust_df], ignore_index=True)
final_robustness_summary_df = pd.DataFrame([base_robust_summary, selected_robust_summary])

print("Full robustness table:")
display(
    final_robustness_df.style.format({
        column: "{:.4f}"
        for column in final_robustness_df.select_dtypes(include=[np.number]).columns
    })
)

print("Robustness summary:")
display(
    final_robustness_summary_df.style.format({
        column: "{:.4f}"
        for column in final_robustness_summary_df.select_dtypes(include=[np.number]).columns
    })
)

final_robustness_df.to_csv(CFT_WORK_DIR / "final_robustness_full.csv", index=False)
final_robustness_summary_df.to_csv(CFT_WORK_DIR / "final_robustness_summary.csv", index=False)

Full robustness table:


,model,condition,severity,accuracy,balanced_accuracy,macro_f1,weighted_f1,ece,nll,brier
0,RTPD-Net clean reference,clean,0.0000,0.7500,0.7500,0.7496,0.7496,0.1264,0.7963,0.3424
1,RTPD-Net clean reference,frame_mask_10,0.1000,0.7500,0.7500,0.7476,0.7476,0.1418,0.7884,0.3384
2,RTPD-Net clean reference,frame_mask_30,0.3000,0.6000,0.6000,0.5917,0.5917,0.1741,1.1078,0.5280
3,RTPD-Net clean reference,frame_mask_50,0.5000,0.4250,0.4250,0.3201,0.3201,0.3030,2.0323,0.8521
4,RTPD-Net clean reference,temporal_block_30,0.3000,0.7250,0.7250,0.7005,0.7005,0.0944,0.8740,0.3705
5,RTPD-Net clean reference,channel_drop_20,0.2000,0.5250,0.5250,0.4955,0.4955,0.2122,1.2312,0.5696
6,RTPD-Net clean reference,channel_drop_40,0.4000,0.3250,0.3250,0.3430,0.3430,0.3846,2.6690,1.0220
7,RTPD-Net clean reference,gaussian_25,0.2500,0.7000,0.7000,0.7154,0.7154,0.0738,1.0468,0.4584
8,RTPD-Net clean reference,gaussian_50,0.5000,0.5750,0.5750,0.5540,0.5540,0.2189,1.5016,0.6172
9,RTPD-Net clean reference,spatial_mask_30,0.3000,0.4750,0.4750,0.4583,0.4583,0.2893,1.5410,0.6747


Robustness summary:


,model,clean_accuracy,clean_macro_f1,clean_ece,mean_corrupted_accuracy,mean_corrupted_macro_f1,worst_corrupted_macro_f1,worst_condition,relative_robustness,mean_corrupted_ece,mean_corrupted_nll,mean_corrupted_brier
0,RTPD-Net clean reference,0.7500,0.7496,0.1264,0.5750,0.5484,0.3201,frame_mask_50,0.7315,0.2026,1.4272,0.5976
1,Constrained RTPD-FT (preserve_clean),0.7750,0.7746,0.1230,0.6045,0.5837,0.3476,channel_drop_40,0.7535,0.2014,1.2990,0.5558


In [38]:
# ============================================================
# Statistical comparison: constrained FT vs RTPD reference
# ============================================================

def paired_bootstrap_macro_f1(labels, baseline_predictions, candidate_predictions, samples, seed):
    rng = np.random.default_rng(seed)
    differences = []

    for _ in range(samples):
        indices = rng.integers(0, len(labels), size=len(labels))
        baseline = f1_score(
            labels[indices],
            baseline_predictions[indices],
            average="macro",
            zero_division=0,
        )
        candidate = f1_score(
            labels[indices],
            candidate_predictions[indices],
            average="macro",
            zero_division=0,
        )
        differences.append(candidate - baseline)

    differences = np.asarray(differences)

    return {
        "mean_delta_macro_f1": float(differences.mean()),
        "ci_low": float(np.quantile(differences, 0.025)),
        "ci_high": float(np.quantile(differences, 0.975)),
        "probability_positive": float((differences > 0).mean()),
    }


def paired_permutation_macro_f1(labels, baseline_predictions, candidate_predictions, samples, seed):
    rng = np.random.default_rng(seed)

    observed = (
        f1_score(labels, candidate_predictions, average="macro", zero_division=0)
        - f1_score(labels, baseline_predictions, average="macro", zero_division=0)
    )

    null_deltas = []

    for _ in range(samples):
        swap = rng.random(len(labels)) < 0.5
        first = baseline_predictions.copy()
        second = candidate_predictions.copy()
        first[swap] = candidate_predictions[swap]
        second[swap] = baseline_predictions[swap]

        null_deltas.append(
            f1_score(labels, second, average="macro", zero_division=0)
            - f1_score(labels, first, average="macro", zero_division=0)
        )

    null_deltas = np.asarray(null_deltas)

    p_value = (1.0 + np.sum(np.abs(null_deltas) >= abs(observed))) / (samples + 1.0)

    return {
        "observed_delta_macro_f1": float(observed),
        "two_sided_p_value": float(p_value),
    }


base_predictions = base_test_probabilities.argmax(axis=1)
selected_predictions = selected_test_probabilities.argmax(axis=1)
labels = base_test_prediction["y_true"]

bootstrap_result = paired_bootstrap_macro_f1(
    labels,
    base_predictions,
    selected_predictions,
    cft.BOOTSTRAP_SAMPLES,
    cfg.SEED,
)

permutation_result = paired_permutation_macro_f1(
    labels,
    base_predictions,
    selected_predictions,
    cft.PERMUTATION_SAMPLES,
    cfg.SEED,
)

statistical_comparison_df = pd.DataFrame([{
    "comparison": f"{selected_model_name} minus RTPD-Net clean reference",
    **bootstrap_result,
    **permutation_result,
}])

display(statistical_comparison_df)
statistical_comparison_df.to_csv(CFT_WORK_DIR / "statistical_comparison.csv", index=False)

,comparison,mean_delta_macro_f1,ci_low,ci_high,probability_positive,observed_delta_macro_f1,two_sided_p_value
0,Constrained RTPD-FT (preserve_clean) minus RTP...,0.021725,-0.02381,0.08423,0.6755,0.024969,1.0


# Multi-Objective Interpretation for Sensors

This section produces the paper-facing comparison table recommended by the supervisor.

The claim is **not** that one model wins every metric. The claim is:

> Wearable-assisted radar learning improves radar-only rehabilitation monitoring, but the optimal model depends on deployment priority.

The table combines the final constrained fine-tuning results with the best verified results from the earlier notebooks.

In [39]:
# ============================================================
# Paper-facing multi-objective comparison table
# ============================================================

# Historical values come from the previously completed notebook reports.
# Update these values only if a rerun changes the reported numbers.
historical_methods = [
    {
        "method": "Standard KD",
        "full_name": "Standard Knowledge Distillation",
        "deployment_role": "Simple radar-only student baseline",
        "main_architecture": "Radar-IMU teacher to radar-only student",
        "clean_accuracy": 0.675,
        "clean_macro_f1": 0.653,
        "mean_corrupted_macro_f1": np.nan,
        "worst_corrupted_macro_f1": np.nan,
        "cross_subject_accuracy": np.nan,
        "cross_subject_macro_f1": np.nan,
        "ece": np.nan,
        "main_strength": "Simple and interpretable baseline",
        "main_limitation": "Lower clean performance than RTPD-Net",
    },
    {
        "method": "RTPD-Net",
        "full_name": "Reliability-Aware Temporal Prototype Distillation Network",
        "deployment_role": "Clean-performance reference",
        "main_architecture": "Reliability teacher + temporal, relation and prototype distillation into radar-only student",
        "clean_accuracy": base_test_row["accuracy"],
        "clean_macro_f1": base_test_row["macro_f1"],
        "mean_corrupted_macro_f1": base_robust_summary["mean_corrupted_macro_f1"],
        "worst_corrupted_macro_f1": base_robust_summary["worst_corrupted_macro_f1"],
        "cross_subject_accuracy": 0.625,
        "cross_subject_macro_f1": 0.595,
        "ece": base_test_row["ece"],
        "main_strength": "Best clean fixed-split radar-only recognition in earlier results",
        "main_limitation": "Corruption robustness can degrade under channel and temporal damage",
    },
    {
        "method": "PARETO-RKD",
        "full_name": "Pareto-Safe Adaptive Reliability, Calibration and Corruption-Robust Knowledge Distillation",
        "deployment_role": "Robustness and cross-subject stability evidence",
        "main_architecture": "Counterfactual reliability teacher + worst-view radar curriculum + Pareto-safe selection",
        "clean_accuracy": 0.800,
        "clean_macro_f1": 0.792,
        "mean_corrupted_macro_f1": 0.723,
        "worst_corrupted_macro_f1": 0.644,
        "cross_subject_accuracy": 0.710,
        "cross_subject_macro_f1": 0.684,
        "ece": 0.145,
        "main_strength": "Best cross-subject and radar-corruption profile among prior enhanced students",
        "main_limitation": "Did not beat RTPD-Net on clean fixed-split accuracy/F1",
    },
    {
        "method": "AURORA-RKD",
        "full_name": "Adaptive Uncertainty-Calibrated Reliability-Oriented Robust Aggregated Knowledge Distillation",
        "deployment_role": "Calibration and complete missing-modality safety evidence",
        "main_architecture": "Quality-aware multimodal teacher + multi-resolution radar candidate library + validation ensemble",
        "clean_accuracy": 0.800,
        "clean_macro_f1": 0.793,
        "mean_corrupted_macro_f1": 0.711,
        "worst_corrupted_macro_f1": 0.444,
        "cross_subject_accuracy": 0.643,
        "cross_subject_macro_f1": 0.613,
        "ece": 0.0889,
        "main_strength": "Best student calibration and complete missing-modality routing",
        "main_limitation": "Not strongest for clean accuracy, robustness, or cross-subject generalisation",
    },
    {
        "method": selected_model_name,
        "full_name": "Constrained Fine-Tuned RTPD-Net",
        "deployment_role": "Final targeted robustness-preserving test",
        "main_architecture": "Same RTPD-Net architecture; small robust fine-tuning constrained by clean RTPD anchor",
        "clean_accuracy": selected_test_row["accuracy"],
        "clean_macro_f1": selected_test_row["macro_f1"],
        "mean_corrupted_macro_f1": selected_robust_summary["mean_corrupted_macro_f1"],
        "worst_corrupted_macro_f1": selected_robust_summary["worst_corrupted_macro_f1"],
        "cross_subject_accuracy": np.nan,
        "cross_subject_macro_f1": np.nan,
        "ece": selected_test_row["ece"],
        "main_strength": "Tests whether RTPD clean accuracy can be preserved while improving robustness",
        "main_limitation": "Accepted only if validation constraints pass; otherwise fallback confirms RTPD remains clean reference",
    },
]

multi_objective_df = pd.DataFrame(historical_methods)

display(
    multi_objective_df.style.format({
        column: "{:.4f}"
        for column in multi_objective_df.select_dtypes(include=[np.number]).columns
    })
)

multi_objective_df.to_csv(CFT_WORK_DIR / "paper_multi_objective_method_comparison.csv", index=False)

,method,full_name,deployment_role,main_architecture,clean_accuracy,clean_macro_f1,mean_corrupted_macro_f1,worst_corrupted_macro_f1,cross_subject_accuracy,cross_subject_macro_f1,ece,main_strength,main_limitation
0,Standard KD,Standard Knowledge Distillation,Simple radar-only student baseline,Radar-IMU teacher to radar-only student,0.6750,0.6530,nan,nan,nan,nan,nan,Simple and interpretable baseline,Lower clean performance than RTPD-Net
1,RTPD-Net,Reliability-Aware Temporal Prototype Distillation Network,Clean-performance reference,"Reliability teacher + temporal, relation and prototype distillation into radar-only student",0.7500,0.7496,0.5484,0.3201,0.6250,0.5950,0.1264,Best clean fixed-split radar-only recognition in earlier results,Corruption robustness can degrade under channel and temporal damage
2,PARETO-RKD,"Pareto-Safe Adaptive Reliability, Calibration and Corruption-Robust Knowledge Distillation",Robustness and cross-subject stability evidence,Counterfactual reliability teacher + worst-view radar curriculum + Pareto-safe selection,0.8000,0.7920,0.7230,0.6440,0.7100,0.6840,0.1450,Best cross-subject and radar-corruption profile among prior enhanced students,Did not beat RTPD-Net on clean fixed-split accuracy/F1
3,AURORA-RKD,Adaptive Uncertainty-Calibrated Reliability-Oriented Robust Aggregated Knowledge Distillation,Calibration and complete missing-modality safety evidence,Quality-aware multimodal teacher + multi-resolution radar candidate library + validation ensemble,0.8000,0.7930,0.7110,0.4440,0.6430,0.6130,0.0889,Best student calibration and complete missing-modality routing,"Not strongest for clean accuracy, robustness, or cross-subject generalisation"
4,Constrained RTPD-FT (preserve_clean),Constrained Fine-Tuned RTPD-Net,Final targeted robustness-preserving test,Same RTPD-Net architecture; small robust fine-tuning constrained by clean RTPD anchor,0.7750,0.7746,0.5837,0.3476,nan,nan,0.1230,Tests whether RTPD clean accuracy can be preserved while improving robustness,Accepted only if validation constraints pass; otherwise fallback confirms RTPD remains clean reference


In [40]:
# ============================================================
# Automatic champion table by deployment priority
# ============================================================

champion_rows = []

priority_definitions = [
    ("Clean fixed-split accuracy", "clean_accuracy", "max"),
    ("Clean fixed-split macro-F1", "clean_macro_f1", "max"),
    ("Mean radar-corruption macro-F1", "mean_corrupted_macro_f1", "max"),
    ("Worst radar-corruption macro-F1", "worst_corrupted_macro_f1", "max"),
    ("Cross-subject accuracy", "cross_subject_accuracy", "max"),
    ("Cross-subject macro-F1", "cross_subject_macro_f1", "max"),
    ("Calibration / lowest ECE", "ece", "min"),
]

for priority, column, direction in priority_definitions:
    available = multi_objective_df.dropna(subset=[column]).copy()
    if available.empty:
        continue

    if direction == "max":
        winner = available.loc[available[column].idxmax()]
    else:
        winner = available.loc[available[column].idxmin()]

    champion_rows.append({
        "deployment_priority": priority,
        "best_method": winner["method"],
        "value": float(winner[column]),
        "interpretation": winner["main_strength"],
    })

deployment_champion_df = pd.DataFrame(champion_rows)

display(
    deployment_champion_df.style.format({
        "value": "{:.4f}",
    })
)

deployment_champion_df.to_csv(CFT_WORK_DIR / "deployment_priority_champions.csv", index=False)

,deployment_priority,best_method,value,interpretation
0,Clean fixed-split accuracy,PARETO-RKD,0.8000,Best cross-subject and radar-corruption profile among prior enhanced students
1,Clean fixed-split macro-F1,AURORA-RKD,0.7930,Best student calibration and complete missing-modality routing
2,Mean radar-corruption macro-F1,PARETO-RKD,0.7230,Best cross-subject and radar-corruption profile among prior enhanced students
3,Worst radar-corruption macro-F1,PARETO-RKD,0.6440,Best cross-subject and radar-corruption profile among prior enhanced students
4,Cross-subject accuracy,PARETO-RKD,0.7100,Best cross-subject and radar-corruption profile among prior enhanced students
5,Cross-subject macro-F1,PARETO-RKD,0.6840,Best cross-subject and radar-corruption profile among prior enhanced students
6,Calibration / lowest ECE,AURORA-RKD,0.0889,Best student calibration and complete missing-modality routing


In [41]:
# ============================================================
# Supervisor-facing conclusion generator
# ============================================================

clean_preserved = (
    selected_test_row["macro_f1"]
    >= base_test_row["macro_f1"] - cft.CLEAN_F1_TOLERANCE
)

robustness_improved = (
    selected_robust_summary["mean_corrupted_macro_f1"]
    > base_robust_summary["mean_corrupted_macro_f1"]
)

if selected_status == "accepted_constrained_finetune" and clean_preserved and robustness_improved:
    constrained_conclusion = (
        "The constrained fine-tuning experiment supports the supervisor's hypothesis: "
        "RTPD-Net's clean performance can be approximately preserved while improving mean radar-corruption robustness."
    )
elif selected_status == "accepted_constrained_finetune":
    constrained_conclusion = (
        "A constrained fine-tuned checkpoint passed validation constraints, but the test-set trade-off should be interpreted cautiously."
    )
else:
    constrained_conclusion = (
        "No constrained fine-tuned checkpoint passed the validation constraints. "
        "This supports using RTPD-Net as the clean-performance reference and treating robustness as a separate deployment objective."
    )

paper_claim = (
    "The strongest paper claim is a multi-objective radar-only deployment study: "
    "wearable-assisted training improves radar-only rehabilitation monitoring, "
    "but optimal model choice depends on deployment priority rather than one universally dominant architecture."
)

conclusion_df = pd.DataFrame([
    {
        "item": "Final constrained experiment status",
        "text": constrained_conclusion,
    },
    {
        "item": "Recommended main paper claim",
        "text": paper_claim,
    },
    {
        "item": "Recommended primary model framing",
        "text": (
            "Use RTPD-Net as the clean-performance reference, not as the only final answer. "
            "Use PARETO-RKD and AURORA-RKD as evidence for robustness, cross-subject stability, calibration, "
            "and missing-modality operating regimes."
        ),
    },
    {
        "item": "Sensors suitability framing",
        "text": (
            "The study is suitable as a Sensors-style deployment analysis if it reports all objectives honestly: "
            "clean accuracy, macro-F1, robustness, calibration, grouped-subject generalisation, and limitations."
        ),
    },
])

for _, row in conclusion_df.iterrows():
    print(f"\n{row['item']}:\n{row['text']}")

conclusion_df.to_csv(CFT_WORK_DIR / "supervisor_facing_conclusions.csv", index=False)


Final constrained experiment status:
The constrained fine-tuning experiment supports the supervisor's hypothesis: RTPD-Net's clean performance can be approximately preserved while improving mean radar-corruption robustness.

Recommended main paper claim:
The strongest paper claim is a multi-objective radar-only deployment study: wearable-assisted training improves radar-only rehabilitation monitoring, but optimal model choice depends on deployment priority rather than one universally dominant architecture.

Recommended primary model framing:
Use RTPD-Net as the clean-performance reference, not as the only final answer. Use PARETO-RKD and AURORA-RKD as evidence for robustness, cross-subject stability, calibration, and missing-modality operating regimes.

Sensors suitability framing:
The study is suitable as a Sensors-style deployment analysis if it reports all objectives honestly: clean accuracy, macro-F1, robustness, calibration, grouped-subject generalisation, and limitations.


In [42]:
# ============================================================
# Save full constrained experiment package
# ============================================================

summary = {
    "paper_framing": "Multi-objective radar-only deployment study",
    "supervisor_message_integrated": True,
    "selected_status": selected_status,
    "selected_model_name": selected_model_name,
    "base_validation_clean": base_val_row,
    "base_validation_robustness": base_val_robust_summary,
    "base_test_clean": base_test_row,
    "base_test_robustness": base_robust_summary,
    "selected_test_clean": selected_test_row,
    "selected_test_robustness": selected_robust_summary,
    "statistical_comparison": statistical_comparison_df.to_dict("records"),
    "deployment_champions": deployment_champion_df.to_dict("records"),
    "conclusions": conclusion_df.to_dict("records"),
    "config": {
        "main_cfg": asdict(cfg),
        "constrained_ft_cfg": cft.__dict__,
    },
}

with open(CFT_WORK_DIR / "sensors_constrained_ft_summary.json", "w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2, default=str)

archive_path = shutil.make_archive(
    str(Path(cfg.WORK_DIR) / "sensors_constrained_ft_outputs"),
    "zip",
    CFT_WORK_DIR,
)

print("Saved summary JSON:", CFT_WORK_DIR / "sensors_constrained_ft_summary.json")
print("Saved archive:", archive_path)
print("Saved all CSV outputs in:", CFT_WORK_DIR)

Saved summary JSON: /kaggle/working/rtpdnet_mri/sensors_constrained_ft/sensors_constrained_ft_summary.json
Saved archive: /kaggle/working/rtpdnet_mri/sensors_constrained_ft_outputs.zip
Saved all CSV outputs in: /kaggle/working/rtpdnet_mri/sensors_constrained_ft


# How to use this notebook in the paper

## Main claim

**Wearable-assisted radar learning improves radar-only rehabilitation monitoring, but optimal model choice depends on deployment priority.**

## Recommended model roles

- **RTPD-Net:** clean fixed-split performance reference.
- **PARETO-RKD:** robustness and grouped-subject stability evidence.
- **AURORA-RKD:** calibration and complete missing-modality safety evidence.
- **Constrained RTPD-FT:** final hypothesis test for preserving clean RTPD performance while improving robustness.

## Recommended Sensors-style contribution list

1. A privacy-preserving radar-only deployment analysis for rehabilitation-exercise monitoring.
2. A wearable-assisted training paradigm where IMU is used during learning but not required at radar-only deployment.
3. A multi-objective comparison across clean recognition, calibration, corruption robustness and subject generalisation.
4. A constrained fine-tuning experiment showing whether the clean-performance reference can be made more robust without changing architecture.
5. Evidence that no single architecture should be claimed as universally optimal; model choice depends on deployment risk and operating condition.

## Discussion language to avoid

Avoid:

> “Our final architecture outperforms all methods in all aspects.”

Use:

> “The experiments reveal complementary operating regimes. RTPD-Net is the clean-performance reference, PARETO-RKD provides the strongest robustness/generalisation evidence, AURORA-RKD provides the strongest calibration and missing-modality behaviour, and constrained fine-tuning tests whether RTPD-Net can be adapted without sacrificing its clean recognition advantage.”